# DCF Automation Notebook — Refactored
## Run Order
1. **Part 0** — Master Config (edit only here)
2. **Part 1** — Dependencies & Imports
3. **Part 2** — File Extraction Engine
4. **Part 3** — Data Fetch & Historical Build
5. **Part 4** — DCF Build (Forecast + WACC + Valuation)
6. **Part 5** — Excel Output Writer
7. **Part 6** — Sensitivity Table
8. **Part 7** — Financial Statement Sheets
9. **Part 8** — Industry Comparison
10. **Part 9** — Final Formatting & Cleanup


# Part 0 — Master Config
Edit **only** this cell, then **Restart Kernel → Run All**.

In [1]:
# =============================================================================
# ★  MASTER CONFIG — EDIT ONLY THIS CELL
# =============================================================================

# ── 1. COMPANY ───────────────────────────────────────────────────────────────
TICKER_INPUT = "RELIANCE.NS "      # e.g. AAPL | MSFT | TCS.NS | RELIANCE.NS

# ── 2. WHICH SOURCES TO USE ──────────────────────────────────────────────────
SOURCES = {
    "api":   True,    # Yahoo Finance / yfinance (listed companies)
    "excel": False,   # Excel files inside ZIP or folder
    "pdf":   False,   # PDF financial statements inside ZIP or folder
    "ppt":   False,   # PowerPoint files inside ZIP or folder
}

# ── 3. PRIORITY: first = wins when two sources have the same metric ───────────
SOURCE_PRIORITY = ["excel", "pdf", "ppt", "api"]

# ── 4. FILE PATH (only needed when excel / pdf / ppt is True) ────────────────
FILE_SOURCE_PATH = ""

# ── 5. TEMPLATE ──────────────────────────────────────────────────────────────
TEMPLATE_XLSM = "DCF.xlsm"   # master template (never overwritten)

# ── 6. WACC INPUTS (use globals() fallback so file-injected values are kept) ─
WACC_DIRECT         = globals().get("WACC_DIRECT", None)
RISK_FREE           = globals().get("RISK_FREE") or 4.0
EQUITY_RISK_PREMIUM = globals().get("EQUITY_RISK_PREMIUM") or 5.0
BETA                = globals().get("BETA") or 1.10
COST_OF_DEBT_PRETAX = globals().get("COST_OF_DEBT_PRETAX") or 5.0
TAX_RATE            = globals().get("TAX_RATE") or 21.0
TARGET_DEBT_WEIGHT  = globals().get("TARGET_DEBT_WEIGHT") or 20.0

# ── 7. DCF PARAMETERS ────────────────────────────────────────────────────────
FORECAST_YEARS   = 5
TERMINAL_GROWTH  = globals().get("TERMINAL_GROWTH") or 2.0
EXIT_MULTIPLE    = 12.0
USE_EXIT_MULTIPLE = False
MID_YEAR         = True
BASE_YEAR        = None

# ── 8. OUTPUT SWITCHES ────────────────────────────────────────────────────────
CURRENCY_UNITS      = "Millions"    # Display in Millions (KGI raw dollars ÷ 1,000,000)
EXPORT_FLAT_XLSX    = False
WRITE_RUN_META_JSON = False
PEERS_OVERRIDE      = None

# ── 9. SENSITIVITY GRIDS ─────────────────────────────────────────────────────
import numpy as np
WACC_GRID      = list(np.round(np.arange(0.06, 0.12 + 0.0001, 0.01), 4))
GROWTH_GRID    = [0.01, 0.02, 0.025, 0.03, 0.035]
EXIT_MULT_GRID = [10, 11, 12, 13, 14]
MARGIN_ADJ     = [-0.03, -0.015, 0.0, 0.015, 0.03]

print("✅ Config loaded.")
print(f"   Ticker  : {TICKER_INPUT}")
print(f"   Active  : { {k:v for k,v in SOURCES.items() if v} }")
print(f"   Priority: {SOURCE_PRIORITY}")


✅ Config loaded.
   Ticker  : RELIANCE.NS 
   Active  : {'api': True}
   Priority: ['excel', 'pdf', 'ppt', 'api']


# Part 1 — Dependencies & Imports

In [2]:
import importlib, sys, subprocess

def pip_install(spec):
    print(f"Installing {spec}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", spec])

def ensure(pkg, install_spec=None):
    spec = install_spec or pkg
    try:
        importlib.import_module(pkg)
    except ImportError:
        pip_install(spec)

for pkg, spec in [("pandas",None),("numpy",None),("yfinance",None),
                  ("openpyxl",None),("xlsxwriter",None)]:
    ensure(pkg, spec)
print("✅ All dependencies ready.")


✅ All dependencies ready.


In [3]:
import math, json, re, os, shutil, zipfile, tempfile, warnings
import numpy as np
import pandas as pd
from datetime import datetime, date
from pathlib import Path
import yfinance as yf
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.utils.cell import column_index_from_string, range_boundaries
from openpyxl.cell.cell import MergedCell
from openpyxl.styles import Font, Alignment

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
RUN_TS = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Imports ready — {RUN_TS}")


✅ Imports ready — 2026-04-17 03:14:42


# Part 2 — File Extraction Engine
Do NOT edit this cell.

In [4]:
# =============================================================================
# FILE EXTRACTION ENGINE  (Excel / PDF / PPT → unified DCF dataset)
# This cell runs automatically — do NOT edit.
# To add new line-item aliases, extend METRIC_PATTERNS below.
# =============================================================================
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


import os
import re
import io
import json
import math
import shutil
import zipfile
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


# =========================
# FILE DISCOVERY + CLASSIFICATION
# =========================

def infer_year(text: str):
    txt = str(text)

    years4 = [int(y) for y in re.findall(r'(?<!\d)(20\d{2})(?!\d)', txt)]
    if years4:
        return max(years4)

    fy = [int(y) for y in re.findall(r'FY\s*[-\']?\s*(20\d{2}|\d{2})[A-Z]*', txt, flags=re.I)]
    if fy:
        vals = []
        for y in fy:
            y = int(y)
            vals.append(y if y >= 100 else 2000 + y)
        return max(vals)

    m = re.search(r'(?:^|[^0-9])(?:\d{1,2}[\/_-]?\d{1,2}[\/_-]?)(\d{2})(?:[^0-9]|$)', txt)
    if m:
        yy = int(m.group(1))
        return 2000 + yy if yy <= 50 else 1900 + yy

    return None


def classify_file(path: Path, strict_annual_only: bool = True) -> dict:
    name = path.name.lower()
    ext = path.suffix.lower()

    result = {
        "path": str(path),
        "name": path.name,
        "ext": ext,
        "statement_type": None,
        "period_type": "other",
        "include": False,
        "reason": "",
        "year_hint": infer_year(path.name),
    }

    rolling_terms = [
        "ytd", "year to date", "12 months", "12 month", "12 periods", "12 period",
        "rolling", "trailing", "ttm", "monthly", "quarter", "q1", "q2", "q3", "q4",
    ]
    noisy_pdf_terms = ["client copy", "articles of incorporation", "incorporation", "tax return"]
    ignore_excel_terms = ["readme", "instructions"]

    if ext in [".xlsx", ".xls", ".xlsm"]:
        if any(t in name for t in ignore_excel_terms):
            result["reason"] = "excluded helper/instruction workbook"
            return result

        if strict_annual_only and any(t in name for t in rolling_terms):
            result["period_type"] = "interim_or_rolling"
            result["reason"] = "excluded interim/rolling excel"
            return result

        # Keep annual financial workbooks and useful support schedules.
        result["include"] = True
        result["period_type"] = "annual"

        if any(t in name for t in ["income", "p&l", "profit", "operations"]):
            result["statement_type"] = "income_statement"
        elif "balance" in name or "financial position" in name:
            result["statement_type"] = "balance_sheet"
        elif "cash flow" in name or "cashflow" in name:
            result["statement_type"] = "cash_flow"
        elif "capex" in name or "working capital" in name or "financial data" in name or "operational data" in name:
            result["statement_type"] = "support_schedule"
        else:
            result["statement_type"] = "excel_other"

        result["reason"] = "included excel workbook"
        return result

    if ext == ".pdf":
        if any(t in name for t in noisy_pdf_terms):
            result["reason"] = "excluded noisy/non-financial pdf"
            return result
        result["include"] = True
        result["period_type"] = "annual"
        result["statement_type"] = "financial_pdf"
        result["reason"] = "included pdf"
        return result

    if ext in [".ppt", ".pptx"]:
        result["include"] = True
        result["statement_type"] = "presentation"
        result["period_type"] = "other"
        result["reason"] = "included presentation"
        return result

    if ext == ".csv":
        result["include"] = True
        result["period_type"] = "annual"
        name_lower = name.lower()
        if any(k in name_lower for k in ["income", "p&l", "profit", "pnl"]):
            result["statement_type"] = "income_statement"
        elif any(k in name_lower for k in ["balance", "bs"]):
            result["statement_type"] = "balance_sheet"
        elif any(k in name_lower for k in ["cash", "cf"]):
            result["statement_type"] = "cash_flow"
        else:
            result["statement_type"] = "csv_other"
        result["reason"] = "included csv file"
        return result

    if ext == ".txt":
        # Include only as metadata source (WACC inputs etc.), not financial data
        if any(t in name for t in ["article", "readme", "instruction"]):
            result["reason"] = "excluded noise txt"
            return result
        result["include"] = True
        result["statement_type"] = "txt_metadata"
        result["period_type"] = "other"
        result["reason"] = "included txt (metadata/WACC)"
        return result

    result["reason"] = "unsupported file type"
    return result


def resolve_root(input_path: str | Path):
    input_path = Path(input_path)
    temp_dir = None

    if input_path.is_file() and input_path.suffix.lower() == ".zip":
        temp_dir = tempfile.mkdtemp(prefix="dcf_zip_")
        with zipfile.ZipFile(input_path, "r") as zf:
            zf.extractall(temp_dir)
        root = Path(temp_dir)
    else:
        root = input_path

    return root, temp_dir


def discover_files(input_path: str | Path, strict_annual_only: bool = True) -> tuple[pd.DataFrame, str | None]:
    root, temp_dir = resolve_root(input_path)

    if not root.exists():
        raise FileNotFoundError(f"ROOT_PATH not found: {root}")

    rows = []
    for p in root.rglob("*"):
        if p.is_file():
            rows.append(classify_file(p, strict_annual_only=strict_annual_only))

    files_df = pd.DataFrame(rows)
    if files_df.empty:
        raise ValueError(f"No files found under: {root}")

    files_df = files_df.sort_values(["ext", "name"]).reset_index(drop=True)
    return files_df, temp_dir


# =========================
# EXTRACTORS
# =========================

def normalize_line_item(text: str) -> str:
    text = str(text).lower().strip()
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def sheet_text_sample(df: pd.DataFrame, nrows: int = 20) -> str:
    return " ".join(df.head(nrows).fillna("").astype(str).values.flatten().tolist())


def detect_unit_scale(text: str) -> float:
    txt = str(text).lower()
    if "in billions" in txt or "usd billions" in txt or "$bn" in txt:
        return 1_000_000_000.0
    # Extended patterns: "usd million", "in usd millions", "figures in USD millions"
    if ("in millions" in txt or "usd millions" in txt or "usd million" in txt
            or "in usd million" in txt or "figures in usd" in txt
            or "$ millions" in txt or "$m)" in txt or "$m " in txt):
        return 1_000_000.0
    if "in thousands" in txt or "$000" in txt or "$ 000" in txt or "000s" in txt:
        return 1_000.0
    return 1.0


def parse_year_token(val):
    if pd.isna(val):
        return None

    # Handle numeric types directly (int/float cell values)
    if isinstance(val, (int, float)):
        iv = int(val)
        if 2000 <= iv <= 2040:
            return iv
        # Excel date serial: 40000-55000 covers ~2009-2050
        if 40000 <= iv <= 55000:
            try:
                from datetime import date, timedelta
                d = date(1899, 12, 30) + timedelta(days=iv)
                if 2000 <= d.year <= 2040:
                    return d.year
            except Exception:
                pass
        return None

    txt = str(val).strip()
    if not txt:
        return None

    patterns = [
        r'(?<!\d)(20\d{2})(?!\d)',
        r'FY\s*[-\']?\s*(20\d{2}|\d{2})[A-Z]*',
        r'(?:fiscal\s+year|year\s+ended?)\s+(20\d{2})',
        r'(?:ended?|ending)\s+(?:\w+\s+)?(?:\d{1,2}[,\s]+)?(20\d{2})',
    ]
    for pat in patterns:
        m = re.search(pat, txt, flags=re.I)
        if m:
            y = m.group(1)
            y = int(y)
            return y if y >= 100 else 2000 + y

    months = r'jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec|december'
    if re.search(months, txt, flags=re.I):
        m = re.search(r'(20\d{2})', txt)
        if m:
            return int(m.group(1))

    return None


def numeric_or_none(x):
    if pd.isna(x):
        return None
    if isinstance(x, (int, float, np.number)):
        return float(x)
    txt = str(x).strip()
    if not txt or txt.lower() in {"nan", "none", "-", "—", "–"}:
        return None
    txt = txt.replace("$", "").replace(",", "").replace("%", "").strip()
    neg = txt.startswith("(") and txt.endswith(")")
    if neg:
        txt = txt[1:-1].strip()
    try:
        val = float(txt)
        return -val if neg else val
    except Exception:
        return None


def detect_sheet_role(df: pd.DataFrame, file_name: str = "", sheet_name: str = ""):
    sample = f"{file_name} {sheet_name} {sheet_text_sample(df, 18)}".lower()

    ignore_terms = [
        "projection inputs", "projection assumptions", "forecast", "budget",
        "segment data", "internal use only", "not for distribution",
    ]
    if any(t in sample for t in ignore_terms):
        return None

    # Helpful support schedules / KPI tabs should not be skipped outright.
    if "working capital detail" in sample or re.search(r'\bwc detail\b', sample):
        return "balance_sheet"
    if "capex schedule" in sample or "capital expenditure" in sample or "total capital expenditures" in sample:
        return "cash_flow"
    if "kpis" in sample or "ratios" in sample or "key performance indicators" in sample:
        return "operating_kpi"
    if "balance sheet" in sample or "financial position" in sample:
        return "balance_sheet"
    if "cash flow" in sample or "cash flow stmt" in sample or "statement of cash flows" in sample:
        return "cash_flow"
    if (
        "income statement" in sample or "profit and loss" in sample or "p and l" in sample or
        "p&l" in sample or "statement of operations" in sample or "consolidated profit" in sample
    ):
        return "income_statement"
    return None


def detect_header_row(df: pd.DataFrame, max_scan_rows: int = 25):
    best_row = None
    best_year_cols = []
    best_score = -1

    for i in range(min(max_scan_rows, len(df))):
        row = df.iloc[i]
        year_cols = []
        score = 0
        for j, val in enumerate(row.tolist()):
            year = parse_year_token(val)
            if year is not None:
                year_cols.append((j, year))
                score += 3

            sval = str(val).lower()
            if any(k in sval for k in ["fy", "december 31", "year ended"]):
                score += 1

        if len(year_cols) >= 2:
            first_year_col = min(c for c, _ in year_cols)
            # Prefer rows where the label column sits immediately to the left of the first year column.
            if first_year_col <= 2:
                score += 2

        if score > best_score and len(year_cols) >= 2:
            best_row = i
            best_year_cols = year_cols
            best_score = score

    return best_row, best_year_cols


def guess_label_column(df: pd.DataFrame, header_row: int, year_cols: list[tuple[int, int]]):
    if not year_cols:
        return 0
    first_year_col = min(c for c, _ in year_cols)

    # Prefer the nearest non-empty column to the left of the first year column.
    for c in range(first_year_col - 1, -1, -1):
        header_val = str(df.iloc[header_row, c]).strip().lower()
        if header_val and header_val != "nan":
            return c

    return 0


def should_skip_label(label: str) -> bool:
    label = str(label).strip()
    if not label or label.lower() == "nan":
        return True
    if len(label) > 180:
        return True

    low = label.lower()
    skip_terms = [
        "income statement", "balance sheet", "statement of cash flow",
        "for the twelve months", "for the year ended", "december 31",
        "all figures in", "source:", "prepared by:", "last updated:",
        "based on", "draft", "not final", "do not distribute",
    ]
    if any(t in low for t in skip_terms):
        return True

    safe_all_caps = {
        "EBIT", "EBITDA", "D&A", "EPS", "DPS", "COGS", "SG&A",
        "CASH FROM OPERATIONS", "FREE CASH FLOW", "NET DEBT", "SHARES OUTSTANDING",
        "OPERATING CASH FLOW", "NET INCOME", "CAPITAL EXPENDITURES",
    }
    if label.upper() in safe_all_caps:
        return False

    # Section headings without numbers.
    if re.fullmatch(r"[A-Z\s&/:()'\-]+", label) and label.upper() == label and "TOTAL" not in label and "SUB-TOTAL" not in label:
        return True

    return False


def extract_structured_multi_year(df: pd.DataFrame, file_path: Path, sheet_name: str, statement_type: str):
    header_row, year_cols = detect_header_row(df)
    if header_row is None or len(year_cols) < 2:
        return pd.DataFrame()

    label_col = guess_label_column(df, header_row, year_cols)
    top_text = sheet_text_sample(df, 12)
    unit_scale = detect_unit_scale(f"{file_path.name} {sheet_name} {top_text}")

    records = []
    for r in range(header_row + 1, len(df)):
        row = df.iloc[r]
        label = row.iloc[label_col] if label_col < len(row) else None
        if should_skip_label(label):
            continue
        label = str(label).strip()

        found_any = False
        for c, year in year_cols:
            if c >= len(row):
                continue
            val = numeric_or_none(row.iloc[c])
            if val is None:
                continue
            found_any = True
            records.append({
                "source_type": "excel",
                "source_file": file_path.name,
                "sheet_name": sheet_name,
                "statement_type": statement_type,
                "year": int(year),
                "raw_line_item": label,
                "line_item_norm": normalize_line_item(label),
                "value_num": float(val) * unit_scale,
                "source_kind": "structured_multi_year",
                "unit_scale": unit_scale,
            })

        if not found_any:
            continue

    return pd.DataFrame(records)


def choose_value_column_legacy(df: pd.DataFrame, statement_type: str):
    top = df.head(min(15, len(df))).fillna("")
    header_scores = {}

    max_cols = min(15, len(df.columns))
    for col in range(max_cols):
        score = 0
        vals = top.iloc[:, col].astype(str).str.lower()
        joined = " ".join(vals.tolist())

        if statement_type in ("income_statement", "cash_flow"):
            if "year to date" in joined or "ytd" in joined:
                score += 100
            if "twelve months" in joined or "12 months" in joined or "for the year ended" in joined:
                score += 25
            if "current month" in joined:
                score -= 20
            if "%" in joined or "percent" in joined:
                score -= 50

        series = pd.to_numeric(df.iloc[:, col], errors="coerce")
        score += series.notna().sum() * 0.2
        header_scores[col] = score

    if not header_scores:
        return None

    return max(header_scores, key=header_scores.get)


def extract_legacy_single_year(df: pd.DataFrame, file_path: Path, sheet_name: str, statement_type: str):
    top_text = sheet_text_sample(df, 12)
    year = infer_year(file_path.name) or infer_year(top_text)
    if year is None:
        # Last resort: use BASE_YEAR from globals if set (e.g. most recent annual file)
        _by = globals().get('BASE_YEAR') or globals().get('_base_year_hint')
        if _by and 2000 <= int(_by) <= 2040:
            year = int(_by)
        else:
            return pd.DataFrame()

    unit_scale = detect_unit_scale(f"{file_path.name} {sheet_name} {top_text}")
    value_col = choose_value_column_legacy(df, statement_type)

    records = []
    for _, row in df.iterrows():
        label = row.iloc[0] if len(row) else None
        if should_skip_label(label):
            continue

        label = str(label).strip()
        if statement_type == "balance_sheet":
            nums = pd.to_numeric(row, errors="coerce")
            nums = nums[nums.notna()]
            if len(nums) == 0:
                continue
            value = float(nums.iloc[-1])
        else:
            if value_col is None or value_col >= len(row):
                continue
            value = pd.to_numeric(row.iloc[value_col], errors="coerce")
            if pd.isna(value):
                continue
            value = float(value)

        records.append({
            "source_type": "excel",
            "source_file": file_path.name,
            "sheet_name": sheet_name,
            "statement_type": statement_type,
            "year": int(year),
            "raw_line_item": label,
            "line_item_norm": normalize_line_item(label),
            "value_num": float(value) * unit_scale,
            "source_kind": "legacy_single_year",
            "unit_scale": unit_scale,
        })

    return pd.DataFrame(records)


def parse_excel_statement(file_path: str | Path, forced_statement_type: str | None = None) -> pd.DataFrame:
    file_path = Path(file_path)
    xls = pd.ExcelFile(file_path)
    parts = []

    for sheet_name in xls.sheet_names:
        try:
            df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
        except Exception:
            continue

        if df.empty:
            continue

        statement_type = forced_statement_type if forced_statement_type in {"income_statement", "balance_sheet", "cash_flow"} else None
        sheet_role = detect_sheet_role(df, file_path.name, sheet_name)
        statement_type = sheet_role or statement_type

        if statement_type is None:
            continue

        # First try a dynamic multi-year parser. If that fails, fall back to the original single-year style parser.
        structured = extract_structured_multi_year(df, file_path, sheet_name, statement_type)
        if not structured.empty:
            parts.append(structured)
            continue

        legacy = extract_legacy_single_year(df, file_path, sheet_name, statement_type)
        if not legacy.empty:
            parts.append(legacy)

    if not parts:
        return pd.DataFrame(columns=[
            "source_type", "source_file", "sheet_name", "statement_type",
            "year", "raw_line_item", "line_item_norm", "value_num", "source_kind", "unit_scale"
        ])

    out = pd.concat(parts, ignore_index=True)
    out = out[out["year"].notna()].copy()
    out["year"] = out["year"].astype(int)
    return out


def extract_excel_master(files_df: pd.DataFrame) -> pd.DataFrame:
    excel_df = files_df[(files_df["include"]) & (files_df["ext"].isin([".xlsx", ".xls", ".xlsm"]))].copy()
    parts = []

    for _, row in excel_df.iterrows():
        try:
            part = parse_excel_statement(row["path"], forced_statement_type=row["statement_type"])
            if not part.empty:
                parts.append(part)
        except Exception as e:
            print(f"Skipped {row['name']} because of parsing error: {e}")

    if not parts:
        return pd.DataFrame(columns=[
            "source_type", "source_file", "sheet_name", "statement_type",
            "year", "raw_line_item", "line_item_norm", "value_num", "source_kind", "unit_scale"
        ])

    out = pd.concat(parts, ignore_index=True)
    out = out[out["year"].notna()].copy()
    out["year"] = out["year"].astype(int)
    return out



def _pdf_line_amount_tokens(line: str):
    pattern = r'\(\s*[-+]?\d[\d,]*(?:\.\d+)?\s*\)|[-+]?\d[\d,]*(?:\.\d+)?'
    toks = re.findall(pattern, str(line))
    cleaned = []
    for t in toks:
        ts = str(t).strip()
        if re.fullmatch(r'20\d{2}', ts):
            continue
        if re.fullmatch(r'\d+(?:\.\d+)?%', ts):
            continue
        cleaned.append(ts)
    return cleaned


def _pdf_line_to_values(line: str, years: list[int]):
    line = re.sub(r'\[[0-9*]+\]', ' ', str(line))
    toks = _pdf_line_amount_tokens(line)
    if len(toks) < len(years):
        return None, None

    value_toks = toks[-len(years):]
    values = []
    for tok in value_toks:
        val = numeric_or_none(tok)
        if val is None:
            return None, None
        values.append(float(val))

    label = str(line)
    for tok in value_toks:
        pos = label.rfind(tok)
        if pos != -1:
            label = label[:pos]
            break

    label = re.sub(r'\[[0-9*]+\]', '', label)
    label = re.sub(r'\s+', ' ', label).strip(' -:\t')
    if should_skip_label(label):
        return None, None
    return label, values


def detect_pdf_statement_section(line: str, current_section: str | None = None):
    low = str(line).strip().lower()
    if not low:
        return current_section

    exact_income = {'income statement', 'statement of operations', 'consolidated statement of operations'}
    exact_balance = {'balance sheet', 'balance sheets', 'statement of financial position', 'consolidated balance sheet'}
    exact_cash = {'cash flow', 'cash flows', 'statement of cash flows', 'consolidated statement of cash flows'}

    if low in exact_income:
        return 'income_statement'
    if low in exact_balance:
        return 'balance_sheet'
    if low in exact_cash:
        return 'cash_flow'

    short_line = len(low) <= 60
    if short_line and ('balance sheet' in low or 'financial position' in low):
        return 'balance_sheet'
    if short_line and ('cash flow' in low or 'cash flows' in low):
        return 'cash_flow'
    if short_line and ('income statement' in low or 'statement of operations' in low or 'profit and loss' in low or 'p&l' in low):
        return 'income_statement'
    return current_section


def detect_pdf_years_in_line(line: str, doc_year_hint: int | None = None):
    txt = str(line).strip()
    low = txt.lower()
    if not txt:
        return None
    if 'maturity year' in low or 'debt maturity' in low:
        return None
    if '%' in txt:
        return None

    years = []
    seen = set()
    for m in re.finditer(r"FY\s*[-\']?\s*(20\d{2}|\d{2})[A-Z]*", txt, flags=re.I):
        y = int(m.group(1))
        y = y if y >= 100 else 2000 + y
        if y not in seen:
            years.append(y)
            seen.add(y)

    if len(years) < 2:
        bare = re.findall(r"(?<!\d)(20\d{2})(?!\d)", txt)
        if len(bare) >= 2:
            for y in bare:
                yi = int(y)
                if yi not in seen:
                    years.append(yi)
                    seen.add(yi)

    if len(years) < 2:
        return None

    if doc_year_hint is not None:
        years = [y for y in years if y <= int(doc_year_hint) + 1 and y >= int(doc_year_hint) - 10]
        if len(years) < 2:
            return None

    return years


def extract_pdf_master(files_df: pd.DataFrame) -> pd.DataFrame:
    pdf_df = files_df[(files_df['include']) & (files_df['ext'] == '.pdf')].copy()
    rows = []

    try:
        import pdfplumber
    except Exception:
        return pd.DataFrame(columns=[
            'source_type', 'source_file', 'sheet_name', 'statement_type',
            'year', 'raw_line_item', 'line_item_norm', 'value_num', 'source_kind', 'unit_scale'
        ])

    for _, row in pdf_df.iterrows():
        try:
            with pdfplumber.open(row['path']) as pdf:
                doc_scale = 1.0
                doc_year_hint = row.get('year_hint')
                if pd.isna(doc_year_hint) or doc_year_hint is None:
                    try:
                        first_pages = ' '.join((pdf.pages[i].extract_text() or '') for i in range(min(3, len(pdf.pages))))
                        doc_year_hint = infer_year(first_pages)
                    except Exception:
                        doc_year_hint = None

                for page_num, page in enumerate(pdf.pages, start=1):
                    current_section = None
                    current_years = None
                    text = page.extract_text() or ''
                    if not text.strip():
                        continue

                    page_scale = detect_unit_scale(f"{row['name']} {text[:3000]}")
                    if page_scale != 1.0:
                        doc_scale = page_scale
                    scale = page_scale if page_scale != 1.0 else doc_scale

                    lines = [re.sub(r'\s+', ' ', ln).strip() for ln in text.splitlines() if str(ln).strip()]
                    for ln in lines:
                        maybe_section = detect_pdf_statement_section(ln, current_section)
                        current_section = maybe_section

                        maybe_years = detect_pdf_years_in_line(ln, doc_year_hint=doc_year_hint)
                        if maybe_years is not None:
                            current_years = maybe_years
                            continue

                        if current_section is None or current_years is None:
                            continue

                        if re.search(r'\bmargin\b|\bEPS\b|% of total debt|amount \(\$m\)', ln, flags=re.I):
                            continue
                        if ln.lower().startswith('section ') or ln.lower().startswith('note ') or ln.startswith('['):
                            continue

                        label, values = _pdf_line_to_values(ln, current_years)
                        if label is None or values is None:
                            continue

                        for year, val in zip(current_years, values):
                            rows.append({
                                'source_type': 'pdf',
                                'source_file': row['name'],
                                'sheet_name': f'page_{page_num}',
                                'statement_type': current_section,
                                'year': int(year),
                                'raw_line_item': label,
                                'line_item_norm': normalize_line_item(label),
                                'value_num': float(val) * scale,
                                'source_kind': 'pdf_primary_text_table',
                                'unit_scale': scale,
                            })
        except Exception as e:
            print(f"Skipped PDF {row['name']} because of parsing error: {e}")
            continue

    if not rows:
        return pd.DataFrame(columns=[
            'source_type', 'source_file', 'sheet_name', 'statement_type',
            'year', 'raw_line_item', 'line_item_norm', 'value_num', 'source_kind', 'unit_scale'
        ])

    out = pd.DataFrame(rows)
    out = out[out['year'].notna()].copy()
    out['year'] = out['year'].astype(int)
    out['line_item_norm'] = out['line_item_norm'].astype(str)
    out = out.drop_duplicates(subset=['source_file', 'sheet_name', 'statement_type', 'year', 'line_item_norm', 'value_num'])
    return out


def extract_pdf_text_metadata(files_df: pd.DataFrame) -> pd.DataFrame:
    pdf_df = files_df[(files_df["include"]) & (files_df["ext"] == ".pdf")].copy()
    rows = []

    try:
        import pdfplumber
    except Exception:
        return pd.DataFrame(columns=["source_file", "year", "page_no", "text"])

    for _, row in pdf_df.iterrows():
        try:
            with pdfplumber.open(row["path"]) as pdf:
                for i, page in enumerate(pdf.pages):
                    text = page.extract_text() or ""
                    if text.strip():
                        rows.append({
                            "source_file": row["name"],
                            "year": row.get("year_hint"),
                            "page_no": i + 1,
                            "text": text[:4000],
                        })
        except Exception:
            continue

    return pd.DataFrame(rows)


def extract_ppt_text_metadata(files_df: pd.DataFrame) -> pd.DataFrame:
    ppt_df = files_df[(files_df["include"]) & (files_df["ext"].isin([".ppt", ".pptx"]))].copy()
    rows = []

    try:
        from pptx import Presentation
    except Exception:
        return pd.DataFrame(columns=["source_file", "slide_no", "text"])

    for _, row in ppt_df.iterrows():
        try:
            prs = Presentation(row["path"])
            for i, slide in enumerate(prs.slides):
                texts = []
                for shape in slide.shapes:
                    if hasattr(shape, "text") and shape.text:
                        texts.append(shape.text)
                text = "\n".join(texts).strip()
                if text:
                    rows.append({
                        "source_file": row["name"],
                        "slide_no": i + 1,
                        "text": text[:4000],
                    })
        except Exception:
            continue

    return pd.DataFrame(rows)


def _safe_amount_from_text(token: str):
    token = str(token).strip()
    if token in {"", "-", "—", "–", "nan", "None"}:
        return 0.0
    token = token.replace("$", "").replace(",", "").replace(" ", "").strip()
    neg = False
    if token.startswith("(") and token.endswith(")"):
        neg = True
        token = token[1:-1]
    try:
        val = float(token)
        return -val if neg else val
    except Exception:
        return None


def extract_pdf_balance_sheet_fallback(files_df: pd.DataFrame) -> pd.DataFrame:
    """
    Targeted fallback for compiled annual financial PDFs.
    Only extracts a small set of balance sheet metrics when Excel statements do not carry them cleanly.
    Uses line-based parsing so it is safer on accountant-style annual statements.
    """
    pdf_df = files_df[(files_df["include"]) & (files_df["ext"] == ".pdf")].copy()
    rows = []

    try:
        import pdfplumber
    except Exception:
        return pd.DataFrame(columns=["source_file", "year", "statement_type", "line_item_norm", "value_num", "source_kind"])

    metric_patterns = {
        "accounts_receivable": [
            r"accounts receivable",
            r"trade receivables?",
        ],
        "inventory": [
            r"\binventory\b",
        ],
        "prepaids": [
            r"prepaid expenses?",
            r"prepayments?",
            r"\bprepaids?\b",
        ],
    }

    amt = r"(?:\(\s*(?:\d\s*)?\d{1,3},\d{3}(?:\.\d+)?\s*\)|[-—–]|(?:\d\s*)?\d{1,3},\d{3}(?:\.\d+)?)"

    def squash_ws(s: str) -> str:
        s = str(s)
        s = re.sub(r'\s+', ' ', s).strip()
        return s

    for _, row in pdf_df.iterrows():
        try:
            with pdfplumber.open(row["path"]) as pdf:
                for page in pdf.pages:
                    text = page.extract_text() or ""
                    if "balance sheet" not in text.lower():
                        continue

                    lines = [squash_ws(x) for x in text.splitlines() if str(x).strip()]
                    if not lines:
                        continue

                    year_line = None
                    for ln in lines[:12]:
                        if re.fullmatch(r'20\d{2}\s+20\d{2}', ln.strip()):
                            year_line = ln.strip()
                            break

                    if year_line:
                        ys = re.findall(r'20\d{2}', year_line)
                        current_year, prior_year = int(ys[0]), int(ys[1])
                    else:
                        yr_hint = row.get("year_hint")
                        if pd.notna(yr_hint):
                            current_year, prior_year = int(yr_hint), int(yr_hint) - 1
                        else:
                            continue

                    scale = detect_unit_scale(text)

                    for metric_name, patterns in metric_patterns.items():
                        found = False
                        for i, ln in enumerate(lines):
                            low = ln.lower()
                            if not any(re.search(p, low) for p in patterns):
                                continue

                            candidate = ln
                            nums = re.findall(amt, candidate)
                            if len(nums) < 2 and i + 1 < len(lines):
                                candidate = candidate + " " + lines[i + 1]
                                nums = re.findall(amt, candidate)

                            if len(nums) < 2:
                                continue

                            cur_tok, prv_tok = nums[-2], nums[-1]
                            cur = _safe_amount_from_text(cur_tok)
                            prv = _safe_amount_from_text(prv_tok)

                            if cur is not None:
                                rows.append({
                                    "source_file": row["name"],
                                    "year": current_year,
                                    "statement_type": "balance_sheet",
                                    "line_item_norm": metric_name,
                                    "value_num": cur * scale,
                                    "source_kind": "pdf_balance_sheet_fallback",
                                })
                            if prv is not None:
                                rows.append({
                                    "source_file": row["name"],
                                    "year": prior_year,
                                    "statement_type": "balance_sheet",
                                    "line_item_norm": metric_name,
                                    "value_num": prv * scale,
                                    "source_kind": "pdf_balance_sheet_fallback",
                                })
                            found = True
                            break
                        if found:
                            continue
        except Exception:
            continue

    out = pd.DataFrame(rows)
    if out.empty:
        return pd.DataFrame(columns=["source_file", "year", "statement_type", "line_item_norm", "value_num", "source_kind"])

    out = (
        out.sort_values(["year", "line_item_norm", "source_file"])
           .drop_duplicates(subset=["year", "line_item_norm"], keep="last")
           .reset_index(drop=True)
    )
    return out


# =========================
# MULTI-SHEET INDIAN EXCEL EXTRACTOR
# Handles analyst-style workbooks where:
#   - Years are column headers (not row headers)
#   - Row 0-3 are title/notes, header row has year labels
#   - Col 0 = line item label, Col 1 = unit (₹ Cr etc.), data starts col 2+
#   - Some cells contain "formula", "formula col", "── " section dividers
#   - Mixed year formats: FY14, FY-2018, 2018-19, FY23 etc.
# =========================

def normalise_indian_fy(raw):
    """Normalise messy Indian FY labels → integer year (ending year convention)."""
    import re
    # Handle float years like 2015.0 (pandas reads numeric year columns as float)
    try:
        fv = float(raw)
        if not (fv != fv) and 2000.0 <= fv <= 2035.0:  # not NaN and in range
            return int(fv)
    except (TypeError, ValueError):
        pass
    s = str(raw).strip()
    # Strip trailing .0 from float-as-string
    s = re.sub(r'\.0+$', '', s)
    m = re.match(r'FY[-\s]?(\d{4})$', s, re.I)
    if m: return int(m.group(1))
    m = re.match(r'FY[-\s]?(\d{2})$', s, re.I)
    if m: return 2000 + int(m.group(1))
    m = re.match(r'(\d{4})[-/](\d{2,4})$', s)
    if m:
        y2 = int(m.group(2)); base = int(m.group(1))
        return base + 1 if y2 < 100 else y2
    m = re.match(r'^(\d{4})$', s)
    if m:
        y = int(m.group(1))
        if 2000 <= y <= 2035: return y
    return None


_NOISE_TERMS = {
    'formula', '── ', 'source:', 'warning:', 'caution:', 'note:',
    'historical calc', 'management guidance', 'revenue /', 'formula;',
    'formula =', 'formula col', 'formula - ', 'should be 0',
    'cross-check', 'bscash', 'variance',
}

def _is_noise_cell(val):
    s = str(val).strip().lower()
    if s in ('nan', 'none', ''): return True
    return any(t in s for t in _NOISE_TERMS)


def _find_year_header_row(df, max_scan=6):
    """Find the row where year column headers appear (FY14, 2018-19, etc.)."""
    for i in range(min(max_scan, len(df))):
        row = df.iloc[i]
        year_cols = []
        for j, val in enumerate(row):
            y = normalise_indian_fy(val)
            if y is not None:
                year_cols.append((j, y))
        if len(year_cols) >= 3:
            return i, year_cols
    return None, []


def _detect_sheet_stmt_type(sheet_name: str, df: pd.DataFrame) -> str:
    """Detect statement type from sheet name or content."""
    name = sheet_name.lower()
    if any(k in name for k in ['p&l', 'pl', 'income', 'profit', 'revenue', 'pnl']):
        return 'income_statement'
    if any(k in name for k in ['bal', 'balance', 'bs', 'assets']):
        return 'balance_sheet'
    if any(k in name for k in ['cash', 'cf', 'cfs', 'cashflow']):
        return 'cash_flow'
    if any(k in name for k in ['kpi', 'ops', 'operational', 'ratio']):
        return 'operating_kpi'
    # Try content
    sample = ' '.join(df.head(5).fillna('').astype(str).values.flatten()).lower()
    if 'profit' in sample or 'revenue' in sample or 'ebitda' in sample:
        return 'income_statement'
    if 'assets' in sample or 'liabilit' in sample or 'equity' in sample:
        return 'balance_sheet'
    if 'cash flow' in sample or 'capex' in sample:
        return 'cash_flow'
    return None


def parse_indian_excel_workbook(file_path: Path) -> pd.DataFrame:
    """
    Extract financial data from multi-sheet Indian analyst Excel workbooks.
    Handles messy year formats, formula cells, unit columns, and section headers.
    """
    try:
        xls = pd.ExcelFile(file_path)
    except Exception as e:
        return pd.DataFrame()

    SKIP_SHEETS = ['readme', 'instructions', 'cover', 'contents', 'index',
                   'dcf_inputs', 'dcf inputs', 'wacc', 'assumptions', 'summary']

    rows = []
    for sheet_name in xls.sheet_names:
        if any(s in sheet_name.lower() for s in SKIP_SHEETS):
            continue

        try:
            df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
        except Exception:
            continue

        if df.empty:
            continue

        stmt_type = _detect_sheet_stmt_type(sheet_name, df)
        if stmt_type is None:
            continue

        header_row_idx, year_cols = _find_year_header_row(df)
        if not year_cols:
            continue

        unit_scale = detect_unit_scale(
            ' '.join(df.head(5).fillna('').astype(str).values.flatten())
        )

        for i in range(header_row_idx + 1, len(df)):
            row = df.iloc[i]
            label = str(row.iloc[0]).strip()

            # Skip blank, section dividers, formula rows
            if not label or label.lower() in ('nan', 'none'): continue
            if label.startswith('──') or label.startswith('—'): continue
            if _is_noise_cell(label): continue
            if len(label) > 120: continue

            for col_idx, year in year_cols:
                if col_idx >= len(row): continue
                val = row.iloc[col_idx]
                if _is_noise_cell(val): continue
                try:
                    fval = float(val)
                    if pd.isna(fval): continue
                    rows.append({
                        'source_type':    'excel',
                        'source_file':    file_path.name,
                        'sheet_name':     sheet_name,
                        'statement_type': stmt_type,
                        'year':           int(year),
                        'raw_line_item':  label,
                        'line_item_norm': normalize_line_item(label),
                        'value_num':      float(fval) * unit_scale,
                        'source_kind':    'indian_excel_multisheet',
                        'unit_scale':     unit_scale,
                    })
                except (ValueError, TypeError):
                    continue

    if not rows:
        return pd.DataFrame(columns=[
            'source_type', 'source_file', 'sheet_name', 'statement_type',
            'year', 'raw_line_item', 'line_item_norm', 'value_num',
            'source_kind', 'unit_scale'
        ])
    out = pd.DataFrame(rows)
    out['year'] = out['year'].astype(int)
    return out


def parse_wacc_from_excel_inputs(file_path: Path) -> dict:
    """
    Parse WACC inputs from a dedicated DCF inputs sheet in the workbook.
    Looks for sheets named: DCF_Inputs, WACC, Assumptions, DCF Inputs etc.
    """
    import re
    wacc = {}
    WACC_SHEETS = ['dcf_inputs', 'dcf inputs', 'wacc', 'assumptions',
                   'dcf_inputs_clean', 'inputs']
    PATTERNS = {
        'RISK_FREE':           r'risk.?free.?rate',
        'BETA':                r'\bbeta\b',
        'EQUITY_RISK_PREMIUM': r'(?:equity risk premium|market risk premium|erp)',
        'COST_OF_DEBT_PRETAX': r'(?:pre.?tax cost of debt|cost of debt)',
        'TAX_RATE':            r'(?:effective tax rate|tax rate)',
        'TARGET_DEBT_WEIGHT':  r'debt.{1,20}(?:debt \+ equity|total capital)',
    }
    try:
        xls = pd.ExcelFile(file_path)
    except Exception:
        return wacc

    for sheet_name in xls.sheet_names:
        if sheet_name.lower().replace(' ', '_') not in WACC_SHEETS:
            continue
        try:
            df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
        except Exception:
            continue

        for _, row in df.iterrows():
            label = str(row.iloc[0]).strip().lower()
            # Try columns 1, 2, 3 for the value
            val = None
            for c in range(1, min(4, len(row))):
                try:
                    v = float(row.iloc[c])
                    if not pd.isna(v) and v != 0:
                        val = v; break
                except Exception:
                    pass
            if val is None:
                continue
            for key, pat in PATTERNS.items():
                if key not in wacc and re.search(pat, label):
                    # Convert to percentage if stored as decimal (< 1.0)
                    if key != 'BETA' and key != 'TARGET_DEBT_WEIGHT' and val < 1.0:
                        val = val * 100.0
                    wacc[key] = round(val, 4)
                    break
    return wacc


# =========================
# CSV EXTRACTOR (for Indian / unlisted company packages)
# Handles messy year labels: FY15, FY-2017, 2018-19, FY23 etc.
# =========================

def normalise_indian_year(raw):
    """Normalise messy Indian FY labels to integer year (ending year convention)."""
    import re
    try:
        fv = float(raw)
        if not (fv != fv) and 2000.0 <= fv <= 2035.0:
            return int(fv)
    except (TypeError, ValueError):
        pass
    s = str(raw).strip()
    s = re.sub(r'\.0+$', '', s)
    # FY2024 or FY2023
    m = re.match(r'FY[-\s]?(\d{4})$', s, re.I)
    if m: return int(m.group(1))
    # FY24, FY-21
    m = re.match(r'FY[-\s]?(\d{2})$', s, re.I)
    if m: return 2000 + int(m.group(1))
    # 2018-19 → 2019 (Indian split year)
    m = re.match(r'(\d{4})[-/](\d{2,4})$', s)
    if m:
        y2 = int(m.group(2)); base = int(m.group(1))
        return base + 1 if y2 < 100 else y2
    # Plain 2024
    m = re.match(r'^(\d{4})$', s)
    if m:
        y = int(m.group(1))
        if 2000 <= y <= 2035: return y
    # Plain 15 → 2015
    m = re.match(r'^(\d{2})$', s)
    if m: return 2000 + int(m.group(1))
    return None


def extract_csv_master(files_df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract financial data from CSV files.
    Handles messy year labels and partial data (NaN values skipped).
    Statement type detected from filename keywords.
    """
    csv_df = files_df[(files_df["include"]) & (files_df["ext"] == ".csv")].copy()
    if csv_df.empty:
        return pd.DataFrame(columns=[
            "source_type","source_file","sheet_name","statement_type",
            "year","raw_line_item","line_item_norm","value_num","source_kind","unit_scale"
        ])

    IS_KEYWORDS  = ["income", "p&l", "profit", "pnl", "revenue", "earning"]
    BS_KEYWORDS  = ["balance", "financial position", "assets"]
    CF_KEYWORDS  = ["cash", "cashflow", "cash flow", "cfs"]
    YEAR_COLS    = {"year", "fy", "period", "date", "fiscal year", "financial year"}
    SKIP_COLS    = {"notes", "remarks", "comment", "unnamed", "flag", "adj"}

    rows = []
    for _, row in csv_df.iterrows():
        path = row["path"]
        name = row["name"].lower()

        # Detect statement type from filename
        if any(k in name for k in IS_KEYWORDS):
            stmt = "income_statement"
        elif any(k in name for k in BS_KEYWORDS):
            stmt = "balance_sheet"
        elif any(k in name for k in CF_KEYWORDS):
            stmt = "cash_flow"
        else:
            stmt = "unknown"

        try:
            df = pd.read_csv(path)
        except Exception as e:
            print(f"  Skipped CSV {row['name']}: {e}")
            continue

        if df.empty:
            continue

        # Find year column (case-insensitive)
        year_col = None
        for col in df.columns:
            if col.strip().lower() in YEAR_COLS:
                year_col = col; break

        if year_col is None:
            # Try first column as year if it looks like years
            first_col_vals = df.iloc[:, 0].astype(str).tolist()
            if any(normalise_indian_year(v) for v in first_col_vals[:3]):
                year_col = df.columns[0]

        if year_col is None:
            print(f"  Skipped {row['name']}: no year column found")
            continue

        # Metric columns — skip year col and noise cols
        metric_cols = [
            c for c in df.columns
            if c != year_col
            and not any(s in c.lower() for s in SKIP_COLS)
            and not c.lower().startswith("unnamed")
        ]

        for _, drow in df.iterrows():
            year = normalise_indian_year(drow[year_col])
            if year is None:
                continue

            for col in metric_cols:
                try:
                    fval = float(drow[col])
                    if pd.isna(fval):
                        continue
                    rows.append({
                        "source_type":    "csv",
                        "source_file":    row["name"],
                        "sheet_name":     row["name"].replace(".csv", ""),
                        "statement_type": stmt,
                        "year":           int(year),
                        "raw_line_item":  col,
                        "line_item_norm": col.strip().lower().replace("_", " "),
                        "value_num":      float(fval),
                        "source_kind":    "csv_annual",
                        "unit_scale":     1.0,
                    })
                except (ValueError, TypeError):
                    continue

    if not rows:
        return pd.DataFrame(columns=[
            "source_type","source_file","sheet_name","statement_type",
            "year","raw_line_item","line_item_norm","value_num","source_kind","unit_scale"
        ])

    out = pd.DataFrame(rows)
    out = out[out["year"].notna()].copy()
    out["year"] = out["year"].astype(int)
    return out


def parse_wacc_from_txt(files_df: pd.DataFrame) -> dict:
    """
    Parse WACC inputs from .txt files in the package.
    Returns dict of {variable_name: float_value}.
    """
    import re
    txt_df = files_df[(files_df["include"]) & (files_df["ext"] == ".txt")].copy()
    wacc = {}
    patterns = {
        "RISK_FREE":           r"risk\s*free\s*rate\s*[:\-=]\s*([\d.]+)",
        "BETA":                r"beta\s*[:\-=]\s*([\d.]+)",
        "EQUITY_RISK_PREMIUM": r"(?:market\s*risk\s*premium|equity\s*risk\s*premium|erp)\s*[:\-=]\s*([\d.]+)",
        "COST_OF_DEBT_PRETAX": r"cost\s*of\s*debt\s*[:\-=]\s*([\d.]+)",
        "TAX_RATE":            r"tax\s*rate\s*[:\-=]\s*([\d.]+)",
        "TERMINAL_GROWTH":     r"(?:terminal\s*growth|gordon\s*growth)\s*[:\-=]\s*([\d.]+)",
    }
    for _, row in txt_df.iterrows():
        try:
            text = Path(row["path"]).read_text(errors="ignore").lower()
            for key, pat in patterns.items():
                if key not in wacc:
                    m = re.search(pat, text)
                    if m:
                        wacc[key] = float(m.group(1))
        except Exception:
            continue
    return wacc


# =========================
# DCF MAPPING
# =========================

METRIC_PATTERNS = {
    "revenue_primary": [
        r"^total revenues?$", r"^revenues?$", r"^net sales$", r"^sales$",
        r"^total sales$", r"^service revenues?$", r"^contract revenues?$",
        r"^turnover$", r"^total turnover$", r"^revenue \$m$",
    ],
    "revenue_fallback": [
        r"revenue", r"sales", r"turnover", r"contract income", r"service income",
    ],
    "gross_profit": [
        r"^gross profit$", r"^gross margin$",
    ],
    "cogs": [
        r"total cost of sales", r"total cogs", r"cost of goods sold", r"cost of sales",
        r"cost of services", r"direct costs?", r"cost of contract revenue",
    ],
    "depreciation": [
        r"^depreciation expense$", r"depreciation expense", r"depreciation\b",
        r"depreciation and amorti[sz]ation", r"depreciation and amortisation",
        r"accum.*depreci", r"^d and a$", r"^d and a total$", r"^d\&a$", r"^d\&a total$",
    ],
    "amortization": [
        r"^amortization expense$", r"amortization expense", r"amortization\b",
        r"amortisation\b", r"depreciation and amorti[sz]ation", r"depreciation and amortisation",
        r"accum amort",
    ],
    "tax": [
        r"income tax expense", r"income tax provision", r"provision for income taxes",
        r"provision for taxes", r"tax provision", r"income taxes", r"corporate income tax",
        r"income tax corporate", r"current tax expense", r"taxes payable",
    ],
    "interest_expense": [
        r"interest expense", r"cc interest", r"bank charges .* interest",
        r"interest .*loan",
    ],
    "interest_income": [
        r"interest income",
    ],
    "net_income": [
        r"^net income$", r"^net income attributable", r"net earnings", r"net profit", r"net loss",
    ],
    "ebitda_reported": [
        r"^ebitda$", r"^ebitda reported$", r"^ebitda \(reported\)$",
    ],
    "ebitda_adjusted": [
        r"^adjusted ebitda$", r"^adj ebitda$", r"^ebitda adjusted$", r"^ebitda \(adjusted\)$",
    ],
    "operating_income": [
        r"operating income", r"income from operations", r"\bebit\b",
    ],
    "cash_from_operations": [
        r"cash from operations", r"cash provided by operating activities",
        r"net cash from operating activities", r"operating cash flow",
        r"cash flow from operations", r"cash generated from operations", r"net cash provided by operations", r"net cash provided by operating activities", r"provided by operations",
    ],
    "free_cash_flow": [
        r"^free cash flow$", r"free cash flow levered", r"levered free cash flow",
        r"free cash flow unlevered", r"unlevered free cash flow", r"fcf",
    ],
    "capex": [
        r"capital expenditures?", r"purchase of property", r"purchase of equipment",
        r"fixed asset additions?", r"purchase of fixed assets?",
        r"acquisition of property", r"acquisition of equipment",
        r"total capital expenditures?", r"subtotal.*capex", r"sub total.*capex",
    ],
    "cash": [
        r"^cash$", r"cash on hand", r"checking", r"bank",
        r"cash and cash equivalents?", r"cash \& cash equivalents",
    ],
    "total_assets": [
        r"^total assets$", r"assets total",
    ],
    "net_debt": [
        r"^net debt$", r"^net debt\s+m$", r"^net debt cash$", r"^net debt net cash$",
        r"^net debt \(cash\)$", r"net debt / net cash",
    ],
    "total_equity": [
        r"^total equity$", r"total shareholders? equity", r"stockholders? equity total",
    ],
    "shares_outstanding": [
        r"shares outstanding", r"shares outstanding m", r"shares outstanding mm",
        r"diluted shares", r"weighted average diluted shares",
        r"share count", r"basic shares outstanding", r"diluted shares outstanding",
    ],
    "total_current_assets": [
        r"total current assets",
    ],
    "total_current_liabilities": [
        r"total current liabilities",
    ],
    "current_debt": [
        r"current portion long term debt", r"short term debt", r"current maturities",
        r"credit line", r"line of credit", r"revolver", r"current notes payable",
        r"notes payable current", r"bank overdraft",
    ],
    "accounts_receivable": [
        r"accounts receivable", r"trade receivables?", r"customer receivables?",
        r"billed receivables?", r"receivables? net", r"trade receivables? \(net\)",
    ],
    "inventory": [
        r"^inventory\b", r"inventories", r"inventory materials",
        r"raw materials inventory", r"finished goods inventory", r"work in process",
        r"total inventories",
    ],
    "prepaids": [
        r"prepaid", r"prepayments?", r"prepaid expenses?",
    ],
    "accounts_payable": [
        r"accounts payable", r"trade payables?", r"vendor payables?",
        r"ap trade", r"visa payables?", r"amex payables?",
        r"credit card payables?", r"supplier payables?",
    ],
    "accrued_liabilities": [
        r"accrued", r"payroll taxes payable", r"federal payroll taxes payable",
        r"state payroll taxes payable", r"futa tax payable", r"suta tax payable",
        r"retirement contribution payabl", r"income taxes payable",
        r"sales tax payable", r"due to officer", r"customer deposits?",
        r"deferred revenue current", r"other current liabilities",
    ],
}


def _combine_patterns(patterns):
    return "|".join(f"(?:{p})" for p in patterns)


def value_for(df_year: pd.DataFrame, statement_type=None, include=None, exclude=None, how="first", positive_only=False):
    d = df_year.copy()

    if statement_type:
        valid_types = statement_type if isinstance(statement_type, list) else [statement_type]
        d = d[d["statement_type"].isin(valid_types)]

    if include:
        patterns = include if isinstance(include, list) else [include]
        mask = pd.Series(False, index=d.index)
        for p in patterns:
            mask |= d["line_item_norm"].str.contains(p, regex=True, na=False)
        d = d[mask]

    if exclude:
        patterns = exclude if isinstance(exclude, list) else [exclude]
        for p in patterns:
            d = d[~d["line_item_norm"].str.contains(p, regex=True, na=False)]

    if positive_only:
        d = d[d["value_num"] > 0]

    if d.empty:
        return None

    vals = d["value_num"].tolist()

    if how == "sum":
        return float(np.nansum(vals))
    if how == "maxabs":
        idx = d["value_num"].abs().idxmax()
        return float(d.loc[idx, "value_num"])
    if how == "last":
        return float(vals[-1])
    if how == "median":
        return float(np.nanmedian(vals))

    return float(vals[0])


def metric_value(df_year: pd.DataFrame, metric_name: str, statement_type, how="sum", exclude=None, positive_only=False):
    pats = METRIC_PATTERNS[metric_name]
    return value_for(
        df_year,
        statement_type=statement_type,
        include=_combine_patterns(pats),
        exclude=exclude,
        how=how,
        positive_only=positive_only,
    )


def metric_value_safe(df_year: pd.DataFrame, metric_name: str, statement_type, how="sum", exclude=None, positive_only=False):
    """Try the expected statement type first, then fall back to any statement type.
    This is safer for PDF-derived tables where a row can be classified slightly differently.
    """
    val = metric_value(df_year, metric_name, statement_type, how=how, exclude=exclude, positive_only=positive_only)
    if val is not None:
        return val
    pats = METRIC_PATTERNS[metric_name]
    return value_for(
        df_year,
        statement_type=None,
        include=_combine_patterns(pats),
        exclude=exclude,
        how=how,
        positive_only=positive_only,
    )


def preferred_balance_value(
    df_year: pd.DataFrame,
    include_patterns,
    exclude=None,
    positive_only=False,
):
    """Pick one ending balance safely without double counting near-duplicate rows."""
    if isinstance(include_patterns, str):
        include_patterns = [include_patterns]

    for pat in include_patterns:
        val = value_for(
            df_year,
            statement_type="balance_sheet",
            include=pat,
            exclude=exclude,
            how="first",
            positive_only=positive_only,
        )
        if val is not None:
            return val

    return value_for(
        df_year,
        statement_type="balance_sheet",
        include=_combine_patterns(include_patterns),
        exclude=exclude,
        how="maxabs",
        positive_only=positive_only,
    )


def preferred_tax_value(df_year: pd.DataFrame):
    """Prefer explicit annual tax expense/provision rows and avoid balance-sheet tax assets/liabilities."""
    explicit_patterns = [
        r"^income tax provision$",
        r"^income tax expense$",
        r"^provision for income taxes$",
        r"^tax provision$",
        r"^provision for taxes$",
        r"^current tax expense$",
        r"^income tax corporate$",
    ]
    candidates = []
    for pat in explicit_patterns:
        val = value_for(
            df_year,
            statement_type=["income_statement", "cash_flow", "operating_kpi"],
            include=pat,
            exclude=r"rate|receivable|asset|liabilit|payable|deferred",
            how="maxabs",
            positive_only=False,
        )
        if val is not None:
            candidates.append(val)
    if candidates:
        candidates = [v for v in candidates if pd.notna(v)]
        if candidates:
            non_zero = [v for v in candidates if abs(v) > 1e-9]
            return float(max(non_zero, key=lambda x: abs(x))) if non_zero else float(candidates[0])

    return metric_value_safe(
        df_year,
        "tax",
        ["income_statement", "cash_flow", "operating_kpi"],
        how="maxabs",
        exclude=r"rate|receivable|asset|liabilit|payable",
    )


def calc_gross_ppe(df_year: pd.DataFrame):
    total_ppe = value_for(
        df_year,
        statement_type="balance_sheet",
        include=r"total property and equipment|property plant and equipment|net property and equipment|property plant equipment",
        how="first",
    )
    accum_dep = value_for(
        df_year,
        statement_type="balance_sheet",
        include=r"accum.*depreci",
        how="sum",
    )

    if total_ppe is not None and accum_dep is not None:
        return total_ppe + abs(accum_dep)

    return None


def finalize_dcf_last_3_years_in_millions(
    dcf_long: pd.DataFrame,
    metric_col: str = "metric",
    year_col: str = "year",
    value_col: str = "value",
    decimals: int = 3,
):
    if dcf_long is None or dcf_long.empty:
        raise ValueError("dcf_long is empty.")

    df = dcf_long.copy()
    if metric_col not in df.columns or year_col not in df.columns or value_col not in df.columns:
        raise ValueError(f"Expected columns: {metric_col}, {year_col}, {value_col}. Found: {list(df.columns)}")

    df[year_col] = pd.to_numeric(df[year_col], errors="coerce")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df = df.dropna(subset=[year_col]).copy()
    df[year_col] = df[year_col].astype(int)

    latest_3_years = sorted(df[year_col].unique())[-3:]
    df = df[df[year_col].isin(latest_3_years)].copy()

    df[value_col] = (df[value_col] / 1_000_000.0).round(decimals)

    metric_order = [
        "revenue",
        "gross_profit",
        "ebitda_reported",
        "ebitda_adjusted",
        "ebit",
        "net_income",
        "depreciation_and_amortization",
        "tax",
        "cash_from_operations",
        "free_cash_flow",
        "capex",
        "cash",
        "total_assets",
        "net_debt",
        "total_equity",
        "shares_outstanding",
        "accounts_receivable",
        "inventory",
        "prepaids",
        "accounts_payable",
        "accrued_liabilities",
        "current_debt",
        "simplified_nwc",
    ]

    known = df[df[metric_col].isin(metric_order)].copy()
    unknown = df[~df[metric_col].isin(metric_order)].copy()

    if not known.empty:
        known[metric_col] = pd.Categorical(known[metric_col], categories=metric_order, ordered=True)
        known = known.sort_values([metric_col, year_col])

    if not unknown.empty:
        unknown = unknown.sort_values([metric_col, year_col])

    out_long = pd.concat([known, unknown], ignore_index=True)
    out_wide = out_long.pivot_table(index=metric_col, columns=year_col, values=value_col, aggfunc="first").reset_index()

    return out_long, out_wide, latest_3_years


def build_dcf_output(master_long: pd.DataFrame, n_years: int = 3, pdf_bs_fallback: pd.DataFrame | None = None):
    if master_long.empty:
        raise ValueError("No extracted Excel statement data found.")

    agg = (
        master_long
        .groupby(["year", "statement_type", "line_item_norm"], as_index=False)["value_num"]
        .sum()
    )

    years = sorted(agg["year"].dropna().astype(int).unique())
    if not years:
        raise ValueError("No valid years found after extraction.")

    selected_years = years[-n_years:]
    rows = []
    prev_gross_ppe = None

    pdf_bs_fallback = pdf_bs_fallback.copy() if isinstance(pdf_bs_fallback, pd.DataFrame) else pd.DataFrame()

    def fallback_balance_value(year: int, metric_name: str):
        if pdf_bs_fallback.empty:
            return None
        d = pdf_bs_fallback[
            (pd.to_numeric(pdf_bs_fallback["year"], errors="coerce") == year) &
            (pdf_bs_fallback["line_item_norm"] == metric_name)
        ]
        if d.empty:
            return None
        vals = pd.to_numeric(d["value_num"], errors="coerce").dropna()
        if vals.empty:
            return None
        return float(vals.iloc[-1])

    for year in selected_years:
        ydf = agg[agg["year"] == year].copy()

        revenue = value_for(
            ydf, "income_statement",
            include=_combine_patterns(METRIC_PATTERNS["revenue_primary"]),
            exclude=r"cost|growth|margin|yoy"
        )
        if revenue is None:
            revenue = value_for(
                ydf, "income_statement",
                include=_combine_patterns(METRIC_PATTERNS["revenue_fallback"]),
                exclude=r"cost|discount|tax|growth|margin|yoy",
                how="maxabs",
                positive_only=True,
            )

        gross_profit = metric_value(ydf, "gross_profit", "income_statement", how="first")
        if gross_profit is None:
            cogs = metric_value(ydf, "cogs", "income_statement", how="maxabs")
            gross_profit = (revenue - abs(cogs)) if revenue is not None and cogs is not None else None

        da = metric_value_safe(
            ydf,
            "depreciation",
            ["income_statement", "cash_flow", "operating_kpi"],
            how="maxabs",
        )
        if da is None:
            depreciation = metric_value_safe(ydf, "depreciation", ["income_statement", "cash_flow"], how="sum")
            amortization = metric_value_safe(ydf, "amortization", ["income_statement", "cash_flow"], how="sum")
            da = (depreciation or 0) + (amortization or 0)
        elif da == 0:
            da = None

        tax = preferred_tax_value(ydf)

        interest_exp = metric_value(
            ydf, "interest_expense", "income_statement", how="sum", exclude=r"income"
        )
        interest_inc = metric_value(ydf, "interest_income", "income_statement", how="sum") or 0

        net_income = metric_value_safe(ydf, "net_income", "income_statement", how="first")
        operating_income = metric_value(ydf, "operating_income", "income_statement", how="first")

        ebit = operating_income
        if ebit is None and net_income is not None:
            ebit = net_income + (tax or 0) + (interest_exp or 0) - (interest_inc or 0)

        ebitda_reported = metric_value(ydf, "ebitda_reported", "income_statement", how="first")
        ebitda_adjusted = metric_value(ydf, "ebitda_adjusted", "income_statement", how="first")
        if ebitda_reported is None and ebit is not None and da is not None:
            ebitda_reported = ebit + da
        if ebitda_adjusted is None:
            ebitda_adjusted = ebitda_reported

        cash_from_operations = metric_value_safe(ydf, "cash_from_operations", ["cash_flow", "operating_kpi"], how="maxabs")

        # Prefer explicit unlevered/UFCF rows first when KPI/support tabs provide them.
        free_cash_flow = value_for(
            ydf,
            statement_type="operating_kpi",
            include=[r"unlevered free cash flow", r"\bufcf\b"],
            how="first",
        )
        if free_cash_flow is None:
            free_cash_flow = metric_value_safe(ydf, "free_cash_flow", "operating_kpi", how="first")
        if free_cash_flow is None:
            free_cash_flow = metric_value_safe(ydf, "free_cash_flow", "cash_flow", how="first")

        capex_direct = value_for(
            ydf,
            statement_type=["cash_flow", "balance_sheet"],
            include=_combine_patterns(METRIC_PATTERNS["capex"]),
            how="maxabs",
            positive_only=False,
        )
        if capex_direct is not None:
            capex_direct = abs(capex_direct)
        gross_ppe = calc_gross_ppe(ydf)
        if capex_direct is not None and capex_direct != 0:
            capex = abs(capex_direct)
        elif gross_ppe is not None and prev_gross_ppe is not None:
            capex = max(0.0, gross_ppe - prev_gross_ppe)
        else:
            capex = None
        prev_gross_ppe = gross_ppe if gross_ppe is not None else prev_gross_ppe

        cash = metric_value(
            ydf, "cash", "balance_sheet", how="maxabs", exclude=r"credit|loan|line of credit"
        )
        total_assets = metric_value_safe(ydf, "total_assets", "balance_sheet", how="first")
        net_debt = metric_value_safe(ydf, "net_debt", ["balance_sheet", "operating_kpi"], how="first")
        total_equity = metric_value_safe(ydf, "total_equity", "balance_sheet", how="first")
        shares_outstanding = metric_value_safe(ydf, "shares_outstanding", ["balance_sheet", "operating_kpi"], how="first")
        total_ca = metric_value(ydf, "total_current_assets", "balance_sheet", how="first")
        total_cl = metric_value(ydf, "total_current_liabilities", "balance_sheet", how="first")
        current_debt = metric_value(ydf, "current_debt", "balance_sheet", how="sum")

        ar = preferred_balance_value(
            ydf,
            include_patterns=[
                r"^accounts receivable net$",
                r"^trade receivables net$",
                r"^accounts receivable$",
                r"^trade receivables?$",
            ],
            exclude=r"income tax receivable|other receivable|intercompany|related party",
            positive_only=True,
        )
        if ar is None:
            ar = fallback_balance_value(year, "accounts_receivable")

        inventory = preferred_balance_value(
            ydf,
            include_patterns=[
                r"^inventories$",
                r"^inventory$",
                r"^total inventory$",
            ],
            positive_only=True,
        )
        if inventory is None:
            inventory = fallback_balance_value(year, "inventory")

        prepaid = preferred_balance_value(
            ydf,
            include_patterns=[
                r"^prepaid and other current assets$",
                r"^prepaid expenses?$",
                r"^prepayments?$",
                r"^prepaids?$",
            ],
            positive_only=True,
        )
        if prepaid is None:
            prepaid = fallback_balance_value(year, "prepaids")

        ap = preferred_balance_value(
            ydf,
            include_patterns=[
                r"^accounts payable$",
                r"^trade payables?$",
                r"^vendor payables?$",
                r"^supplier payables?$",
                r"^visa payables?$",
                r"^amex payables?$",
                r"^credit card payables?$",
            ],
            exclude=r"days payable|dpo|change in payables",
            positive_only=True,
        )
        accrued = metric_value(
            ydf, "accrued_liabilities", "balance_sheet", how="sum",
            exclude=_combine_patterns(METRIC_PATTERNS["accounts_payable"] + METRIC_PATTERNS["current_debt"])
        )

        simplified_nwc = None
        if all(x is not None for x in [ar, inventory, prepaid, ap]):
            simplified_nwc = (ar or 0) + (inventory or 0) + (prepaid or 0) - (ap or 0) - (accrued or 0)
        elif total_ca is not None and total_cl is not None:
            simplified_nwc = (total_ca - (cash or 0)) - (total_cl - (current_debt or 0))

        rows.extend([
            {"metric": "revenue", "year": year, "value": revenue},
            {"metric": "gross_profit", "year": year, "value": gross_profit},
            {"metric": "ebitda_reported", "year": year, "value": ebitda_reported},
            {"metric": "ebitda_adjusted", "year": year, "value": ebitda_adjusted},
            {"metric": "ebit", "year": year, "value": ebit},
            {"metric": "net_income", "year": year, "value": net_income},
            {"metric": "depreciation_and_amortization", "year": year, "value": da},
            {"metric": "tax", "year": year, "value": tax},
            {"metric": "cash_from_operations", "year": year, "value": cash_from_operations},
            {"metric": "free_cash_flow", "year": year, "value": free_cash_flow},
            {"metric": "capex", "year": year, "value": capex},
            {"metric": "cash", "year": year, "value": cash},
            {"metric": "total_assets", "year": year, "value": total_assets},
            {"metric": "net_debt", "year": year, "value": net_debt},
            {"metric": "total_equity", "year": year, "value": total_equity},
            {"metric": "shares_outstanding", "year": year, "value": shares_outstanding},
            {"metric": "accounts_receivable", "year": year, "value": ar},
            {"metric": "inventory", "year": year, "value": inventory},
            {"metric": "prepaids", "year": year, "value": prepaid},
            {"metric": "accounts_payable", "year": year, "value": ap},
            {"metric": "accrued_liabilities", "year": year, "value": accrued},
            {"metric": "current_debt", "year": year, "value": current_debt},
            {"metric": "simplified_nwc", "year": year, "value": simplified_nwc},
        ])

    dcf_long = pd.DataFrame(rows)
    dcf_wide = dcf_long.pivot(index="metric", columns="year", values="value").sort_index()

    return {
        "selected_years": selected_years,
        "dcf_long": dcf_long,
        "dcf_wide": dcf_wide,
    }


REQUIRED_METRICS = [
    "revenue",
    "gross_profit",
    "ebitda_reported",
    "ebitda_adjusted",
    "ebit",
    "net_income",
    "depreciation_and_amortization",
    "tax",
    "cash_from_operations",
    "free_cash_flow",
    "capex",
    "cash",
    "total_assets",
    "net_debt",
    "total_equity",
    "shares_outstanding",
    "accounts_receivable",
    "inventory",
    "prepaids",
    "accounts_payable",
    "accrued_liabilities",
    "current_debt",
    "simplified_nwc",
]

def build_metric_coverage_report(master_long: pd.DataFrame, dcf_long: pd.DataFrame) -> pd.DataFrame:
    """
    Shows whether each metric has extracted final values and whether related source rows exist.
    This helps distinguish:
    - missing because source truly lacks the metric
    - missing because mapping rules need more work
    """
    if dcf_long is None or dcf_long.empty:
        return pd.DataFrame()

    years = sorted(pd.to_numeric(dcf_long["year"], errors="coerce").dropna().astype(int).unique())
    rows = []

    source_df = master_long.copy() if isinstance(master_long, pd.DataFrame) else pd.DataFrame()
    if not source_df.empty:
        source_df["year"] = pd.to_numeric(source_df["year"], errors="coerce").astype("Int64")
        source_df["line_item_norm"] = source_df["line_item_norm"].astype(str)

    availability_patterns = {
        "depreciation_and_amortization": [r"depreciation", r"amorti", r"d and a", r"d&a"],
        "cash_from_operations": METRIC_PATTERNS["cash_from_operations"],
        "free_cash_flow": METRIC_PATTERNS["free_cash_flow"],
        "net_debt": METRIC_PATTERNS["net_debt"],
        "shares_outstanding": METRIC_PATTERNS["shares_outstanding"],
        "accounts_receivable": METRIC_PATTERNS["accounts_receivable"],
        "accounts_payable": METRIC_PATTERNS["accounts_payable"],
        "tax": METRIC_PATTERNS["tax"] + [r"income tax corporate"],
    }

    for metric in REQUIRED_METRICS:
        for year in years:
            out = dcf_long[(dcf_long["metric"] == metric) & (pd.to_numeric(dcf_long["year"], errors="coerce") == year)]
            final_value = out["value"].iloc[0] if not out.empty else np.nan
            extracted = pd.notna(final_value)

            source_available = None
            matching_labels = []
            if not source_df.empty:
                pats = availability_patterns.get(metric)
                if pats:
                    yrdf = source_df[source_df["year"] == year]
                    mask = pd.Series(False, index=yrdf.index)
                    for p in pats:
                        mask |= yrdf["line_item_norm"].str.contains(p, regex=True, na=False)
                    hits = yrdf[mask]
                    source_available = not hits.empty
                    matching_labels = sorted(hits["line_item_norm"].dropna().astype(str).unique().tolist())[:5]

            rows.append({
                "metric": metric,
                "year": year,
                "final_value_present": extracted,
                "source_rows_detected": source_available,
                "sample_source_labels": "; ".join(matching_labels),
            })

    return pd.DataFrame(rows)


# ── Compiled-PDF override helpers (patch layer) ──────────────────────────────

# =========================
# SAFE POST-PROCESS PATCH — compiled annual PDF overrides
# This does NOT change the main extraction logic.
# It only corrects select metrics when accountant-style compiled annual PDFs are present.
# =========================

def _patch_parse_last_two_amounts(line: str):
    s = str(line).replace("$", " ").replace("(", " (").replace(")", " ) ")
    s = re.sub(r"(?<=\d)\s*,\s*(?=\d)", ",", s)
    toks = s.split()

    numt = []
    i = 0
    while i < len(toks):
        t = toks[i]
        if t == "(":
            buf = []
            i += 1
            while i < len(toks) and toks[i] != ")":
                buf.append(toks[i])
                i += 1
            merged = "".join(buf)
            if merged:
                numt.append(f"({merged})")
            i += 1
            continue

        if re.fullmatch(r"[-–—]", t):
            numt.append("-")
            i += 1
            continue

        if re.search(r"\d", t) or t.startswith(","):
            numt.append(t)

        i += 1

    merged = []
    i = 0
    while i < len(numt):
        t = numt[i]

        if t.startswith(",") and merged:
            merged[-1] += t
            i += 1
            continue

        if re.fullmatch(r"\d{1,2}", t) and i + 1 < len(numt):
            n = numt[i + 1]
            if n.startswith(",") or re.fullmatch(r"\d{1,3},\d{3}(?:\.\d+)?", n):
                merged.append(t + n)
                i += 2
                continue
            if re.fullmatch(r"\d{2,3}", n):
                merged.append(t + n)
                i += 2
                continue

        merged.append(t)
        i += 1

    vals = []
    for t in merged:
        if t in {"-", "–", "—"}:
            vals.append(0.0)
            continue
        neg = t.startswith("(") and t.endswith(")")
        tt = t.strip("()").replace(",", "")
        if re.fullmatch(r"-?\d+(?:\.\d+)?", tt):
            v = float(tt)
            vals.append(-v if neg else v)

    return vals[-2:]


def extract_compiled_pdf_current_year_metrics(files_df: pd.DataFrame) -> pd.DataFrame:
    """
    Targeted parser for accountant-style compiled annual PDFs.
    Uses ONLY the current-year value from each annual compiled PDF.
    This avoids polluted prior-year carryover rows and fixes KGI-style private-company packages.
    """
    pdf_df = files_df[(files_df["include"]) & (files_df["ext"] == ".pdf")].copy()
    if pdf_df.empty:
        return pd.DataFrame(columns=["year", "metric", "value", "source_file", "source_kind"])

    try:
        import pdfplumber
    except Exception:
        return pd.DataFrame(columns=["year", "metric", "value", "source_file", "source_kind"])

    bs_patterns = {
        "accounts_receivable": [r"^Accounts Receivable(?:, Net)?\b"],
        "total_assets": [r"^TOTAL ASSETS\b"],
        "total_equity": [r"^Total Stockholders'? Equity\b", r"^Total Equity\b"],
        "accounts_payable": [r"^Accounts Payable\b"],
    }
    cf_patterns = {
        "cash_from_operations": [r"^Net Cash Flows From Operating Activities\b", r"^Net Cash Provided By Operating Activities\b"],
        "depreciation_and_amortization": [r"^Depreciation(?: and Amortization)?\b", r"^Depreciation & Amorti[sz]ation\b"],
        "capex": [r"^Purchase of Property and Equipment\b", r"^Capital Expenditures?\b"],
    }

    rows = []
    for _, row in pdf_df.iterrows():
        name_l = str(row.get("name", "")).lower()
        if not any(k in name_l for k in ["comp", "compiled", "compilation"]):
            continue

        year_hint = row.get("year_hint")
        if pd.isna(year_hint):
            continue
        year_hint = int(year_hint)

        try:
            with pdfplumber.open(row["path"]) as pdf:
                for page in pdf.pages:
                    text = page.extract_text() or ""
                    if not text.strip():
                        continue

                    low = text.lower()
                    is_bs = "balance sheet" in low
                    is_cf = "statement of cash flows" in low or "statement of cash flow" in low
                    if not (is_bs or is_cf):
                        continue

                    patterns = bs_patterns if is_bs else cf_patterns
                    lines = [ln.strip() for ln in text.splitlines() if str(ln).strip()]

                    for ln in lines:
                        for metric, plist in patterns.items():
                            if any(re.search(p, ln, flags=re.I) for p in plist):
                                vals = _patch_parse_last_two_amounts(ln)
                                if not vals:
                                    continue
                                current_val = vals[0]
                                if metric == "capex":
                                    current_val = abs(current_val)

                                rows.append({
                                    "year": year_hint,
                                    "metric": metric,
                                    "value": current_val,
                                    "source_file": row["name"],
                                    "source_kind": "compiled_pdf_current_year_override",
                                })
                                break
        except Exception:
            continue

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out["year"] = pd.to_numeric(out["year"], errors="coerce").astype("Int64")
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out = out.dropna(subset=["year", "value"]).copy()
    out["year"] = out["year"].astype(int)

    # Keep one clean value per year/metric. Since each annual compiled PDF contributes its own current-year value,
    # this is safe and avoids prior-year carryover contamination.
    out = out.sort_values(["year", "metric", "source_file"]).drop_duplicates(["year", "metric"], keep="last")
    return out


def apply_compiled_pdf_overrides(result: dict, compiled_override_df: pd.DataFrame, config: dict) -> dict:
    """
    Safe override layer:
    - does not change the main extraction engine
    - only updates select private-company metrics when compiled annual PDF values exist
    """
    if not isinstance(result, dict) or "dcf_long" not in result:
        return result
    if compiled_override_df is None or compiled_override_df.empty:
        return result

    override_metrics = {
        "accounts_receivable",
        "accounts_payable",
        "total_assets",
        "total_equity",
        "cash_from_operations",
        "depreciation_and_amortization",
        "capex",
    }

    dcf_long = result["dcf_long"].copy()
    dcf_long["year"] = pd.to_numeric(dcf_long["year"], errors="coerce").astype("Int64")
    dcf_long["value"] = pd.to_numeric(dcf_long["value"], errors="coerce")

    compiled_override_df = compiled_override_df.copy()
    compiled_override_df = compiled_override_df[compiled_override_df["metric"].isin(override_metrics)].copy()

    for _, r in compiled_override_df.iterrows():
        y = int(r["year"])
        m = str(r["metric"])
        v = float(r["value"])

        mask = (dcf_long["year"] == y) & (dcf_long["metric"] == m)
        if mask.any():
            dcf_long.loc[mask, "value"] = v
        else:
            dcf_long = pd.concat([
                dcf_long,
                pd.DataFrame([{"metric": m, "year": y, "value": v}])
            ], ignore_index=True)

    dcf_long["year"] = dcf_long["year"].astype(int)

    dcf_wide = (
        dcf_long
        .pivot_table(index="metric", columns="year", values="value", aggfunc="first")
        .sort_index()
    )

    result["dcf_long"] = dcf_long
    result["dcf_wide"] = dcf_wide
    result["compiled_pdf_overrides"] = compiled_override_df

    if "metric_coverage_report" in globals():
        try:
            result["metric_coverage_report"] = metric_coverage_report(master_long, dcf_long)
        except Exception:
            pass

    if config.get("DISPLAY_IN_MILLIONS", True) and "finalize_dcf_last_3_years_in_millions" in globals():
        try:
            dcf_long_last3_mn, dcf_wide_last3_mn, latest_3_years = finalize_dcf_last_3_years_in_millions(
                dcf_long=result["dcf_long"],
                metric_col="metric",
                year_col="year",
                value_col="value",
                decimals=config.get("ROUND_MILLIONS_TO", 3),
            )
            result["dcf_long_millions"] = dcf_long_last3_mn
            result["dcf_wide_millions"] = dcf_wide_last3_mn
            result["selected_years_millions"] = latest_3_years
        except Exception:
            pass

    return result


# NOTE: compiled_pdf_override_df and apply_compiled_pdf_overrides() are called
# inside merge_sources() below — NOT here. These functions are just defined above.
# Running them here would fail because files_df and result don't exist yet.

print("✅ File extraction engine ready (Excel / PDF / PPT).")


✅ File extraction engine ready (Excel / PDF / PPT).


# Part 3 — Data Fetch & Historical Build

In [5]:
# ── 3A: Resolve TICKER ───────────────────────────────────────────────────────
_ti = str(globals().get("TICKER_INPUT", "")).strip().upper()
if not _ti:
    # fallback to run_meta.json
    try:
        _meta = json.loads(Path("run_meta.json").read_text())
        _ti = str(_meta.get("ticker", "")).strip().upper()
    except Exception:
        pass
if not _ti:
    raise ValueError("TICKER_INPUT is empty. Set it in the Master Config cell.")

TICKER = _ti
# Accept injected values from papermill; fall back to suffix detection
_suffix_ccy_map = {".NS": "INR", ".BO": "INR", ".L": "GBP", ".PA": "EUR", ".DE": "EUR", ".T": "JPY", ".HK": "HKD", ".AX": "AUD"}
_detected_ccy = next((ccy for sfx, ccy in _suffix_ccy_map.items() if TICKER.endswith(sfx)), "USD")
LOCAL_CURRENCY = globals().get("LOCAL_CURRENCY") or _detected_ccy
_region_map = {"INR": "India", "USD": "US", "GBP": "GB", "EUR": "EU", "JPY": "Japan", "HKD": "HK", "AUD": "AU"}
REGION = _region_map.get(LOCAL_CURRENCY, "US")
print(f"✅ TICKER = {TICKER}  |  Currency = {LOCAL_CURRENCY}  |  Region = {REGION}")

# ── 3B: Company profile from yfinance ────────────────────────────────────────
company_profile = {}
if SOURCES.get("api"):
    try:
        _info = yf.Ticker(TICKER).info or {}
        company_profile = {
            "ticker":   TICKER,
            "region":   REGION,
            "currency": _info.get("currency", LOCAL_CURRENCY),
            "sector":   _info.get("sector"),
            "industry": _info.get("industry"),
        }
        print(f"✅ Profile: {company_profile.get('sector')} / {company_profile.get('industry')}")
    except Exception as _e:
        print(f"⚠️  Profile fetch failed: {_e}")


✅ TICKER = RELIANCE.NS  |  Currency = INR  |  Region = India
✅ Profile: Energy / Oil & Gas Refining & Marketing


In [6]:
# ── 3C: yfinance statement fetchers ──────────────────────────────────────────

def _to_numeric_df(df):
    if df is None or df.empty: return pd.DataFrame()
    out = df.copy()
    for c in out.columns:
        if c != "FY":
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def _tidy_yf_statement(df):
    if df is None or df.empty: return pd.DataFrame()
    t = df.T.copy()
    try:
        yrs = pd.to_datetime(t.index).year
    except Exception:
        yrs = pd.Index([int(str(x)[:4]) for x in t.index])
    t.insert(0, "FY", yrs.astype(int))
    return _to_numeric_df(t.reset_index(drop=True))

def fetch_yf_statements(ticker):
    tk = yf.Ticker(ticker)
    is_df = _tidy_yf_statement(tk.financials)
    bs_df = _tidy_yf_statement(tk.balance_sheet)
    cf_df = _tidy_yf_statement(tk.cashflow)
    info  = {}
    try: info = tk.info or {}
    except Exception: pass

    # Shares table
    sh_out = _to_numeric_df(pd.DataFrame({
        "FY": [np.nan],
        "SharesOutstanding": [info.get("sharesOutstanding")],
        "MarketCap":         [info.get("marketCap")],
        "Price":             [info.get("currentPrice") or info.get("regularMarketPrice")],
    }))

    # Fallback to quarterly TTM if annual empty
    if is_df.empty:
        try:
            iq = _tidy_yf_statement(tk.quarterly_financials)
            if not iq.empty:
                is_df = iq.sort_values("FY").groupby("FY").sum(numeric_only=True).reset_index()
                print("   ℹ️  Using quarterly IS (annual empty)")
        except Exception: pass
    if bs_df.empty:
        try:
            bq = _tidy_yf_statement(tk.quarterly_balance_sheet)
            if not bq.empty:
                bs_df = bq.sort_values("FY").groupby("FY").last(numeric_only=True).reset_index()
                print("   ℹ️  Using quarterly BS (annual empty)")
        except Exception: pass
    if cf_df.empty:
        try:
            cq = _tidy_yf_statement(tk.quarterly_cashflow)
            if not cq.empty:
                cf_df = cq.sort_values("FY").groupby("FY").sum(numeric_only=True).reset_index()
                print("   ℹ️  Using quarterly CF (annual empty)")
        except Exception: pass

    # quarterly fallback handled above

    return is_df, bs_df, cf_df, sh_out

def normalize_units(df, unit_factor):
    if df is None or df.empty: return df
    out = df.copy()
    for c in out.columns:
        if c != "FY":
            out[c] = pd.to_numeric(out[c], errors="coerce") / unit_factor
    return out

# Unit factor
unit_factor = 1.0
_u = str(CURRENCY_UNITS).lower()
if   _u.startswith("billion"):  unit_factor = 1_000_000_000.0
elif _u.startswith("million"):  unit_factor = 1_000_000.0
elif _u.startswith("thousand"): unit_factor = 1_000.0
elif _u.startswith("dollar"):   unit_factor = 1.0  # raw dollars, no division

# Run fetch
_api_on = SOURCES.get("api", True)
if _api_on:
    IS, BS, CF, SH = fetch_yf_statements(TICKER)
    IS = normalize_units(IS, unit_factor)
    BS = normalize_units(BS, unit_factor)
    CF = normalize_units(CF, unit_factor)
    # Validation: show what was actually fetched
    _is_rows = len(IS) if not IS.empty else 0
    _bs_rows = len(BS) if not BS.empty else 0
    _cf_rows = len(CF) if not CF.empty else 0
    print(f"✅ yfinance fetch complete | IS:{_is_rows}yr BS:{_bs_rows}yr CF:{_cf_rows}yr")
    if not IS.empty:
        _fy_list = sorted(IS["FY"].dropna().astype(int).tolist())
        _rev_col = next((c for c in ["Total Revenue","Revenue"] if c in IS.columns), None)
        if _rev_col:
            _rev_vals = IS.set_index("FY")[_rev_col].to_dict()
            print(f"   Revenue (in {CURRENCY_UNITS}): { {k:f'{v:,.0f}' for k,v in _rev_vals.items()} }")
        print(f"   IS fields: {[c for c in IS.columns if c != 'FY'][:8]}...")
        print(f"   BS fields: {[c for c in BS.columns if c != 'FY'][:8]}..." if not BS.empty else "   BS: empty")
else:
    IS = BS = CF = SH = pd.DataFrame()
    print("ℹ️  API off — skipping yfinance fetch.")


✅ yfinance fetch complete | IS:4yr BS:4yr CF:4yr
   Revenue (in Millions): {2025: '9,646,930', 2024: '9,010,640', 2023: '8,778,350', 2022: '6,959,630'}
   IS fields: ['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Normalized EBITDA', 'Total Unusual Items', 'Total Unusual Items Excluding Goodwill', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Reconciled Cost Of Revenue']...
   BS fields: ['Ordinary Shares Number', 'Share Issued', 'Net Debt', 'Total Debt', 'Tangible Book Value', 'Invested Capital', 'Working Capital', 'Net Tangible Assets']...


In [7]:
# ── 3D: build_historical (from yfinance statements) ─────────────────────────

def coalesce(df, candidates):
    for c in candidates:
        if c in df.columns: return df[c]
    return pd.Series(np.nan, index=df.index)

def build_historical(IS, BS, CF, SH):
    """
    Build unified HIST DataFrame from yfinance IS/BS/CF statements.
    All field names match yfinance 0.2.x actual output.
    Values are already normalized to CURRENCY_UNITS by the caller.
    """
    fys = set()
    for d in (IS, BS, CF):
        if d is not None and not d.empty and "FY" in d.columns:
            fys |= set(pd.to_numeric(d["FY"], errors="coerce").dropna().astype(int).tolist())
    if not fys: return pd.DataFrame()

    base = pd.DataFrame({"FY": sorted(fys)})
    out  = base.merge(IS, on="FY", how="left") if not IS.empty else base
    if not BS.empty: out = out.merge(BS, on="FY", how="left")
    if not CF.empty: out = out.merge(CF, on="FY", how="left")
    if not SH.empty: out = out.merge(SH, on="FY", how="left")

    def _c(candidates):
        """Return first column found in out, or NaN Series."""
        for name in candidates:
            if name in out.columns:
                return out[name]
        return pd.Series(np.nan, index=out.index)

    # ── Revenue ────────────────────────────────────────────────────────────────
    # yfinance: "Total Revenue" (most), "Revenue" (some)
    out["Revenue"] = _c(["Total Revenue", "Revenue"])

    # ── EBIT ───────────────────────────────────────────────────────────────────
    # Financial company detection: COGS > 40% of revenue means borrowing cost
    # is booked as COGS (NBFC pattern like IRFC, HDFC, power finance companies).
    pretax   = _c(["Pretax Income", "Income Before Tax"])
    _ni_for_ebit = _c(["Net Income", "Net Income Common Stockholders"])
    _tax_for_ebit = _c(["Tax Provision", "Income Tax Expense"]).abs()
    _cogs_s  = _c(["Cost Of Revenue", "Cost of Revenue"])
    _rev_s   = out["Revenue"]
    _is_financial = (
        _cogs_s.notna() & _rev_s.notna() &
        (_cogs_s.abs() > _rev_s.abs() * 0.40)
    )
    ebit_raw = _c(["Ebit", "Operating Income"])
    # For financial companies: EBIT = NI / (1 - tax_rate)
    # Most accurate when yfinance Tax Provision = 0 (common for Indian NBFCs)
    # Falls back to NI + Tax when tax is properly reported
    _tr_local   = globals().get("TAX_RATE", 25.0) / 100.0
    _ebit_ni_tr = _ni_for_ebit / max(1 - _tr_local, 0.01)
    _ebit_ni_tax = _ni_for_ebit.fillna(0) + _tax_for_ebit.fillna(0)
    # Use NI/(1-t) when tax ratio < 1% of NI (yfinance Tax = 0 case)
    _tax_ratio  = (_tax_for_ebit.fillna(0) / _ni_for_ebit.abs().replace(0, np.nan)).fillna(0)
    _ebit_fin   = _ebit_ni_tr.where(_tax_ratio < 0.01, _ebit_ni_tax)
    out["EBIT"] = ebit_raw.copy()
    _fin_valid  = _ebit_fin.notna() & (_ebit_fin > 0) & _is_financial
    out.loc[_fin_valid, "EBIT"] = _ebit_fin[_fin_valid]
    out["EBIT"] = out["EBIT"].fillna(pretax)

    # ── D&A ────────────────────────────────────────────────────────────────────
    # yfinance 0.2.x: "Reconciled Depreciation" is the primary IS field
    # Also try CF statement D&A add-back which is more reliable
    da_is = _c(["Reconciled Depreciation",
                "Depreciation Amortization Depletion Income Statement",
                "Depreciation And Amortization",
                "Depreciation"])
    da_cf = _c(["Depreciation Amortization Depletion",
                "Depreciation And Amortization In Cash Flow",
                "Depreciation"])
    # Prefer IS D&A, fallback to CF D&A
    out["DandA"] = da_is.fillna(da_cf)
    # D&A should be positive — abs() to handle sign inconsistencies
    out["DandA"] = out["DandA"].abs()

    # ── CapEx ──────────────────────────────────────────────────────────────────
    # yfinance CF "Capital Expenditure" is NEGATIVE (cash outflow)
    # We store Capex as POSITIVE (magnitude of spend)
    capex_raw = _c(["Capital Expenditure",
                    "Capital Expenditures",
                    "Purchase Of Ppe",
                    "Purchases Of Property Plant And Equipment",
                    "Capital Expenditures Reported"])
    out["Capex"] = capex_raw.abs()  # always positive magnitude

    # ── Working Capital ────────────────────────────────────────────────────────
    ca   = _c(["Current Assets",  "Total Current Assets"])
    cash = _c(["Cash And Cash Equivalents",
               "Cash Cash Equivalents And Short Term Investments",
               "Cash And Cash Equivalents And Short Term Investments",
               "Cash"])
    cl   = _c(["Current Liabilities", "Total Current Liabilities"])
    std  = _c(["Short Long Term Debt",          # yfinance primary name
               "Current Debt",
               "Short Term Debt",
               "Current Portion Of Long Term Debt",
               "Short-term Debt"])
    out["NWC"]      = (ca - cash) - (cl - std.fillna(0))
    out["DeltaNWC"] = out["NWC"].diff()

    # ── FCFF ───────────────────────────────────────────────────────────────────
    # Method 1: CFO - Capex (more reliable, especially for financial companies)
    cfo_raw = _c(["Operating Cash Flow",
                  "Cash Flows From Operating Activities",
                  "Net Cash Provided By Operating Activities",
                  "Cash From Operations"])
    out["CFO"] = cfo_raw

    # Method 2: NOPAT + DandA - Capex - DeltaNWC (standard DCF approach)
    tr = TAX_RATE / 100.0
    out["NOPAT"] = out["EBIT"] * (1 - tr)
    fcff_nopat = (out["NOPAT"].fillna(0) + out["DandA"].fillna(0)
                  - out["Capex"].fillna(0) - out["DeltaNWC"].fillna(0))
    # Use CFO-based FCFF when DeltaNWC is extreme (financial companies)
    # Threshold: |DeltaNWC| > Revenue → likely financial company distortion
    _nwc_extreme = (out["DeltaNWC"].abs() > out["Revenue"].abs() * 0.5) if "Revenue" in out.columns else pd.Series(False, index=out.index)
    fcff_cfo = out["CFO"].fillna(0) - out["Capex"].fillna(0)
    out["FCFF"] = fcff_nopat.where(~_nwc_extreme, fcff_cfo)
    # If NOPAT is NaN (EBIT missing), fall back to CFO method
    out["FCFF"] = out["FCFF"].where(out["NOPAT"].notna(), fcff_cfo)

    # ── Balance Sheet fields ───────────────────────────────────────────────────
    out["CashBal"] = cash
    out["DebtBal"] = _c(["Total Debt"])
    out["Cash"]        = cash
    out["AR"]          = _c(["Accounts Receivable", "Receivables", "Net Receivables"])
    out["Inventory"]   = _c(["Inventory", "Inventories"])
    out["TCA"]         = ca
    out["TotalAssets"] = _c(["Total Assets"])
    out["AP"]          = _c(["Accounts Payable", "Payables"])
    out["ShortTermDebt"]= std
    out["TCL"]         = cl
    out["LTDebt"]      = _c(["Long Term Debt",
                              "Long Term Debt And Capital Lease Obligation",
                              "Long-term Debt"])
    out["TotalLiab"]   = _c(["Total Liabilities Net Minority Interest",
                              "Total Liabilities"])
    out["TotalEquity"] = _c(["Stockholders Equity",
                              "Total Equity Gross Minority Interest",
                              "Common Stock Equity",
                              "Total Equity"])
    out["RetainedEarnings"] = _c(["Retained Earnings"])
    out["Goodwill"]    = _c(["Goodwill", "Goodwill And Other Intangible Assets"])
    out["PPE"]         = _c(["Net Ppe", "Net Property Plant And Equipment",
                              "Property Plant Equipment Net"])

    # ── Income Statement extras ────────────────────────────────────────────────
    out["COGS"]        = _c(["Cost Of Revenue", "Cost of Revenue",
                              "Cost Of Goods Sold", "Total Cost of Sales"])
    out["GrossProfit"] = _c(["Gross Profit"])
    # Compute GrossProfit if not reported directly
    _gp_computed = out["Revenue"] - out["COGS"].abs()
    out["GrossProfit"] = out["GrossProfit"].fillna(_gp_computed)

    out["EBITDA"]      = _c(["Normalized EBITDA", "EBITDA"])
    # Compute EBITDA if not reported
    out["EBITDA"] = out["EBITDA"].fillna(out["EBIT"] + out["DandA"].fillna(0))

    out["OpEx"]        = _c(["Operating Expense", "Total Operating Expenses",
                              "Total Expenses", "Operating Expenses"])
    # NetIncome: back-calculate from pretax-tax if direct field is missing
    _ni_direct = _c(["Net Income", "Net Income Common Stockholders",
                     "Net Income Including Noncontrolling Interests",
                     "Net Income From Continuing Operations"])
    # Back-calc NI = Pretax - Tax when direct field is missing
    _ni_calc   = pretax.fillna(0) - _tax_for_ebit.fillna(0)
    out["NetIncome"] = _ni_direct.fillna(_ni_calc)

    # InterestExp: try all cost-of-borrowing fields
    # For NBFCs (IRFC etc.): "Interest Expense" may only capture part of borrowing cost
    # "Cost Of Revenue" for NBFC = interest paid on bonds ≈ true borrowing cost
    _int_direct = _c(["Interest Expense", "Interest Expense Non Operating"])
    _cogs_v     = out.get("COGS", pd.Series(np.nan, index=out.index))
    _rev_v      = out["Revenue"].abs().replace(0, np.nan)
    _int_pct    = (_int_direct.fillna(0).abs() / _rev_v).fillna(0)
    _cogs_pct   = (_cogs_v.fillna(0).abs()     / _rev_v).fillna(0)
    # For financial companies (COGS > 40% of revenue): COGS = borrowing cost
    # Use COGS as interest expense for NBFC/banks regardless of direct interest amount
    _is_nbfc_interest = (_cogs_pct > 0.40)
    out["InterestExp"] = _cogs_v.abs().where(_is_nbfc_interest, _int_direct.abs())

    out["InterestInc"] = _c(["Interest Income", "Interest And Investment Income"])
    # TaxExp: multiple field names + always positive
    _tax_direct = _c(["Tax Provision", "Income Tax Expense",
                       "Current Tax Expense", "Income Tax Payable"]).abs()
    # Back-calc Tax = EBIT - NI when yfinance Tax = 0 (e.g. IRFC)
    _tax_calc   = (out["EBIT"] - _ni_for_ebit.fillna(0)).abs()
    # Use direct tax if > 1% of NI; else use back-calc
    _ni_nonzero = _ni_for_ebit.abs().replace(0, np.nan)
    _tax_ratio  = (_tax_direct.fillna(0) / _ni_nonzero.fillna(1)).fillna(0)
    out["TaxExp"] = _tax_direct.where(_tax_ratio > 0.01, _tax_calc.abs())
    # Back-fill NetIncome using EBIT*(1-t) as ultimate fallback
    _tr_local2  = globals().get("TAX_RATE", 25.0) / 100.0
    _ni_calc    = out["EBIT"] * (1 - _tr_local2)
    out["NetIncome"] = out["NetIncome"].fillna(_ni_calc)
    out["PreTaxIncome"]= _c(["Pretax Income", "Income Before Tax"])

    # ── Cash Flow extras ───────────────────────────────────────────────────────
    out["CFI"]         = _c(["Investing Cash Flow",
                              "Cash Flows From Investing Activities",
                              "Net Cash Used For Investing Activities"])
    out["CFF"]         = _c(["Financing Cash Flow",
                              "Cash Flows From Financing Activities",
                              "Net Cash Used Provided By Financing Activities"])
    out["EndCash"]     = _c(["End Cash Position", "Cash At End Of Period",
                              "Cash And Cash Equivalents End Of Period"])
    out["BegCash"]     = _c(["Beginning Cash Position", "Cash At Beginning Of Period"])
    out["FreeCashFlow"]= _c(["Free Cash Flow"])  # yfinance computed FCF (CFO + Capex)

    # ── Shares ─────────────────────────────────────────────────────────────────
    # yfinance sharesOutstanding is RAW count (e.g. 13,105,370,260 for IRFC)
    # Divide by unit_factor so DilutedShares is in same units as financial data
    _uf_shares = globals().get("unit_factor", 1_000_000.0)
    if "SharesOutstanding" in out.columns and out["SharesOutstanding"].notna().any():
        out["DilutedShares"] = out["SharesOutstanding"] / _uf_shares
    elif not SH.empty and "SharesOutstanding" in SH.columns:
        shares_val = SH["SharesOutstanding"].dropna().values
        raw_sh = shares_val[0] if len(shares_val) > 0 else np.nan
        out["DilutedShares"] = raw_sh / _uf_shares if pd.notna(raw_sh) else np.nan
    else:
        # Fallback: diluted/basic average shares from IS (already normalized)
        out["DilutedShares"] = _c(["Diluted Average Shares",
                                    "Basic Average Shares",
                                    "Ordinary Shares Number"])

    # Shares from file extraction (unlisted companies)
    if "DilutedShares" not in out.columns or out["DilutedShares"].isna().all():
        _dcf_wide = globals().get("_dcf_wide_from_files")
        if _dcf_wide is not None and "shares_outstanding" in _dcf_wide.index:
            _sh_row = _dcf_wide.loc["shares_outstanding"]
            _sh_val = _sh_row.dropna().values
            if len(_sh_val) > 0:
                out["DilutedShares"] = float(_sh_val[-1])
                print(f"   ✅ Shares from file extraction: {float(_sh_val[-1]):,.2f}")

    # ── Keep list ──────────────────────────────────────────────────────────────
    keep = ["FY", "Revenue", "EBIT", "DandA", "Capex", "NWC", "DeltaNWC", "NOPAT", "FCFF",
            "DilutedShares", "CashBal", "DebtBal",
            "Cash", "AR", "Inventory", "TCA", "TotalAssets", "AP", "ShortTermDebt",
            "TCL", "LTDebt", "TotalLiab", "TotalEquity", "RetainedEarnings",
            "Goodwill", "PPE", "TotalLE",
            "COGS", "GrossProfit", "EBITDA", "OpEx", "NetIncome",
            "InterestExp", "InterestInc", "TaxExp", "PreTaxIncome",
            "CFO", "CFI", "CFF", "EndCash", "BegCash", "FreeCashFlow"]
    out = out[[c for c in keep if c in out.columns]].sort_values("FY").reset_index(drop=True)
    out["FY"] = out["FY"].astype(int)
    out["DeltaNWC"] = out["NWC"].diff()
    return out




In [8]:
# =============================================================================
# KGI EXTRACTOR — QuickBooks-style private company file parser
# Reads KGI annual IS/BS/CF xlsx files and builds HIST directly.
# Runs automatically when api=False and FILE_SOURCE_PATH points to KGI.zip/folder.
# Values are in RAW DOLLARS — converted to display units at the end.
# =============================================================================
import re, zipfile, tempfile, math
from pathlib import Path
import pandas as pd
import openpyxl

def _kgi_extract_hist(file_source_path, currency_units="Thousands"):
    """
    Parse KGI QuickBooks export files and return a HIST DataFrame.
    Handles the non-standard layout: 50+ granular line items, 
    YTD values in col index 1 (2023/2024) or col index 3 (2022).
    All totals are computed from components (BS formulas = None in data_only mode).
    """

    # ── Resolve zip or folder ─────────────────────────────────────
    src = Path(file_source_path)
    _tmp = None
    if src.is_file() and src.suffix.lower() == ".zip":
        _tmp = tempfile.mkdtemp(prefix="kgi_")
        with zipfile.ZipFile(src, "r") as zf:
            zf.extractall(_tmp)
        root = Path(_tmp)
    else:
        root = src

    # ── File discovery ────────────────────────────────────────────
    ROLLING = ["ytd","12 months","12 periods","12 period","rolling",
               "trailing","ttm","quarterly","client copy","articles"]
    def _keep(p):
        n = p.name.lower()
        if any(t in n for t in ROLLING): return False
        return p.suffix.lower() in [".xlsx",".xls"]

    files = [p for p in root.rglob("*") if p.is_file() and _keep(p)]
    print(f"  [KGI] Files found: {[f.name for f in files]}")

    def _classify(p):
        n = p.name.lower()
        yr_m = re.search(r'12(\d{2})(\d{2})', n)  # e.g. 123124 → Dec 2024
        year = 2000 + int(yr_m.group(2)) if yr_m else None
        if "income" in n:   return "IS", year
        if "balance" in n:  return "BS", year
        if "cash flow" in n or "cashflow" in n: return "CF", year
        return None, None

    IS_files, BS_files, CF_files = {}, {}, {}
    for p in files:
        kind, yr = _classify(p)
        if kind == "IS" and yr: IS_files[yr] = p
        if kind == "BS" and yr: BS_files[yr] = p
        if kind == "CF" and yr: CF_files[yr] = p

    all_years = sorted(set(list(IS_files) + list(BS_files) + list(CF_files)))
    print(f"  [KGI] Years: IS={sorted(IS_files)}, BS={sorted(BS_files)}, CF={sorted(CF_files)}")

    # ── Read a sheet into {label: value} dict ─────────────────────
    def _read_sheet(path, ytd_col=1):
        wb = openpyxl.load_workbook(str(path), data_only=True)
        ws = wb.active
        result = {}
        for row in ws.iter_rows(values_only=True):
            lbl = str(row[0] or "").strip()
            if not lbl: continue
            val = row[ytd_col] if len(row) > ytd_col else None
            if val is None and ytd_col != 1:
                val = row[1] if len(row) > 1 else None
            try:    result[lbl] = float(val)
            except: result[lbl] = None
        return result

    def _ytd_col(path, year):
        # 2022 IS has 'Current Month' in col1 and 'Year to Date' in col3
        n = str(path).lower()
        if "income" in n and year == 2022: return 3
        return 1

    def _get(d, *keys):
        for k in keys:
            for dk in d:
                if dk.lower() == k.lower():
                    v = d[dk]
                    return float(v) if v is not None else 0.0
        return 0.0

    def _sum(d, *keys):
        return sum(_get(d, k) for k in keys)

    # ── Build records per year ────────────────────────────────────
    records = []
    for yr in all_years:
        rec = {"FY": yr}

        # ── Income Statement ──────────────────────────────────────
        if yr in IS_files:
            IS = _read_sheet(IS_files[yr], _ytd_col(IS_files[yr], yr))
            rec["Revenue"]    = _get(IS, "Total Revenues")
            rec["COGS"]       = _get(IS, "Total Cost of Sales")
            rec["GrossProfit"]= _get(IS, "Gross Profit")
            rec["OpEx"]       = _get(IS, "Total Expenses")
            rec["NetIncome"]  = _get(IS, "Net Income")
            # EBIT = Net Income + Interest + Tax (back-calculate)
            _int  = _get(IS, "Interest Expense") + _get(IS, "Interest - Material Loan #1") + _get(IS, "Interest - Material Loan #2")
            _tax  = _get(IS, "Income Tax Expense") + _get(IS, "Income Tax - Corporate")
            rec["EBIT"]       = rec["NetIncome"] + _int + _tax
            rec["DandA"]      = _get(IS, "Depreciation Expense") + _get(IS, "Amortization Expense")
            rec["InterestExp"]= _int
            rec["TaxExp"]     = _tax

        # ── Balance Sheet ─────────────────────────────────────────
        if yr in BS_files:
            BS = _read_sheet(BS_files[yr], 1)

            rec["Cash"]  = _get(BS, "MN Bank Checking Acct")
            rec["AR"]    = _get(BS, "Accounts Receivable") + _get(BS, "Allowance for Doubtful Account")
            rec["Inventory"] = _get(BS, "Inventory- Materials")

            # Compute totals from components (formulas = None in data_only mode)
            _ca_keys = ["MN Bank Checking Acct","Due From Employee","Due From Officer",
                        "Due from Steiger Lake Lane","Accounts Receivable",
                        "Allowance for Doubtful Account","Inventory- Materials","Prepaid Expenses"]
            rec["TCA"] = _sum(BS, *_ca_keys)

            _ppe_keys = ["Computers and Furniture","Equipment","Automobiles",
                         "Accum. Depreciation - General","Leasehold Improvements"]
            _ppe = _sum(BS, *_ppe_keys)

            _oa_keys = ["Goodwill & Non Compete","Accum Amortization","Organization Costs"]
            _oa = _sum(BS, *_oa_keys)
            rec["Goodwill"]   = max(0, _get(BS,"Goodwill & Non Compete") + _get(BS,"Accum Amortization"))
            rec["TotalAssets"]= rec["TCA"] + _ppe + _oa

            _cl_keys = ["Current Portion Long-Term Debt","MN Bank Credit Line #8301",
                        "Visa Payables","Accounts Payable","Sales Tax Payable",
                        "Federal Payroll Taxes Payable","State Payroll Taxes Payable","Income Taxes Payable"]
            rec["TCL"] = _sum(BS, *_cl_keys)
            rec["AP"]  = _get(BS, "Accounts Payable")
            rec["ShortTermDebt"] = _get(BS,"Current Portion Long-Term Debt") + _get(BS,"MN Bank Credit Line #8301")

            _ltd_keys = ["Compressor Loan (MNBT #2000)","2018 Ch G2500 Van (MNBT #8935)",
                         "2018 Ch G2500 Van (MNBT #8933)","2015 Chev 1500 P/U (MNBT#4879)",
                         "2017 & 2021 Vans (MNBT#1712)","Less Current Portion LT Debt",
                         "2019 DW SK1050 (MNBT #1103)","Highland Bank Term Loan #506",
                         "KGI Consolidation Loan #8302"]
            rec["LTDebt"]     = _sum(BS, *_ltd_keys)
            rec["TotalLiab"]  = rec["TCL"] + rec["LTDebt"]

            _eq_keys = ["Common Stock","Retained Earnings","Net Income","Treasury Stock","Beginning Balance Equity"]
            rec["TotalEquity"]= _sum(BS, *_eq_keys)
            rec["RetainedEarnings"] = _get(BS,"Retained Earnings")
            rec["TotalLE"]    = rec["TotalLiab"] + rec["TotalEquity"]
            rec["DebtBal"]    = rec["ShortTermDebt"] + rec["LTDebt"]
            rec["CashBal"]    = rec["Cash"]

            # NWC
            rec["NWC"] = rec["TCA"] - rec["TCL"]

        # ── Cash Flow ─────────────────────────────────────────────
        if yr in CF_files:
            CF = _read_sheet(CF_files[yr], 1)
            rec["CFO"]     = _get(CF, "Net Cash provided by Operations", "Net Cash Provided by Operations")
            rec["CFI"]     = _get(CF, "Net cash used in investing", "Net Cash Used in Investing")
            rec["CFF"]     = _get(CF, "Net cash used in financing", "Net Cash Used in Financing")
            rec["EndCash"] = _get(CF, "Cash Balance at End of Period")
            rec["BegCash"] = _get(CF, "Cash Balance at Beg of Period")
            rec["Capex"]   = abs(_get(CF, "Net cash used in investing", "Net Cash Used in Investing"))

        records.append(rec)

    if not records:
        print("  [KGI] ⚠️  No records extracted")
        return pd.DataFrame()

    hist = pd.DataFrame(records).sort_values("FY").reset_index(drop=True)

    # ── Unit conversion ───────────────────────────────────────────
    # KGI values are raw dollars. Convert to display units.
    _cu = str(currency_units).lower()
    if   _cu.startswith("billion"):  _divisor = 1_000_000_000.0
    elif _cu.startswith("million"):  _divisor = 1_000_000.0
    elif _cu.startswith("thousand"): _divisor = 1_000.0
    else:                            _divisor = 1.0  # raw dollars (no conversion)

    num_cols = [c for c in hist.columns if c != "FY"]
    for c in num_cols:
        hist[c] = pd.to_numeric(hist[c], errors="coerce") / _divisor

    # ── Derived columns ───────────────────────────────────────────
    hist["DeltaNWC"] = hist["NWC"].diff()
    tr = globals().get("TAX_RATE", 25.0) / 100.0
    hist["NOPAT"] = hist["EBIT"] * (1 - tr)
    hist["FCFF"]  = hist["NOPAT"] + hist["DandA"] - hist["Capex"] - hist["DeltaNWC"].fillna(0)
    # Try to find shares outstanding from IS / BS files before falling back to 1.0
    _sh_found = 0.0
    _sh_pats = ["shares outstanding", "diluted shares", "weighted average diluted",
                "common shares outstanding", "shares issued", "number of shares", "share count"]
    for _sh_yr in sorted(all_years, reverse=True):  # most recent year first
        for _sh_fdict, _sh_use_ytd in [(IS_files, True), (BS_files, False)]:
            if _sh_yr not in _sh_fdict: continue
            _sh_d = _read_sheet(_sh_fdict[_sh_yr], _ytd_col(_sh_fdict[_sh_yr], _sh_yr) if _sh_use_ytd else 1)
            for _sh_lbl, _sh_val in _sh_d.items():
                if _sh_val and any(p in str(_sh_lbl).lower() for p in _sh_pats):
                    try:
                        _sv = float(_sh_val)
                        # Heuristic: if value > 1000 it is raw count; if <= 1000 already in display units
                        if _sv > 0:
                            _sh_found = _sv / _divisor if _sv > 1000 else _sv
                    except: pass
            if _sh_found > 0: break
        if _sh_found > 0: break

    if _sh_found > 0:
        hist["DilutedShares"] = _sh_found
        print(f"  [KGI] Shares from files: {_sh_found:.4f} ({currency_units})")
    else:
        hist["DilutedShares"] = 1.0  # no shares data in files — use placeholder
        print("  [KGI] Shares not found in files — using 1.0 (placeholder)")

    print(f"  [KGI] ✅ Extracted {len(hist)} years in {currency_units}")
    print(hist[["FY","Revenue","EBIT","NetIncome","TotalAssets","DebtBal","TotalEquity","CFO"]].to_string(index=False))
    return hist


# ── Run only when api=False and files are available ───────────────
_is_private = not globals().get("SOURCES",{}).get("api", True)
if _is_private:
    _fsp = globals().get("FILE_SOURCE_PATH","")
    _cu  = globals().get("CURRENCY_UNITS","Millions")
    if _fsp and Path(_fsp).exists():
        print(f"\n[KGI Extractor] Running on: {_fsp}")
        HIST = _kgi_extract_hist(_fsp, _cu)
        if HIST is not None and not HIST.empty:
            BASE_YEAR = int(HIST["FY"].max())
            IS = pd.DataFrame()   # signal to downstream: use HIST directly
            BS = pd.DataFrame()
            CF = pd.DataFrame()
            SH = pd.DataFrame()
            globals()["_HIST_TICKER"]  = str(globals().get("TICKER","")).strip().upper()
            globals()["_HIST_PATH"]    = str(globals().get("FILE_SOURCE_PATH","")).strip()
            globals()["_HIST_SOURCES"] = str(sorted([(k,v) for k,v in globals().get("SOURCES",{}).items() if v]))
            print(f"\n✅ KGI HIST ready | BASE_YEAR={BASE_YEAR}")
            print(f"   Years: {sorted(HIST['FY'].tolist())}")
        else:
            print("⚠️  KGI extractor returned empty — check FILE_SOURCE_PATH and file names")
    else:
        print(f"⚠️  FILE_SOURCE_PATH not found: {_fsp}")
        print("   Set FILE_SOURCE_PATH in Master Config to the path of KGI.zip")
else:
    print("ℹ️  KGI Extractor: skipped (api=True mode uses yfinance)")


ℹ️  KGI Extractor: skipped (api=True mode uses yfinance)


In [9]:
# ── 3E: merge_sources() — orchestrate all sources into unified HIST ───────────

def merge_sources():
    _ticker = str(globals().get("TICKER_INPUT","")).strip().upper() or globals().get("TICKER","")
    if not _ticker:
        raise ValueError("TICKER_INPUT is empty.")

    _tax_rate       = globals().get("TAX_RATE", 21.0)
    _currency_units = globals().get("CURRENCY_UNITS", "Millions")

    METRIC_TO_HIST = {
        "revenue":                       "Revenue",
        "ebit":                          "EBIT",
        "depreciation_and_amortization": "DandA",
        "capex":                         "Capex",
        "simplified_nwc":                "NWC",
        "cash":                          "CashBal",
        "shares_outstanding":            "DilutedShares",
    }

    hist_by_source = {}

    # ── File sources ──────────────────────────────────────────────────────────
    _file_sources = [s for s in SOURCE_PRIORITY if s in ("excel","pdf","ppt") and SOURCES.get(s)]
    if _file_sources:
        print(f"  [FILES] Scanning: {FILE_SOURCE_PATH}")
        try:
            _files_df, _tmp = discover_files(FILE_SOURCE_PATH, strict_annual_only=True)
            _included = _files_df[_files_df["include"]]
            print(f"  [FILES] {len(_included)} usable files found.")

            _excel_long = pd.DataFrame()
            _pdf_long   = pd.DataFrame()
            _pdf_bs_fb  = pd.DataFrame()
            _csv_long   = pd.DataFrame()

            if SOURCES.get("excel"):
                _excel_long = extract_excel_master(_files_df)
                print(f"  [Excel] {len(_excel_long)} rows extracted.")
                # Supplement with Indian multi-sheet extractor
                for _, _ef in _files_df[(_files_df["include"]) & (_files_df["ext"].isin([".xlsx",".xls",".xlsm"]))].iterrows():
                    try:
                        _indian_rows = parse_indian_excel_workbook(Path(_ef["path"]))
                        if not _indian_rows.empty:
                            if _excel_long.empty:
                                _excel_long = _indian_rows
                            else:
                                _ex_keys = set(zip(_excel_long["year"], _excel_long["sheet_name"], _excel_long["line_item_norm"]))
                                _new = _indian_rows[[(r["year"],r["sheet_name"],r["line_item_norm"]) not in _ex_keys for _,r in _indian_rows.iterrows()]]
                                if not _new.empty:
                                    _excel_long = pd.concat([_excel_long, _new], ignore_index=True)
                    except Exception: pass
                # WACC from Excel DCF inputs sheet
                for _, _ef in _files_df[(_files_df["include"]) & (_files_df["ext"].isin([".xlsx",".xls",".xlsm"]))].iterrows():
                    try:
                        _wacc_xl = parse_wacc_from_excel_inputs(Path(_ef["path"]))
                        for _wk, _wv in _wacc_xl.items():
                            if globals().get(_wk) is None:
                                globals()[_wk] = _wv
                                print(f"  [Excel+] {_wk} = {_wv} (from file)")
                    except Exception: pass

            if SOURCES.get("pdf"):
                _pdf_long = extract_pdf_master(_files_df)
                print(f"  [PDF]   {len(_pdf_long)} rows extracted.")
                _pdf_bs_fb = extract_pdf_balance_sheet_fallback(_files_df)

            if SOURCES.get("ppt"):
                _ppt_meta = extract_ppt_text_metadata(_files_df)
                print(f"  [PPT]   {len(_ppt_meta)} slide-text rows (metadata only).")

            _csv_long = extract_csv_master(_files_df)
            if not _csv_long.empty:
                print(f"  [CSV]   {len(_csv_long)} rows extracted.")

            # Parse WACC from TXT files
            _wacc_from_txt = parse_wacc_from_txt(_files_df)
            for _wk, _wv in _wacc_from_txt.items():
                if globals().get(_wk) is None:
                    globals()[_wk] = _wv
                    print(f"  [TXT]   {_wk} = {_wv}")

            # Merge file data (Excel > PDF > CSV)
            _parts = []
            if not _excel_long.empty: _parts.append(_excel_long)
            if not _pdf_long.empty:   _parts.append(_pdf_long)
            if not _csv_long.empty:   _parts.append(_csv_long)

            if _parts:
                _master = _parts[0].copy()
                for _extra in _parts[1:]:
                    _keys = set(zip(_master["year"], _master["statement_type"], _master["line_item_norm"]))
                    _supp = _extra[[(y,s,l) not in _keys for y,s,l in zip(_extra["year"],_extra["statement_type"],_extra["line_item_norm"])]].copy()
                    if not _supp.empty:
                        _master = pd.concat([_master, _supp], ignore_index=True)
            else:
                _master = pd.DataFrame()

            if not _master.empty:
                _yr_vals = sorted(_master['year'].dropna().astype(int).unique().tolist())
                print(f"  [FILES] Extracted {len(_master)} raw rows | years detected: {_yr_vals if _yr_vals else 'NONE — year detection failed'}")
                if not _yr_vals:
                    print("  [FILES] ⚠️  No years found in extracted rows.")
                    print("           Possible causes:")
                    print("           1. Excel column headers are Excel date serials (not text/integer years)")
                    print("           2. File names contain no year (e.g. 'Income Statement.xlsx')")
                    print("           3. Year format not recognised (try: 2024, FY2024, FY24, 'Dec 2024')")
                    print("           Tip: set BASE_YEAR in Master Config — it will be used as a fallback year.")
                    # Try once more after setting _base_year_hint from BASE_YEAR config
                    _by_cfg = globals().get('BASE_YEAR')
                    if _by_cfg:
                        globals()['_base_year_hint'] = int(_by_cfg)
                        print(f"  [FILES] Retrying extraction with BASE_YEAR={_by_cfg} as year fallback...")
                        _excel_long2 = extract_excel_master(_files_df) if SOURCES.get('excel') else pd.DataFrame()
                        _pdf_long2   = extract_pdf_master(_files_df)   if SOURCES.get('pdf')   else pd.DataFrame()
                        _csv_long2   = extract_csv_master(_files_df)
                        _parts2 = [p for p in [_excel_long2, _pdf_long2, _csv_long2] if not p.empty]
                        if _parts2:
                            _master2 = pd.concat(_parts2, ignore_index=True)
                            _yr_vals2 = sorted(_master2['year'].dropna().astype(int).unique().tolist())
                            print(f"  [FILES] Retry found years: {_yr_vals2}")
                            if _yr_vals2:
                                _master = _master2
                # ── Build DCF output (safe — handles year detection failure) ───
                _result = None
                try:
                    _result = build_dcf_output(_master, n_years=5, pdf_bs_fallback=_pdf_bs_fb)
                    _compiled = extract_compiled_pdf_current_year_metrics(_files_df)
                    _result   = apply_compiled_pdf_overrides(_result, _compiled, {"DISPLAY_IN_MILLIONS": False})
                except ValueError as _dcf_ve:
                    print(f"  [FILES] ⚠️  build_dcf_output failed: {_dcf_ve}")
                    _yr_sample = _master["year"].value_counts().head(10).to_dict()
                    print(f"           Year values seen in extracted data: {_yr_sample}")
                    print("           → Set BASE_YEAR in Master Config to force year assignment.")
                _dcf_wide = _result["dcf_wide"] if _result is not None else pd.DataFrame()
                if _dcf_wide.empty or len(_dcf_wide.columns) == 0:
                    print("  [FILES] ⚠️  build_dcf_output returned empty wide table.")
                    print("           This usually means no valid years were detected in extracted rows.")
                    print("           Check: (1) file names contain year hints, (2) Excel tables have year headers.")
                    # Skip registering this source — fall through to API or raise clear error
                else:
                    pass  # proceed normally
                globals()["_dcf_wide_from_files"] = _dcf_wide  # expose for shares lookup

                _rows = []
                for _yr in _dcf_wide.columns:
                    _r = {"FY": int(_yr)}
                    for _met, _col in METRIC_TO_HIST.items():
                        _v = _dcf_wide.loc[_met, _yr] if _met in _dcf_wide.index else np.nan
                        _r[_col] = float(_v) if pd.notna(_v) else np.nan
                    _nd = float(_dcf_wide.loc["net_debt", _yr]) if "net_debt" in _dcf_wide.index else np.nan
                    _cs = _r.get("CashBal", np.nan)
                    _r["DebtBal"] = (_nd + _cs) if (pd.notna(_nd) and pd.notna(_cs)) else np.nan
                    _rows.append(_r)

                _hist_files = pd.DataFrame(_rows).sort_values("FY").reset_index(drop=True)
                _hist_files["FY"] = _hist_files["FY"].astype(int)
                # Divide file-extracted values by unit_factor so HIST is in display units
                # (same scale as yfinance path which is normalised by cell 10)
                _uf_files = globals().get("unit_factor", 1_000_000.0)
                for _fc in [c for c in _hist_files.columns if c != "FY"]:
                    _hist_files[_fc] = pd.to_numeric(_hist_files[_fc], errors="coerce") / _uf_files
                if "NWC" in _hist_files.columns:
                    _hist_files["DeltaNWC"] = _hist_files["NWC"].diff()

                # ── Enrich HIST with full BS/IS/CF detail from source files ──
                # Scan all discovered Excel files for standard financial rows
                try:
                    import openpyxl as _opx
                    _BS_MAP = {
                        "cash & cash": "Cash", "accounts receivable": "AR",
                        "inventories": "Inventory", "inventory": "Inventory",
                        "total current assets": "TCA", "total assets": "TotalAssets",
                        "accounts payable": "AP",
                        "total current liab": "TCL",
                        "long-term debt": "LTDebt", "long term debt": "LTDebt",
                        "total liab": "TotalLiab",
                        "retained earnings": "RetainedEarnings",
                        "stockholders": "TotalEquity",
                        "total equity": "TotalEquity",
                        "total capital": "TotalEquity",
                        "short-term debt": "ShortTermDebt",
                        "current portion": "ShortTermDebt",
                        "net ppe": "PPE", "pp&e (net)": "PPE",
                        "goodwill": "Goodwill",
                    }
                    _IS_MAP = {
                        "total net revenue": "Revenue", "net revenue": "Revenue",
                        "total cogs": "COGS", "cost of goods": "COGS",
                        "gross profit": "GrossProfit",
                        "net income": "NetIncome",
                        "interest expense": "InterestExp",
                        "income tax": "TaxExp", "tax provision": "TaxExp",
                        "operating income": "EBIT", "ebit": "EBIT",
                    }
                    _CF_MAP = {
                        "cash from operations": "CFO",
                        "cash provided by operations": "CFO",
                        "net cash provided by operating": "CFO",
                        "cash used in investing": "CFI",
                        "net cash used in investing": "CFI",
                        "financing": "CFF",
                        "capital expenditures": "Capex",
                        "capex": "Capex",
                        "end cash": "EndCash", "ending cash": "EndCash",
                        "beginning cash": "BegCash",
                    }
                    _ALL_MAP = {**_BS_MAP, **_IS_MAP, **_CF_MAP}

                    def _extract_excel_detail(fpath, label_map):
                        """Scan xlsx for year-keyed financial rows."""
                        results = {}
                        try:
                            _wb2 = _opx.load_workbook(str(fpath), data_only=True)
                        except: return results
                        for _sh in _wb2.sheetnames:
                            _ws2 = _wb2[_sh]
                            # Find year header row
                            _yr_row, _yr_cols = None, {}
                            for _r in range(1, 12):
                                for _c in range(1, 15):
                                    _v = _ws2.cell(_r, _c).value
                                    _vs = str(_v or "").strip().replace("FY","").replace("'","")
                                    if _vs.isdigit() and 2010 <= int(_vs) <= 2030:
                                        if _yr_row is None: _yr_row = _r
                                        if _r == _yr_row: _yr_cols[_c] = int(_vs)
                            if not _yr_cols: continue
                            for _r in range(_yr_row+1, _ws2.max_row+1):
                                _lbl = str(_ws2.cell(_r,1).value or "").strip().lower()
                                if not _lbl: continue
                                for _pat, _col in label_map.items():
                                    if _pat in _lbl and _col not in results:
                                        for _c2, _yr2 in _yr_cols.items():
                                            try:
                                                _fv = float(_ws2.cell(_r,_c2).value)
                                                if _col not in results: results[_col] = {}
                                                results[_col][_yr2] = _fv
                                            except: pass
                                        break
                        return results

                    # Scan all Excel files found in this zip
                    _detail = {}
                    _skip = ["ytd","12 months","12 periods","rolling","client copy","kpi","ratio","segment","operational","draft"]
                    for _, _ef in _files_df[(_files_df["include"]) & (_files_df["ext"].isin([".xlsx",".xls",".xlsm"]))].iterrows():
                        _ename = str(_ef.get("name","") or _ef.get("path","")).lower()
                        if any(_s in _ename for _s in _skip): continue
                        _res = _extract_excel_detail(Path(_ef["path"]), _ALL_MAP)
                        for _col, _yr_vals in _res.items():
                            if _col not in _detail:
                                _detail[_col] = _yr_vals
                    
                    # Merge detail into _hist_files (divide by unit_factor since source is raw)
                    _fys = _hist_files["FY"].tolist()
                    for _col, _yr_vals in _detail.items():
                        if _col not in _hist_files.columns:
                            # Detect unit scale: if values >> 1000 they need dividing
                            _vals_list = [_yr_vals.get(int(_fy), np.nan) for _fy in _fys]
                            _max_v = max((abs(v) for v in _vals_list if not np.isnan(v)), default=0)
                            # If source already in Millions (values < 50000) divide by unit_factor
                            # If source in raw dollars (values > 1e8) divide by unit_factor
                            # The detect_unit_scale already scaled: values = raw * unit_scale
                            # But direct xlsx reads are raw source values (already in Millions for Ironclad)
                            _div = _uf_files if _max_v > 1000 else 1.0
                            _hist_files[_col] = [v / _div for v in _vals_list]
                    if _detail:
                        print(f"  ✅ Enriched HIST with BS/IS/CF detail: {list(_detail.keys())}")
                except Exception as _enrich_err:
                    print(f"  ⚠️  BS detail enrichment error: {_enrich_err}")

                if not _hist_files.empty and "FY" in _hist_files.columns:
                    for _s in _file_sources:
                        hist_by_source[_s] = _hist_files
                    print(f"  [FILES] ✅ {len(_hist_files)} year(s) in HIST from files.")
                else:
                    print("  [FILES] ⚠️  File extraction produced no valid year data — skipping file sources.")
                globals()["_master_long_for_sheets"] = _master
                print(f"  [FILES] ✅ Years extracted: {sorted(_hist_files['FY'].tolist())}")
            else:
                print("  [FILES] ⚠️  No usable data extracted from files.")
        except Exception as _e:
            import traceback
            print(f"  [FILES] ⚠️  Extraction failed: {_e}")
            traceback.print_exc()

    # ── API source ────────────────────────────────────────────────────────────
    if SOURCES.get("api"):
        print(f"  [API]   Fetching Yahoo Finance data for {_ticker}...")
        try:
            _uf = 1.0
            _u  = str(_currency_units).lower()
            if _u.startswith("million"):  _uf = 1_000_000.0
            elif _u.startswith("billion"): _uf = 1_000_000_000.0
            # IS/BS/CF already normalized to CURRENCY_UNITS in Part 3 cell 10
            # Do NOT call normalize_units again — that would divide twice
            _hist_api = build_historical(IS, BS, CF, SH)
            if not _hist_api.empty and "FY" in _hist_api.columns:
                hist_by_source["api"] = _hist_api
                print(f"  [API]   ✅ Years: {sorted(_hist_api['FY'].tolist())}")
            else:
                print(f"  [API]   ⚠️  build_historical returned empty — check ticker '{_ticker}'")
        except Exception as _e:
            print(f"  [API]   ⚠️  Failed: {_e}")

    # ── KGI dedicated-extractor fallback ─────────────────────────────────────
    # If generic extraction produced nothing but _kgi_extract_hist is available
    # (defined in cell 12), try it before raising.
    if not hist_by_source and callable(globals().get("_kgi_extract_hist")):
        print("  [KGI]   Generic extractors returned no data — trying KGI-specific extractor...")
        try:
            _kgi_h = globals()["_kgi_extract_hist"](
                globals().get("FILE_SOURCE_PATH", ""),
                globals().get("CURRENCY_UNITS", "Millions"),
            )
            if _kgi_h is not None and not _kgi_h.empty and "FY" in _kgi_h.columns:
                for _ks in [s for s in SOURCE_PRIORITY if s in ("excel","pdf","ppt") and SOURCES.get(s)]:
                    hist_by_source[_ks] = _kgi_h
                    break  # register under first active file source
                print(f"  [KGI]   ✅ KGI fallback succeeded: {sorted(_kgi_h['FY'].tolist())}")
        except Exception as _kgi_fb_e:
            print(f"  [KGI]   ⚠️  KGI fallback failed: {_kgi_fb_e}")

    if not hist_by_source:
        raise ValueError("No data from any source. Check SOURCES switches and ticker.")

    # ── Merge by priority ──────────────────────────────────────────────────────
    # Only use sources that have a non-empty DataFrame with an FY column
    _active = [
        s for s in SOURCE_PRIORITY
        if s in hist_by_source
        and not hist_by_source[s].empty
        and "FY" in hist_by_source[s].columns
    ]
    if not _active:
        raise ValueError(
            "All data sources returned empty results. "
            "Check FILE_SOURCE_PATH, SOURCES switches, and file structure."
        )

    _merged   = hist_by_source[_active[0]].copy().reset_index(drop=True)
    _num_cols = [c for c in _merged.columns if c != "FY"]

    for _src in _active[1:]:
        _other = hist_by_source[_src]
        if _other.empty or "FY" not in _other.columns:
            continue
        for _, _orow in _other.iterrows():
            try:
                _fy = int(_orow["FY"])
            except (ValueError, TypeError):
                continue
            _mask = _merged["FY"] == _fy
            if not _mask.any():
                _new_row = _orow.to_frame().T.reset_index(drop=True)
                _merged = pd.concat([_merged, _new_row], ignore_index=True)
                # Expand _num_cols to include any new columns from this source
                _num_cols = [c for c in _merged.columns if c != "FY"]
            else:
                for _col in _num_cols:
                    if _col not in _orow.index:
                        continue
                    _cur_val = _merged.loc[_mask, _col]
                    if not _cur_val.empty and pd.isna(_cur_val.values[0]) and pd.notna(_orow[_col]):
                        _merged.loc[_mask, _col] = _orow[_col]

    # Guard: ensure FY column exists and is sortable before proceeding
    if "FY" not in _merged.columns:
        raise ValueError("Merged HIST has no FY column — check source DataFrames.")
    _merged = _merged.sort_values("FY").reset_index(drop=True)
    _merged["FY"] = _merged["FY"].astype(int)

    # Recompute derived columns
    if "NWC" in _merged.columns:
        _merged["DeltaNWC"] = _merged["NWC"].diff()
    _tr = _tax_rate / 100.0
    if "EBIT" in _merged.columns:
        _merged["NOPAT"] = _merged["EBIT"] * (1 - _tr)
    if all(c in _merged.columns for c in ["NOPAT","DandA","Capex","DeltaNWC"]):
        _merged["FCFF"] = (_merged["NOPAT"].fillna(0) + _merged["DandA"].fillna(0)
                          - _merged["Capex"].fillna(0) - _merged["DeltaNWC"].fillna(0))

    print(f"\n✅ Unified HIST ready | {len(_merged)} years | active sources: {_active}")
    return _merged

# Run merge_sources
print("\n" + "="*60)
print("Building unified HIST DataFrame...")
# Guard: rebuild HIST when TICKER or FILE_SOURCE_PATH changes
_cur_tk   = str(globals().get("TICKER","")).strip().upper()
_prev_tk  = str(globals().get("_HIST_TICKER","")).strip().upper()
_cur_path = str(globals().get("FILE_SOURCE_PATH","")).strip()
_prev_path= str(globals().get("_HIST_PATH","")).strip()
_hist_exists = (globals().get("HIST") is not None and
                hasattr(globals().get("HIST"),"empty") and
                not globals()["HIST"].empty)
# Include active sources in cache key — switching api↔file rebuilds HIST
_cur_sources = str(sorted([(k,v) for k,v in SOURCES.items() if v]))
_prev_sources = str(globals().get("_HIST_SOURCES",""))
_hist_ok = (_hist_exists and _prev_tk == _cur_tk and _cur_tk != ""
            and _prev_path == _cur_path and _prev_sources == _cur_sources)
if _hist_ok:
    print(f"  ✅ HIST already set for {_cur_tk} — skipping merge_sources()")
else:
    if _hist_exists:
        print(f"  🔄 Config changed — rebuilding HIST for {_cur_tk}")
    HIST = merge_sources()
    globals()["_HIST_TICKER"]  = _cur_tk
    globals()["_HIST_PATH"]    = _cur_path
    globals()["_HIST_SOURCES"] = _cur_sources

# Set BASE_YEAR
if HIST is not None and not HIST.empty:
    if BASE_YEAR is None:
        BASE_YEAR = int(HIST["FY"].max())
    years = sorted(HIST["FY"].tolist())
    print(f"✅ BASE_YEAR = {BASE_YEAR} | Years in HIST = {years}")
    HIST



Building unified HIST DataFrame...
  [API]   Fetching Yahoo Finance data for RELIANCE.NS...
  [API]   ✅ Years: [2022, 2023, 2024, 2025]

✅ Unified HIST ready | 4 years | active sources: ['api']
✅ BASE_YEAR = 2025 | Years in HIST = [2022, 2023, 2024, 2025]


# Part 4 — DCF Build (Forecast + WACC + Valuation)

In [10]:
# ── 4A: Infer drivers & forecast ─────────────────────────────────────────────

def _safe_default(x, fallback):
    try: return fallback if (x is None or (isinstance(x, float) and np.isnan(x))) else x
    except: return fallback

def infer_drivers(hist, window=5):
    h = hist.dropna(subset=["Revenue"]).sort_values("FY").tail(window)
    h = h.copy()
    h["RevenueGrowth"] = h["Revenue"].pct_change()
    def _ratio(num, den): return (h[num]/h[den]).replace([np.inf,-np.inf],np.nan)
    return {
        "RevenueGrowth_default": float(_safe_default(h["RevenueGrowth"].median(skipna=True), 0.02)),
        "EBITMargin_default":    float(_safe_default(_ratio("EBIT","Revenue").median(skipna=True), 0.15)),
        "CapexPct_default":      float(abs(_safe_default(_ratio("Capex","Revenue").median(skipna=True), 0.03))),
        "NWCpct_default":        float(_safe_default(_ratio("NWC","Revenue").median(skipna=True), 0.00)),
        "DApct_default":         float(_safe_default(_ratio("DandA","Revenue").median(skipna=True), 0.03)),
    }

def project_fcff(hist, base_year, years, drivers, overrides=None):
    last_row = hist.loc[hist["FY"].idxmax()].copy()
    revenue   = float(last_row["Revenue"])
    nwc       = float(last_row["NWC"]) if pd.notna(last_row.get("NWC")) else 0.0
    tr        = TAX_RATE / 100.0
    g_def     = max(-0.5, min(0.5, drivers["RevenueGrowth_default"]))
    m_def     = max(-0.5, min(0.5, drivers["EBITMargin_default"]))
    capex_def = max(0.0,  min(0.5, drivers["CapexPct_default"]))
    nwc_def   = max(-0.5, min(0.5, drivers["NWCpct_default"]))
    da_def    = max(0.0,  min(0.5, drivers["DApct_default"]))

    rows = []
    OVR_df = overrides
    for t in range(1, years + 1):
        g = g_def; m = m_def; capex_pct = capex_def; nwc_pct = nwc_def; da_pct = da_def
        if OVR_df is not None and t in OVR_df["YearIdx"].values:
            ov = OVR_df[OVR_df["YearIdx"] == t].iloc[0]
            if pd.notna(ov["RevenueGrowth_override"]): g        = float(ov["RevenueGrowth_override"])
            if pd.notna(ov["EBITMargin_override"]):    m        = float(ov["EBITMargin_override"])
            if pd.notna(ov["CapexPct_override"]):      capex_pct= float(ov["CapexPct_override"])
            if pd.notna(ov["NWCpct_override"]):        nwc_pct  = float(ov["NWCpct_override"])
            if pd.notna(ov["DApct_override"]):         da_pct   = float(ov["DApct_override"])

        revenue      = revenue * (1 + g)
        ebit         = revenue * m
        nopat        = ebit * (1 - tr)
        danda        = revenue * da_pct
        capex        = revenue * capex_pct
        target_nwc   = revenue * nwc_pct
        delta_nwc    = target_nwc - nwc
        nwc          = target_nwc
        fcff         = nopat + danda - capex - delta_nwc
        rows.append({
            "FY": base_year + t, "Revenue": revenue, "EBIT": ebit, "NOPAT": nopat,
            "DandA": danda, "Capex": capex, "DeltaNWC": delta_nwc, "FCFF": fcff,
            "EBITMargin": m, "DApct": da_pct, "CapexPct": capex_pct,
            "NWCpct": nwc_pct, "RevenueGrowth": g,
        })
    return pd.DataFrame(rows)

drivers = infer_drivers(HIST, window=5)

# Year-by-year override table (set values to override defaults)
OVR = pd.DataFrame({
    "YearIdx":               list(range(1, FORECAST_YEARS + 1)),
    "RevenueGrowth_override": np.nan,
    "EBITMargin_override":    np.nan,
    "CapexPct_override":      np.nan,
    "NWCpct_override":        np.nan,
    "DApct_override":         np.nan,
})

FORE = project_fcff(HIST, BASE_YEAR, FORECAST_YEARS, drivers, OVR)
print("✅ Forecast (FORE) built:")
FORE


✅ Forecast (FORE) built:


,FY,Revenue,EBIT,NOPAT,DandA,Capex,DeltaNWC,FCFF,EBITMargin,DApct,CapexPct,NWCpct,RevenueGrowth
0,2026,"10,328,151.88","1,058,590.36","836,286.38","521,533.21","1,578,650.02","528,850.66","-749,681.09",0.10,0.05,0.15,0.10,0.07
1,2027,"11,057,478.51","1,133,343.14","895,341.08","558,361.49","1,690,127.03","76,549.97","-312,974.43",0.10,0.05,0.15,0.10,0.07
2,2028,"11,838,306.84","1,213,374.63","958,565.96","597,790.41","1,809,476.04","81,955.58","-335,075.24",0.10,0.05,0.15,0.10,0.07
3,2029,"12,674,273.69","1,299,057.57","1,026,255.48","640,003.63","1,937,252.92","87,742.90","-358,736.72",0.10,0.05,0.15,0.10,0.07
4,2030,"13,569,272.67","1,390,791.05","1,098,724.93","685,197.74","2,074,052.82","93,938.91","-384,069.06",0.10,0.05,0.15,0.10,0.07


In [11]:
# ── 4B: WACC + DCF Valuation ──────────────────────────────────────────────────

def compute_wacc():
    if WACC_DIRECT is not None: return float(WACC_DIRECT) / 100.0
    ke       = (RISK_FREE / 100.0) + BETA * (EQUITY_RISK_PREMIUM / 100.0)
    kd_after = (COST_OF_DEBT_PRETAX / 100.0) * (1 - (TAX_RATE / 100.0))
    wd       = TARGET_DEBT_WEIGHT / 100.0
    return (1 - wd) * ke + wd * kd_after

def discount_factor(rate, t, mid_year=True):
    exp = (t - 0.5) if mid_year else t
    return (1 + rate) ** (-exp)

def dcf_valuation(fore, mid_year=True, use_exit_multiple=False):
    wacc = compute_wacc()
    if wacc <= 0: raise ValueError("WACC must be > 0")

    pv_fcffs = sum(fore.iloc[i]["FCFF"] * discount_factor(wacc, i + 1, mid_year)
                   for i in range(len(fore)))

    T_row = fore.iloc[-1]
    if use_exit_multiple:
        TV = (T_row["EBIT"] + T_row["DandA"]) * EXIT_MULTIPLE
    else:
        fcff_T1 = T_row["FCFF"] * (1 + (TERMINAL_GROWTH / 100.0))
        if wacc <= (TERMINAL_GROWTH / 100.0):
            raise ValueError("For Gordon Growth, WACC must exceed terminal growth rate.")
        TV = fcff_T1 / (wacc - (TERMINAL_GROWTH / 100.0))

    pv_tv   = TV * discount_factor(wacc, len(fore), mid_year)
    EV      = pv_fcffs + pv_tv

    last_hist = HIST.loc[HIST["FY"].idxmax()]
    cash  = float(last_hist.get("CashBal", 0.0)) if pd.notna(last_hist.get("CashBal", np.nan)) else 0.0
    debt  = float(last_hist.get("DebtBal", 0.0)) if pd.notna(last_hist.get("DebtBal", np.nan)) else 0.0
    net_debt      = debt - cash
    equity_value  = EV - net_debt
    shares        = float(last_hist.get("DilutedShares", np.nan))
    per_share     = equity_value / shares if (shares and not math.isnan(shares) and shares > 0) else np.nan

    return {
        "WACC": wacc, "PV_FCFFs": pv_fcffs, "PV_TV": pv_tv, "EV": EV,
        "NetDebt": net_debt, "EquityValue": equity_value, "PerShare": per_share,
        "TV_Method": "Exit Multiple" if use_exit_multiple else "Gordon Growth",
    }

VAL = dcf_valuation(FORE, MID_YEAR, USE_EXIT_MULTIPLE)
wacc = compute_wacc()

print("=== Valuation Summary ===")
print(f"WACC          : {VAL['WACC']*100:,.2f}%")
print(f"TV Method     : {VAL['TV_Method']}")
print(f"PV FCFFs      : {VAL['PV_FCFFs']:,.2f}")
print(f"PV TV         : {VAL['PV_TV']:,.2f}")
print(f"Enterprise Value: {VAL['EV']:,.2f}")
print(f"Net Debt      : {VAL['NetDebt']:,.2f}")
print(f"Equity Value  : {VAL['EquityValue']:,.2f}")
print(f"Per Share     : {VAL['PerShare']:,.2f}" if pd.notna(VAL["PerShare"]) else "Per Share : N/A (no shares data)")


=== Valuation Summary ===
WACC          : 8.39%
TV Method     : Gordon Growth
PV FCFFs      : -1,809,244.56
PV TV         : -4,266,356.67
Enterprise Value: -6,075,601.23
Net Debt      : 2,689,300.00
Equity Value  : -8,764,901.23
Per Share     : -647.69


In [12]:
# ── 4C: Sensitivity Tables (pure Python — no Excel DataTable) ────────────────

def per_share_given(wacc_rate, g=None, exit_mult=None, margin_bump=0.0):
    adj = FORE.copy()
    if margin_bump != 0.0:
        adj["EBIT"]  = adj["Revenue"] * (adj["EBITMargin"] + margin_bump)
        tr           = TAX_RATE / 100.0
        adj["NOPAT"] = adj["EBIT"] * (1 - tr)
        adj["FCFF"]  = adj["NOPAT"] + adj["DandA"] - adj["Capex"] - adj["DeltaNWC"]

    def df_factor(rate, t):
        exp = (t - 0.5) if MID_YEAR else t
        return (1 + rate) ** (-exp)

    pv_fcffs = sum(adj.iloc[i]["FCFF"] * df_factor(wacc_rate, i + 1) for i in range(len(adj)))

    if exit_mult is not None:
        ebitda_T = adj.iloc[-1]["EBIT"] + adj.iloc[-1]["DandA"]
        TV = ebitda_T * exit_mult
    else:
        fcff_T1 = adj.iloc[-1]["FCFF"] * (1 + g)
        if wacc_rate <= g: return np.nan
        TV = fcff_T1 / (wacc_rate - g)

    pv_tv     = TV * df_factor(wacc_rate, len(adj))
    EV        = pv_fcffs + pv_tv
    last_hist = HIST.loc[HIST["FY"].idxmax()]
    cash      = float(last_hist.get("CashBal", 0.0)) if pd.notna(last_hist.get("CashBal", np.nan)) else 0.0
    debt      = float(last_hist.get("DebtBal", 0.0)) if pd.notna(last_hist.get("DebtBal", np.nan)) else 0.0
    net_debt  = debt - cash
    equity    = EV - net_debt
    shares    = float(last_hist.get("DilutedShares", np.nan))
    if not shares or math.isnan(shares) or shares <= 0: return np.nan
    return equity / shares

# WACC vs Terminal Growth
sens_wacc_g = pd.DataFrame(
    index=[f"{w*100:.1f}%" for w in WACC_GRID],
    columns=[f"{g*100:.1f}%" for g in GROWTH_GRID]
)
for w in WACC_GRID:
    for g in GROWTH_GRID:
        sens_wacc_g.loc[f"{w*100:.1f}%", f"{g*100:.1f}%"] = per_share_given(wacc_rate=w, g=g)

# Exit Multiple vs Margin
sens_mult_margin = pd.DataFrame(
    index=[f"{m:+.1%}" for m in MARGIN_ADJ],
    columns=[f"{em}x" for em in EXIT_MULT_GRID]
)
for m in MARGIN_ADJ:
    for em in EXIT_MULT_GRID:
        sens_mult_margin.loc[f"{m:+.1%}", f"{em}x"] = per_share_given(
            wacc_rate=compute_wacc(), exit_mult=em, margin_bump=m
        )

# Validation checks
CHECKS = pd.DataFrame([
    {"Check": "WACC > 0",       "Pass": wacc > 0,              "Notes": f"WACC={wacc*100:.2f}%"},
    {"Check": "g < WACC",       "Pass": (TERMINAL_GROWTH/100.0) < wacc if not USE_EXIT_MULTIPLE else True,
                                 "Notes": f"g={TERMINAL_GROWTH:.2f}%"},
    {"Check": "FCFF present",   "Pass": FORE["FCFF"].notna().all(), "Notes": ""},
])

print("✅ Sensitivity tables built.")
print("\nWACC vs Terminal Growth:")
print(sens_wacc_g)


✅ Sensitivity tables built.

WACC vs Terminal Growth:
         1.0%    2.0%    2.5%      3.0%      3.5%
6.0%  -779.66 -895.38 -978.04 -1,088.26 -1,242.56
7.0%  -688.29 -762.95 -812.72   -874.93   -954.92
8.0%  -623.03 -674.64 -707.49   -746.91   -795.09
9.0%  -574.07 -611.56 -634.63   -661.54   -693.34
10.0% -536.00 -564.23 -581.18   -600.54   -622.88
11.0% -505.53 -527.42 -540.29   -554.77   -571.19
12.0% -480.60 -497.95 -508.00   -519.16   -531.64


# Part 5 — Excel Output Writer

In [13]:
# ── 5A: Setup output workbook ─────────────────────────────────────────────────

CWD        = Path.cwd()
OUTPUT_DIR = CWD

# Accept papermill-injected currency; fall back to suffix detection
_suffix_ccy_map_19 = {".NS": "INR", ".BO": "INR", ".L": "GBP", ".PA": "EUR", ".DE": "EUR", ".T": "JPY", ".HK": "HKD", ".AX": "AUD"}
_detected_ccy_19 = next((c for s, c in _suffix_ccy_map_19.items() if str(TICKER).upper().endswith(s)), "USD")
LOCAL_CURRENCY = globals().get("LOCAL_CURRENCY") or _detected_ccy_19
_default_sym = {"INR": "₹", "USD": "$", "GBP": "£", "EUR": "€", "JPY": "¥", "HKD": "HK$", "AUD": "A$"}
SYM = globals().get("CURRENCY_SYMBOL") or _default_sym.get(LOCAL_CURRENCY, "$")
# Force into globals so all downstream cells pick up the correct value
globals()["LOCAL_CURRENCY"] = LOCAL_CURRENCY
globals()["SYM"] = SYM
globals()["CURRENCY_SYMBOL"] = SYM

# Find template
MASTER_CANDIDATES = [
    OUTPUT_DIR / "DCF_with_sens_headers.xlsm",
    CWD / "DCF_with_sens_headers.xlsm",
    OUTPUT_DIR / "DCF.xlsm",
    CWD / "DCF.xlsm",
]
MASTER_XLSM = next((p for p in MASTER_CANDIDATES if p.exists()), None)
if MASTER_XLSM is None:
    # Fallback: use any existing DCF_Output file as the template
    import glob as _g18
    _fallbacks = (
        _g18.glob(str(OUTPUT_DIR / "DCF_Output_*.xlsm")) +
        _g18.glob(str(OUTPUT_DIR / "*.xlsm"))
    )
    if _fallbacks:
        MASTER_XLSM = Path(sorted(_fallbacks)[0])
        print(f"⚠️  No template found — using existing file as base: {MASTER_XLSM.name}")
    else:
        raise FileNotFoundError(
            "No DCF template (DCF.xlsm) found next to the notebook. "
            "Please place DCF_with_sens_headers.xlsm or DCF.xlsm in the same folder."
        )

# One stable output file per ticker
OUTPUT_XLSM_PATH = (OUTPUT_DIR / f"DCF_Output_{TICKER}_{LOCAL_CURRENCY}.xlsm").resolve()

# Remove older duplicates for same ticker
for old in OUTPUT_DIR.glob(f"DCF_Output_{TICKER}_*.xlsm"):
    if old.resolve() != OUTPUT_XLSM_PATH:
        try: old.unlink()
        except Exception: pass

shutil.copyfile(MASTER_XLSM, OUTPUT_XLSM_PATH)

# All writing targets this one file
OUTPUT_XLSM   = str(OUTPUT_XLSM_PATH)
TEMPLATE_XLSM = OUTPUT_XLSM

print(f"✅ Template : {MASTER_XLSM.name}")
print(f"✅ Output   : {OUTPUT_XLSM_PATH.name}")
print(f"✅ Currency : {LOCAL_CURRENCY}  ({SYM})")


✅ Template : DCF.xlsm
✅ Output   : DCF_Output_RELIANCE.NS_INR.xlsm
✅ Currency : INR  (₹)


In [14]:
# ── 5B: Excel writing helpers ────────────────────────────────────────────────

def _resolve_sheet(wb, target):
    """Find sheet by name, ignoring leading/trailing spaces and case."""
    t = target.strip().lower()
    for s in wb.sheetnames:
        if s.strip().lower() == t:
            return s
    raise ValueError(f"Sheet '{target}' not found. Sheets: {wb.sheetnames}")

def _sanitize(val):
    """Convert NaN/Inf to None so Excel doesn't create 'Repaired' workbooks."""
    if val is None or val == "": return None
    try:
        import numpy as _np
        if isinstance(val, (_np.generic,)): val = val.item()
    except Exception: pass
    if isinstance(val, float) and (math.isnan(val) or math.isinf(val)): return None
    return val

def _coerce_num(x):
    try:
        if x is None: return None
        if isinstance(x, (np.generic,)): x = x.item()
        s = str(x).strip()
        if s.endswith("%"): return float(s[:-1]) / 100.0
        return float(s)
    except Exception: return None

def write_row(ws, start_cell, values):
    col_letters = "".join(c for c in start_cell if c.isalpha())
    row_idx     = int("".join(c for c in start_cell if c.isdigit()))
    start_col   = column_index_from_string(col_letters)
    for j, v in enumerate(values):
        ws.cell(row=row_idx, column=start_col + j, value=_sanitize(v))

def strip_external_links(wb):
    """Remove any external link formulas to prevent Excel 'Repaired' errors."""
    ext_pattern = re.compile(r"='?\[.+?\]|='?https?://|='?.+\.xl[sm][xm]'?!")
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for cell in row:
                if isinstance(cell.value, str) and cell.value.startswith("="):
                    if ext_pattern.search(cell.value):
                        cell.value = None  # clear broken external ref
    return wb

def safe_save(wb, path):
    """Strip external links then save — prevents 'Repaired' dialog."""
    wb = strip_external_links(wb)
    wb.save(path)

_HD, _SM, _SB, _MT = "1F3864", "2F5496", "D6E4F7", "F5F5F5"

# ── 3-Statement Model functions (write_sheet, build_is/bs/cf, wire_formulas) ──
def _f(bold=False, italic=False, color="000000", sz=10):
    return Font(name="Calibri", size=sz, bold=bold, italic=italic, color=color)
def _fl(h): return PatternFill("solid", fgColor=h)
def _al(h="left", v="center"): return Alignment(horizontal=h, vertical=v)
def _bd(): return Border(bottom=Side(style="thin"))

# ── Data helpers ──────────────────────────────────────────────────────────────
def _g(df, *keys):
    if df is None or df.empty: return None
    for k in keys:
        hits = [i for i in df.index if k.lower() in str(i).lower()]
        if hits: return df.loc[hits[0]]
    return None

def _v(s, j, scale=1_000_000):
    if s is None: return None
    try:
        v = s.values[j]
        if v is None or (isinstance(v, float) and math.isnan(v)): return None
        return float(v) / scale
    except: return None

def _dv(a, b):
    if a is None or b is None or b == 0: return None
    try: return a / b
    except: return None

# ── Sheet writer — returns (ws, row_addr_map) ─────────────────────────────────
def _clear(wb, title):
    t = title[:31]
    if t in wb.sheetnames: del wb[t]
    return wb.create_sheet(t)

def write_sheet(wb, title, sections, col_hdrs, sym="$"):
    """
    Write analyst sheet. Returns (ws, addr_map) where addr_map maps
    section['key'] → list of Excel address strings like "'Sheet'!C5"
    """
    ws  = _clear(wb, title)
    n   = len(col_hdrs)
    D   = 2   # data starts col B
    fmt_ccy = f'"{sym}"#,##0'
    addr_map = {}

    # Title row
    ws.row_dimensions[1].height = 22
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=D+n)
    tc = ws.cell(1, 1)
    tc.value = f"{title}  |  {currency_code}  |  In Millions"
    tc.font = _f(bold=True, color="FFFFFF", sz=12)
    tc.fill = _fl(_HD); tc.alignment = _al("left")

    # Col headers row 2
    ws.row_dimensions[2].height = 18
    ws.cell(2,1).value="Line Item"; ws.cell(2,1).font=_f(bold=True,color="FFFFFF")
    ws.cell(2,1).fill=_fl(_HD); ws.cell(2,1).alignment=_al("left")
    for j, h in enumerate(col_hdrs):
        c = ws.cell(2, D+j)
        c.value=h; c.font=_f(bold=True,color="FFFFFF")
        c.fill=_fl(_HD); c.alignment=_al("right")

    row = 3
    for sec in sections:
        t   = sec.get("type", "item")
        lbl = sec.get("label", "")
        vals= sec.get("values")
        fmt = sec.get("fmt", "currency")
        ind = sec.get("indent", 0)
        key = sec.get("key")

        ws.row_dimensions[row].height = 15

        if t == "spacer": row += 1; continue

        lc = ws.cell(row, 1)
        lc.value = ("  "*ind) + lbl
        lc.alignment = _al("left")

        if t == "header":
            lc.font = _f(bold=True, color="FFFFFF"); lc.fill = _fl(_SM)
            ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=D+n)
            row += 1; continue

        lc.font = (_f(bold=True) if t=="subtotal" else
                   _f(italic=True, color="555555") if t=="metric" else _f())
        if t == "subtotal": lc.fill = _fl(_SB); lc.border = _bd()
        elif t == "metric": lc.fill = _fl(_MT)

        col_addrs = []
        if vals is not None:
            for j, v in enumerate(vals):
                vc = ws.cell(row, D+j)
                if isinstance(v, str) and v.startswith("="):
                    vc.value = v   # pre-built formula
                elif v is None or (isinstance(v, float) and math.isnan(v)):
                    vc.value = None
                else:
                    try: vc.value = round(float(v), 2)
                    except: vc.value = v

                vc.alignment = _al("right")
                vc.font = (_f(bold=True) if t=="subtotal" else
                           _f(italic=True, color="555555") if t=="metric" else
                           _f(color="1F4E79"))
                if t == "subtotal": vc.fill=_fl(_SB); vc.border=_bd()
                elif t == "metric": vc.fill=_fl(_MT)

                if   fmt == "currency": vc.number_format = fmt_ccy
                elif fmt == "percent":  vc.number_format = "0.0%"
                elif fmt == "decimal":  vc.number_format = "0.00"
                elif fmt == "integer":  vc.number_format = "#,##0"
                else:                   vc.number_format = fmt_ccy

                col_addrs.append(f"'{title}'!{get_column_letter(D+j)}{row}")

            if key:
                addr_map[key] = col_addrs

        row += 1

    ws.column_dimensions["A"].width = 42
    for j in range(n):
        ws.column_dimensions[get_column_letter(D+j)].width = 14
    ws.freeze_panes = "B3"
    return ws, addr_map


# ── Section builders ──────────────────────────────────────────────────────────
def build_is(IS, BS, CF, n):
    def R(l,s,key=None,ind=1,fmt="currency"):
        return {"type":"item","label":l,"key":key,
                "values":[_v(s,j) for j in range(n)],"fmt":fmt,"indent":ind}
    def S(l,v,key=None): return {"type":"subtotal","label":l,"key":key,"values":v,"fmt":"currency","indent":0}
    def M(l,v,fmt="percent"): return {"type":"metric","label":l,"values":v,"fmt":fmt,"indent":1}
    def H(l): return {"type":"header","label":l}
    def SP(): return {"type":"spacer"}

    rev   = _g(IS,"Total Revenue","Revenue")
    cogs  = _g(IS,"Cost Of Revenue","Cost of Goods")
    gp    = _g(IS,"Gross Profit")
    sga   = _g(IS,"Selling General Administrative","Selling General And Administration",
                  "General And Administrative Expense","Selling And Marketing Expense")
    rd    = _g(IS,"Research Development","Research And Development")
    opex  = _g(IS,"Operating Expense","Total Operating Expenses")
    da    = _g(IS,"Reconciled Depreciation","Depreciation Amortization Depletion Income Statement",
                  "Depreciation And Amortization")
    ebit  = _g(IS,"Operating Income","Ebit")
    int_e = _g(IS,"Interest Expense")
    int_i = _g(IS,"Interest Income")
    pre   = _g(IS,"Pretax Income","Income Before Tax")
    tax   = _g(IS,"Tax Provision","Income Tax")
    ni    = _g(IS,"Net Income")
    shr   = _g(IS,"Diluted Average Shares","Basic Average Shares","Ordinary Shares Number")

    rev_v  = [_v(rev,j) for j in range(n)]
    gp_v   = [_v(gp,j) or ((_v(rev,j) or 0)-abs(_v(cogs,j) or 0)) for j in range(n)]
    da_v   = [_v(da,j) for j in range(n)]
    ebit_v = [_v(ebit,j) for j in range(n)]
    ni_v   = [_v(ni,j) for j in range(n)]
    ebitda_v = [(_v(ebit,j) or 0)+(_v(da,j) or 0) for j in range(n)]
    pre_v  = [_v(pre,j) for j in range(n)]
    tax_v  = [_v(tax,j) for j in range(n)]
    eps_v  = [_dv(ni_v[j], _v(shr,j)) for j in range(n)]

    return [
        H("REVENUE"),
        R("  Gross Revenue",rev,"is_revenue"),
        R("  Cost of Revenue",cogs),
        S("Gross Profit",gp_v,"is_gross_profit"),
        M("  Gross Margin %",[_dv(gp_v[j],rev_v[j]) for j in range(n)]),SP(),
        H("OPERATING EXPENSES"),
        R("  Selling, General & Administrative",sga),
        R("  Research & Development",rd),
        R("  Other Operating Expenses",opex),
        S("EBITDA",ebitda_v,"is_ebitda"),
        M("  EBITDA Margin %",[_dv(ebitda_v[j],rev_v[j]) for j in range(n)]),SP(),
        H("DEPRECIATION & AMORTIZATION"),
        R("  Total D&A",da,"is_da"),
        S("EBIT (Operating Income)",ebit_v,"is_ebit"),
        M("  EBIT Margin %",[_dv(ebit_v[j],rev_v[j]) for j in range(n)]),SP(),
        H("NON-OPERATING & TAX"),
        R("  Interest Income",int_i),
        R("  Interest Expense",int_e),
        {"type":"item","label":"  EBT (Pre-Tax Income)","key":"is_pretax",
         "values":pre_v,"fmt":"currency","indent":1},
        R("  Income Tax Expense",tax),
        M("  Effective Tax Rate %",[_dv(abs(_v(tax,j) or 0),pre_v[j]) for j in range(n)]),
        S("NET INCOME",ni_v,"is_net_income"),
        M("  Net Margin %",[_dv(ni_v[j],rev_v[j]) for j in range(n)]),SP(),
        H("PER SHARE"),
        M("  Diluted Shares (M)",[_v(shr,j) for j in range(n)],"integer"),
        {"type":"metric","label":"  EPS","key":"is_eps","values":eps_v,"fmt":"decimal","indent":1},
    ]


def build_bs(IS, BS, CF, n):
    def R(l,s,key=None,ind=1):
        return {"type":"item","label":l,"key":key,
                "values":[_v(s,j) for j in range(n)],"fmt":"currency","indent":ind}
    def S(l,v,key=None): return {"type":"subtotal","label":l,"key":key,"values":v,"fmt":"currency","indent":0}
    def M(l,v,fmt="decimal"): return {"type":"metric","label":l,"values":v,"fmt":fmt,"indent":1}
    def H(l): return {"type":"header","label":l}
    def SP(): return {"type":"spacer"}

    cash  = _g(BS,"Cash And Cash Equivalents","Cash")
    ar    = _g(BS,"Accounts Receivable","Net Receivables")
    inv   = _g(BS,"Inventory")
    prep  = _g(BS,"Prepaid Assets","Other Current Assets")
    tca   = _g(BS,"Current Assets","Total Current Assets")
    ppe   = _g(BS,"Net Ppe","Property Plant Equipment Net")
    gw    = _g(BS,"Goodwill")
    # Other intangibles = combined minus goodwill to prevent duplication
    _gw_combined = _g(BS,"Goodwill And Other Intangible Assets","Other Intangible Assets","Intangible Assets")
    oi_v  = [(_v(_gw_combined,j) or 0) - (_v(gw,j) or 0) for j in range(n)]
    ta    = _g(BS,"Total Assets")
    ap    = _g(BS,"Accounts Payable")
    std   = _g(BS,"Current Debt","Short Term Debt")
    tcl   = _g(BS,"Current Liabilities","Total Current Liabilities")
    ltd   = _g(BS,"Long Term Debt")
    tl    = _g(BS,"Total Liabilities Net Minority Interest","Total Liabilities")
    re    = _g(BS,"Retained Earnings")
    equity= _g(BS,"Stockholders Equity","Total Equity Gross Minority Interest")

    rev_s = _g(IS,"Total Revenue","Revenue")
    cogs_s= _g(IS,"Cost Of Revenue","Cost of Goods")

    tca_v = [_v(tca,j) for j in range(n)]
    ta_v  = [_v(ta,j) for j in range(n)]
    tcl_v = [_v(tcl,j) for j in range(n)]
    tl_v  = [_v(tl,j) for j in range(n)]
    eq_v  = [_v(equity,j) for j in range(n)]
    cash_v= [_v(cash,j) for j in range(n)]
    ar_v  = [_v(ar,j) for j in range(n)]
    inv_v = [_v(inv,j) for j in range(n)]
    ap_v  = [_v(ap,j) for j in range(n)]

    # Other CL = TCL - AP - STD  (prevents double-counting)
    other_cl_v = [max(0, (_v(tcl,j) or 0) - abs(_v(ap,j) or 0) - abs(_v(std,j) or 0))
                  for j in range(n)]

    # Total L+E always computed from components
    tleq_v = [(_v(tl,j) or 0) + (_v(equity,j) or 0) for j in range(n)]

    # Balance check = Total Assets - Total L+E  (must be 0)
    bal_v  = [round((_v(ta,j) or 0) - tleq_v[j], 2) for j in range(n)]

    # Non-current assets (total - current)
    nca_v  = [(_v(ta,j) or 0) - (_v(tca,j) or 0) for j in range(n)]
    ncl_v  = [(_v(tl,j) or 0) - (_v(tcl,j) or 0) for j in range(n)]
    nwc_v  = [(_v(tca,j) or 0) - (_v(tcl,j) or 0) for j in range(n)]

    # Other current assets (unlisted items like S-T investments)
    other_ca_v = [(_v(tca,j) or 0) - (_v(cash,j) or 0) - (_v(ar,j) or 0)
                  - (_v(inv,j) or 0) - (_v(prep,j) or 0) for j in range(n)]

    dso  = [_dv(_v(ar,j), _v(rev_s,j)) * 365 if _v(ar,j) and _v(rev_s,j) else None for j in range(n)]
    dio  = [_dv(_v(inv,j), abs(_v(cogs_s,j))) * 365 if _v(inv,j) and _v(cogs_s,j) else None for j in range(n)]
    dpo  = [_dv(_v(ap,j), abs(_v(cogs_s,j))) * 365 if _v(ap,j) and _v(cogs_s,j) else None for j in range(n)]
    ccc  = [(dso[j] or 0) + (dio[j] or 0) - (dpo[j] or 0) if dso[j] else None for j in range(n)]
    de   = [_dv(_v(ltd,j), eq_v[j]) for j in range(n)]

    return [
        H("CURRENT ASSETS"),
        {"type":"item","label":"    Cash & Cash Equivalents","key":"bs_cash",
         "values":cash_v,"fmt":"currency","indent":1},
        R("    Accounts Receivable",ar,"bs_ar"),
        R("    Inventory",inv,"bs_inv"),
        R("    Prepaid Expenses",prep),
        {"type":"item","label":"    Other Current Assets (S-T investments etc.)",
         "values":other_ca_v,"fmt":"currency","indent":1},
        S("Total Current Assets",tca_v,"bs_tca"),SP(),
        H("NON-CURRENT ASSETS"),
        R("    Net PP&E",ppe,"bs_ppe"),
        R("    Goodwill",gw,"bs_goodwill"),
        {"type":"item","label":"    Other Intangible Assets","key":"bs_intangibles",
         "values":oi_v,"fmt":"currency","indent":1},
        S("Total Non-Current Assets",nca_v),
        S("TOTAL ASSETS",ta_v,"bs_total_assets"),SP(),
        H("CURRENT LIABILITIES"),
        R("    Accounts Payable",ap,"bs_ap"),
        R("    Short-Term Debt",std),
        {"type":"item","label":"    Other Current Liabilities",
         "values":other_cl_v,"fmt":"currency","indent":1},
        S("Total Current Liabilities",
          [(_v(ap,j) or 0)+(_v(std,j) or 0)+other_cl_v[j] for j in range(n)],
          "bs_tcl"),SP(),
        H("NON-CURRENT LIABILITIES"),
        R("    Long-Term Debt",ltd,"bs_ltd"),
        S("Total Non-Current Liabilities",ncl_v),
        S("TOTAL LIABILITIES",tl_v,"bs_total_liabilities"),SP(),
        H("SHAREHOLDERS' EQUITY"),
        {"type":"item","label":"    Retained Earnings","key":"bs_retained_earnings",
         "values":[_v(re,j) for j in range(n)],"fmt":"currency","indent":1},
        R("    Total Equity",equity,"bs_equity"),
        S("TOTAL LIABILITIES + EQUITY",tleq_v,"bs_total_le"),
        {"type":"metric","label":"  ✓ Balance Check (must = 0)",
         "key":"bs_balance_check","values":bal_v,"fmt":"currency","indent":1},SP(),
        H("WORKING CAPITAL METRICS"),
        M("  Net Working Capital",nwc_v,"currency"),
        M("  DSO (days)",dso), M("  DIO (days)",dio),
        M("  DPO (days)",dpo), M("  Cash Conversion Cycle",ccc),
        M("  Debt / Equity",de),
    ]


def build_cf(IS, BS, CF, n, is_map=None, bs_map=None):
    def R(l,s,key=None,ind=1):
        return {"type":"item","label":l,"key":key,
                "values":[_v(s,j) for j in range(n)],"fmt":"currency","indent":ind}
    def S(l,v,key=None): return {"type":"subtotal","label":l,"key":key,"values":v,"fmt":"currency","indent":0}
    def M(l,v,fmt="percent"): return {"type":"metric","label":l,"values":v,"fmt":fmt,"indent":1}
    def H(l): return {"type":"header","label":l}
    def SP(): return {"type":"spacer"}

    ni    = _g(IS,"Net Income")
    da    = _g(IS,"Reconciled Depreciation","Depreciation Amortization Depletion Income Statement","Depreciation And Amortization")
    cfo   = _g(CF,"Operating Cash Flow","Cash From Operations","Net Cash From Operating Activities")
    capex = _g(CF,"Capital Expenditure","Purchase Of Ppe","Capital Expenditures")
    cfi   = _g(CF,"Investing Cash Flow","Net Cash From Investing Activities")
    dbt_i = _g(CF,"Issuance Of Debt","Proceeds From Long Term Debt Issuance","Debt Issuance")
    dbt_r = _g(CF,"Repayment Of Debt","Long Term Debt Payments")
    div   = _g(CF,"Payment Of Dividends","Common Stock Dividend Paid","Dividends Paid")
    bb    = _g(CF,"Repurchase Of Capital Stock","Common Stock Repurchased","Share Buyback")
    cff   = _g(CF,"Financing Cash Flow","Net Cash From Financing Activities")
    beg   = _g(CF,"Beginning Cash Position","Cash And Cash Equivalents Beginning Period")
    end_c = _g(CF,"End Cash Position","Cash And Cash Equivalents End Period","Changes In Cash")
    chg_ar  = _g(CF,"Change In Receivables","Changes In Accounts Receivable")
    chg_inv = _g(CF,"Change In Inventory")
    chg_ap  = _g(CF,"Change In Payable","Changes In Accounts Payable")
    chg_wc  = _g(CF,"Change In Working Capital")
    rev     = _g(IS,"Total Revenue","Revenue")

    cfo_v   = [_v(cfo,j) for j in range(n)]
    capex_v = [_v(capex,j) for j in range(n)]
    cfi_v   = [_v(cfi,j) for j in range(n)]
    cff_v   = [_v(cff,j) for j in range(n)]
    ni_v    = [_v(ni,j) for j in range(n)]
    da_v    = [_v(da,j) for j in range(n)]
    end_v   = [_v(end_c,j) for j in range(n)]
    beg_v   = [_v(beg,j) for j in range(n)]
    fcf_v   = [(cfo_v[j] or 0)+(capex_v[j] or 0) for j in range(n)]

    # ΔAR, ΔInv, ΔAP from BS when available
    ar_s  = _g(BS,"Accounts Receivable","Net Receivables")
    inv_s = _g(BS,"Inventory")
    ap_s  = _g(BS,"Accounts Payable")
    ar_d  = [-(_v(ar_s,j)-_v(ar_s,j-1)) if j>0 and _v(ar_s,j) and _v(ar_s,j-1) else _v(chg_ar,j) for j in range(n)]
    inv_d = [-(_v(inv_s,j)-_v(inv_s,j-1)) if j>0 and _v(inv_s,j) and _v(inv_s,j-1) else _v(chg_inv,j) for j in range(n)]
    ap_d  = [(_v(ap_s,j)-_v(ap_s,j-1)) if j>0 and _v(ap_s,j) and _v(ap_s,j-1) else _v(chg_ap,j) for j in range(n)]

    # Other operating items = CFO - listed items (reconciling row)
    other_ops_v = [(cfo_v[j] or 0)-(ni_v[j] or 0)-(da_v[j] or 0)
                   -(ar_d[j] or 0)-(inv_d[j] or 0)-(ap_d[j] or 0) for j in range(n)]

    return [
        H("OPERATING ACTIVITIES"),
        {"type":"item","label":"    Net Income","key":"cf_net_income",
         "values":ni_v,"fmt":"currency","indent":1},
        {"type":"item","label":"    Add: D&A","key":"cf_da",
         "values":da_v,"fmt":"currency","indent":1},
        {"type":"item","label":"    ΔAccounts Receivable","key":"cf_d_ar",
         "values":ar_d,"fmt":"currency","indent":1},
        {"type":"item","label":"    ΔInventory","key":"cf_d_inv",
         "values":inv_d,"fmt":"currency","indent":1},
        {"type":"item","label":"    ΔAccounts Payable","key":"cf_d_ap",
         "values":ap_d,"fmt":"currency","indent":1},
        {"type":"item","label":"    Other Working Capital Changes",
         "values":[_v(chg_wc,j) for j in range(n)],"fmt":"currency","indent":1},
        {"type":"item","label":"    Other Operating Items (SBC, Deferred Tax, etc.)",
         "values":other_ops_v,"fmt":"currency","indent":1},
        S("Cash from Operations (CFO)",cfo_v,"cf_cfo"),
        M("  CFO Margin %",[_dv(cfo_v[j],_v(rev,j)) for j in range(n)]),SP(),
        H("INVESTING ACTIVITIES"),
        R("    Capital Expenditure (CapEx)",capex,"cf_capex"),
        S("Cash from Investing (CFI)",cfi_v,"cf_cfi"),SP(),
        H("FINANCING ACTIVITIES"),
        R("    Debt Raised",dbt_i), R("    Debt Repaid",dbt_r),
        {"type":"item","label":"    Dividends Paid","key":"cf_dividends",
         "values":[_v(div,j) for j in range(n)],"fmt":"currency","indent":1},
        R("    Share Buybacks / Issuance",bb),
        S("Cash from Financing (CFF)",cff_v,"cf_cff"),SP(),
        H("CASH RECONCILIATION"),
        R("    Beginning Cash","cf_beg_cash"),
        {"type":"item","label":"    Ending Cash → Balance Sheet","key":"cf_ending_cash",
         "values":end_v,"fmt":"currency","indent":1},SP(),
        H("FREE CASH FLOW"),
        S("Free Cash Flow (CFO + CapEx)",fcf_v,"cf_fcf"),
        M("  FCF Margin %",[_dv(fcf_v[j],_v(rev,j)) for j in range(n)]),
    ]


# ── Pass 2: Wire live cross-sheet formulas ───────────────────────────────────
def wire_formulas(wb, is_map, bs_map, cf_map, n):
    """
    Write Excel cross-sheet formulas into the three sheets.
    All IS/BS/CF values are already written as hardcoded numbers.
    This pass REPLACES key cells with live formulas so the model is connected.
    """
    def _ws(title):
        return wb[next(s for s in wb.sheetnames if s.strip().lower()==title.strip().lower())]

    def _addr(addr_str):
        m = re.match(r"'?([^'!]+)'?!([A-Z]+\d+)", addr_str)
        return (m.group(1), m.group(2)) if m else (None, None)

    ws_is = _ws(IS_SHEET)
    ws_bs = _ws(BS_SHEET)
    ws_cf = _ws(CF_SHEET)

    def _write(ws, addr_str, formula):
        _, cell_ref = _addr(addr_str)
        if not cell_ref: return
        cell = ws[cell_ref]
        if isinstance(cell, MergedCell): return
        cell.value = formula

    # 1. CF Net Income ← IS Net Income
    if "is_net_income" in is_map and "cf_net_income" in cf_map:
        for j in range(n):
            _write(ws_cf, cf_map["cf_net_income"][j], f"={is_map['is_net_income'][j]}")

    # 2. CF D&A ← IS D&A
    if "is_da" in is_map and "cf_da" in cf_map:
        for j in range(n):
            _write(ws_cf, cf_map["cf_da"][j], f"={is_map['is_da'][j]}")

    # 3. CF ΔAR ← BS AR(curr) - BS AR(prev)
    for bs_key, cf_key, sign in [("bs_ar","cf_d_ar",-1),("bs_inv","cf_d_inv",-1),("bs_ap","cf_d_ap",+1)]:
        if bs_key not in bs_map or cf_key not in cf_map: continue
        for j in range(n):
            if j == 0: continue  # no prior year
            curr = bs_map[bs_key][j]; prev = bs_map[bs_key][j-1]
            formula = f"=-({curr}-{prev})" if sign==-1 else f"={curr}-{prev}"
            _write(ws_cf, cf_map[cf_key][j], formula)

    # 4. CF Ending Cash formula = Beginning + CFO + CFI + CFF
    if "cf_ending_cash" in cf_map and "cf_cfo" in cf_map:
        beg_map = cf_map.get("cf_beg_cash") or []
        for j in range(n):
            dest = cf_map["cf_ending_cash"][j]
            _, cell_ref = _addr(dest)
            if not cell_ref: continue
            cfo_ref = cf_map["cf_cfo"][j]
            cfi_ref = cf_map.get("cf_cfi", [None]*n)[j]
            cff_ref = cf_map.get("cf_cff", [None]*n)[j]
            # Beginning of year j = ending cash of year j-1 (chain)
            if j > 0:
                beg_ref = cf_map["cf_ending_cash"][j-1]
                # Write the chain formula to the beg_cash cell
                if beg_map and j < len(beg_map) and beg_map[j]:
                    _, beg_cell = _addr(beg_map[j])
                    prev_end_sheet, prev_end_cell = _addr(cf_map["cf_ending_cash"][j-1])
                    if beg_cell and prev_end_cell:
                        ws_cf[beg_cell].value = f"={prev_end_cell}"
            elif beg_map and j < len(beg_map) and beg_map[j]:
                beg_ref = beg_map[j]    # explicit beginning cash row (already has value)
            else:
                beg_ref = None          # first year, no prior
            parts = [r for r in [beg_ref, cfo_ref, cfi_ref, cff_ref] if r]
            if parts:
                ws_cf[cell_ref].value = "=" + "+".join(parts)

    # 5. BS Cash ← CF Ending Cash
    if "cf_ending_cash" in cf_map and "bs_cash" in bs_map:
        for j in range(n):
            _write(ws_bs, bs_map["bs_cash"][j], f"={cf_map['cf_ending_cash'][j]}")

    # 6. BS Retained Earnings = prior RE + IS Net Income + CF Dividends
    if "bs_retained_earnings" in bs_map and "is_net_income" in is_map:
        for j in range(1, n):  # skip first year (no prior)
            dest = bs_map["bs_retained_earnings"][j]
            ni_ref   = is_map["is_net_income"][j]
            prev_re  = bs_map["bs_retained_earnings"][j-1]
            div_ref  = cf_map.get("cf_dividends", [None]*n)[j]
            formula  = f"={prev_re}+{ni_ref}+{div_ref}" if div_ref else f"={prev_re}+{ni_ref}"
            _write(ws_bs, dest, formula)

        # 6. BS: Retained Earnings = prior RE + IS Net Income + CF Dividends
    if "bs_retained_earnings" in bs_map and "is_net_income" in is_map:
        for j in range(1, n):
            dest = bs_map["bs_retained_earnings"][j]
            _, cell_ref = _addr(dest)
            if not cell_ref: continue
            ni_ref  = is_map["is_net_income"][j]
            prior_re = bs_map["bs_retained_earnings"][j-1]
            div_ref = cf_map.get("cf_dividends",[None]*n)[j]
            ws_bs[cell_ref].value = f"={prior_re}+{ni_ref}+{div_ref}" if div_ref else f"={prior_re}+{ni_ref}"

# 7. BS Balance Check = Total Assets - Total L+E
    if "bs_total_assets" in bs_map and "bs_total_le" in bs_map and "bs_balance_check" in bs_map:
        for j in range(n):
            ta_ref = bs_map["bs_total_assets"][j]
            le_ref = bs_map["bs_total_le"][j]
            _write(ws_bs, bs_map["bs_balance_check"][j], f"={ta_ref}-{le_ref}")

    print("  ✅ Cross-sheet formulas wired:")
    print("     CF Net Income / D&A ← Income Statement")
    print("     CF ΔAR / ΔInv / ΔAP ← Balance Sheet (curr−prior)")
    print("     CF Ending Cash = Beg + CFO + CFI + CFF (formula)")
    print("     BS Cash ← CF Ending Cash")
    print("     BS Retained Earnings = prior RE + Net Income + Dividends")
    print("     BS Balance Check = Total Assets − Total L+E")





In [15]:
# ── 5C–5I: Write DCF sheet (headers + hardcoded values + formulas) ──────────

# Guard: ensure HIST is available before writing DCF sheet
_hist_g = globals().get('HIST')
if _hist_g is None or (hasattr(_hist_g, 'empty') and _hist_g.empty):
    raise RuntimeError(
        "HIST is not built yet — run cells in order: Part 0 → Part 3 → Part 4 → Part 5."
    )

wb = load_workbook(TEMPLATE_XLSM, keep_vba=True)
DCF_SHEET = _resolve_sheet(wb, "DCF")
ws = wb[DCF_SHEET]

# ── FY headers ────────────────────────────────────────────────────────────────
hist_years = sorted(HIST["FY"].dropna().astype(int).tolist())[-3:]
fore_years = sorted(FORE["FY"].dropna().astype(int).tolist())[:5]
while len(hist_years) < 3: hist_years.insert(0, hist_years[0]-1)
while len(fore_years) < 5: fore_years.append(fore_years[-1]+1)
all_years = hist_years + fore_years
for j, col in enumerate(["E","F","G","H","I","J","K","L"]):
    ws[f"{col}4"]  = all_years[j]; ws[f"{col}4"].number_format  = "0"
    ws[f"{col}22"] = all_years[j]; ws[f"{col}22"].number_format = "0"
print(f"✅ FY headers → Past:{hist_years}  Fore:{fore_years}")

# ── 5D: Historical values (E-G cols, rows 24-31) — hard values from HIST/FORE ─
def ensure_cols(df, tax_rate):
    out = df.copy()
    tr = tax_rate / 100.0 if tax_rate > 1 else tax_rate
    if "EBIT" in out.columns:
        out["Taxes"] = out["EBIT"] * tr
        if "NOPAT" not in out.columns: out["NOPAT"] = out["EBIT"] * (1 - tr)
    else: out["Taxes"] = np.nan
    if all(c in out.columns for c in ["Capex","DeltaNWC","DandA"]):
        out["TotalNetInvestment"] = out["Capex"] + out["DeltaNWC"] - out["DandA"]
    else: out["TotalNetInvestment"] = np.nan
    return out

def get_hist_vals(df, col, n=3):
    h = df.dropna(subset=[col]).sort_values("FY").tail(n)[col].tolist() if col in df.columns else []
    return ([None]*(n-len(h))) + h

HIST2 = ensure_cols(HIST, TAX_RATE)

# Write historical cols E, F, G (rows 24-31)
hist_row_map = {
    "Revenue":24, "EBIT":25, "Taxes":26, "NOPAT":27,
    "Capex":28, "DeltaNWC":29, "DandA":30, "TotalNetInvestment":31
}
for col_name, row in hist_row_map.items():
    vals = get_hist_vals(HIST2, col_name, 3)
    for j, col in enumerate(["E","F","G"]):
        ws[f"{col}{row}"] = _sanitize(vals[j])

# Also write historical FCF row 34
hist_fcff = HIST.dropna(subset=["FCFF"]).sort_values("FY").tail(3)["FCFF"].tolist()
hist_fcff = ([None]*(3-len(hist_fcff))) + hist_fcff
for j, col in enumerate(["E","F","G"]):
    ws[f"{col}34"] = _sanitize(hist_fcff[j])
# ══════════════════════════════════════════════════════════════════
# PERMANENT FIXES — Historical corrections (Issues 1-5, 9 from audit)
# ══════════════════════════════════════════════════════════════════

# FIX-1: Clear ghost "234" hardcodes in ratio rows
ws["E14"] = None   # Revenue growth undefined for first hist year (no prior year)
ws["E18"] = None   # WC% undefined for first hist year

# FIX-2: EBITDA rows E9:G9 — formula not hardcoded value
for col in ["E","F","G"]:
    ws[f"{col}9"] = f"={col}7+{col}8"   # EBITDA = EBIT + D&A

# FIX-3: Historical WC (DeltaNWC) for rows 29 and Net Investment row 31
hist_nwc = HIST.dropna(subset=["DeltaNWC"]).sort_values("FY").tail(3)["DeltaNWC"].tolist()
hist_nwc = ([None]*(3-len(hist_nwc))) + hist_nwc
for j, col in enumerate(["E","F","G"]):
    ws[f"{col}29"] = _sanitize(hist_nwc[j])  # Historical WC requirements
for col in ["E","F","G"]:
    ws[f"{col}31"] = f"=SUM({col}28:{col}30)"  # Total Net Investment formula

# FIX-4: Clear stale billion-scale values from Ratios row 13
for col in ["E","F","G","H","I","J","K","L"]:
    ws[f"{col}13"] = None


# ── 5E: Projection block top (rows 6-11) — formulas referencing key assumptions
# The template uses rows 6-11 for a projection summary that feeds into DCF rows
# R10=RevGrowth, R11=EBITMargin, R14=DApct, R15=CapexPct, R16=WCpct
proj_cols = ["H","I","J","K","L"]
for idx, col in enumerate(proj_cols):
    prev = "G" if idx == 0 else proj_cols[idx-1]
    ws[f"{col}6"]  = f"={prev}6*(1+$R$10)"          # Revenue
    ws[f"{col}7"]  = f"={col}6*$R$11"               # EBIT
    ws[f"{col}8"]  = f"={col}6*$R$14"               # D&A
    ws[f"{col}9"]  = f"={col}7+{col}8"              # EBITDA
    ws[f"{col}10"] = f"=-{col}6*$R$15"              # Capex
    ws[f"{col}11"] = f"=({prev}6-{col}6)*$R$16"     # WC Investment

# ── 5F: Seed G6 with last historical revenue so projection chain works ─────
# Seed all 3 historical cols E,F,G for rows 6-11 from HIST
_rev_vals = get_hist_vals(HIST, "Revenue", 3)
_ebit_vals = get_hist_vals(HIST, "EBIT", 3)
_da_vals   = get_hist_vals(HIST, "DandA", 3)
_cap_vals  = get_hist_vals(HIST, "Capex", 3)
_nwc_vals  = get_hist_vals(HIST, "DeltaNWC", 3)
for _j, _col in enumerate(["E","F","G"]):
    ws[f"{_col}6"]  = _sanitize(_rev_vals[_j])
    ws[f"{_col}7"]  = _sanitize(_ebit_vals[_j])
    ws[f"{_col}8"]  = _sanitize(_da_vals[_j])
    ws[f"{_col}10"] = _sanitize(_cap_vals[_j])
    ws[f"{_col}11"] = _sanitize(_nwc_vals[_j])
# G6 is the seed for the H-L projection chain
last_hist_rev = _rev_vals[-1]
# Link E6:G7 to lower block for consistency
for _c in ["E","F"]:
    ws[f"{_c}6"] = f"={_c}24"
    ws[f"{_c}7"] = f"={_c}25"
    ws[f"{_c}8"] = f"={_c}30"
    ws[f"{_c}10"] = f"={_c}28"
    ws[f"{_c}11"] = f"={_c}29"
ws["G6"] = _sanitize(last_hist_rev)

# ── 5G: DCF analysis block rows 22-31, 34 (forecast cols H-L) — formulas ─────
dcf_cols = ["E","F","G","H","I","J","K","L"]

# Row 22: FY = row 4
for c in dcf_cols:
    ws[f"{c}22"] = f"={c}4"

# Rows 24-31: reference rows 6-11 for forecast cols H-L
# Historical E-G already written as values above
for c in ["H","I","J","K","L"]:
    ws[f"{c}24"] = f"={c}6"                          # Revenue
    ws[f"{c}25"] = f"={c}7"                          # EBIT
    ws[f"{c}26"] = f"={c}25*-$R$12"                 # Taxes
    ws[f"{c}27"] = f"={c}25+{c}26"                  # NOPAT
    ws[f"{c}28"] = f"={c}10"                         # Capex
    ws[f"{c}29"] = f"={c}11"                         # WC Requirements
    ws[f"{c}30"] = f"={c}8"                          # D&A
    ws[f"{c}31"] = f"=SUM({c}28:{c}30)"             # Total Net Investment

# Row 34: Unlevered FCF = NOPAT + Net Investment
for c in dcf_cols:
    ws[f"{c}34"] = f"={c}27+{c}31"

# ── 5H: Discount period row 35 — formula chain from R6 (stub) ─────────────
ws["H35"] = "=R6"
ws["I35"] = "=H35+1"
ws["J35"] = "=I35+1"
ws["K35"] = "=J35+1"
ws["L35"] = "=K35+1"

# ── 5I: Discount rate row 36 — references Cost of Capital!D22 (WACC) ─────
# H36 pulls WACC live from Cost of Capital sheet so changing inputs there
# automatically recalculates the entire DCF without re-running the notebook
_coc_tab = _resolve_sheet(wb, "Cost of Capital")
ws["H36"] = f"='{_coc_tab}'!D22"
ws["H36"].number_format = "0.00%"
for c in ["I","J","K","L"]:
    prev = ["H","I","J","K"][["I","J","K","L"].index(c)]
    ws[f"{c}36"] = f"={prev}36"

# ── 5J: PV of FCF row 37 — formula ────────────────────────────────────────
for c in ["H","I","J","K","L"]:
    ws[f"{c}37"] = f"={c}34/(1+{c}36)^{c}35"

# ── 5K: Terminal Value block (col P) — all formulas ──────────────────────
ws["P6"]  = "=(L34*(1+R18))/(H36-R18)"    # Terminal Value
ws["P7"]  = "=L35"                          # Discount Period
ws["P8"]  = "=H36"                          # Discount Rate
ws["P9"]  = "=P6/(1+P8)^P7"               # PV of TV
ws["P13"] = "=SUM(H37:L37)"               # PV of Proj. Period FCFF
ws["P23"] = "=P9+P13"                      # Implied TEV
ws["P17"] = "=P13/P23"                     # Period Cash Flow %
ws["P18"] = "=1-P17"                       # Terminal Cash Flow %
ws["P19"] = "=P17+P18"                     # Total
ws["P29"] = "=P23-P24-P25-P26+P27"        # Implied Equity Value
ws["P30"] = "=P29/R25"                     # Implied Share Price

# ── 5L: Stub period — formula ─────────────────────────────────────────────
today_dt = date.today()
try:    first_fore_year = int(FORE["FY"].min())
except: first_fore_year = BASE_YEAR + 1
_priv_fy = not SOURCES.get("api", False)
_fym, _fyd = (12, 31) if _priv_fy else (6, 30)
fy_end = date(first_fore_year, _fym, _fyd)
if fy_end <= today_dt:
    fy_end = date(first_fore_year + 1, _fym, _fyd)
ws["R7"] = fy_end;  ws["R7"].number_format = "mm/dd/yyyy"
ws["R8"] = today_dt; ws["R8"].number_format = "mm/dd/yyyy"
ws["R6"] = "=(R7-R8)/365"                 # Stub period formula

# ── 5M: Key Assumptions ───────────────────────────────────────────────────
ws["R10"] = round(drivers["RevenueGrowth_default"], 4); ws["R10"].number_format = "0.00%"
ws["R11"] = round(drivers["EBITMargin_default"], 4);    ws["R11"].number_format = "0.00%"
ws["R12"] = round(TAX_RATE / 100.0, 4);                 ws["R12"].number_format = "0.00%"
ws["R14"] = round(drivers["DApct_default"], 4);         ws["R14"].number_format = "0.00%"
ws["R15"] = round(drivers["CapexPct_default"], 4);      ws["R15"].number_format = "0.00%"
ws["R16"] = round(drivers["NWCpct_default"], 4);        ws["R16"].number_format = "0.00%"
ws["R18"] = round(TERMINAL_GROWTH / 100.0, 4);          ws["R18"].number_format = "0.00%"
ws["R21"] = 0.0025; ws["R21"].number_format = "0.00%"   # g increment
ws["R22"] = 0.0025; ws["R22"].number_format = "0.00%"   # WACC increment

# ── 5N: Shares & Capital structure ────────────────────────────────────────
# Shares in Millions (already normalized in build_historical)
_shares = None
try:
    _sh_series = HIST["DilutedShares"].dropna()
    if not _sh_series.empty:
        _shares = float(_sh_series.iloc[-1])
        print(f"   Shares: {_shares:,.2f}M")
except Exception: pass

if (_shares is None or _shares == 0) and SOURCES.get("api"):
    try:
        _info2  = yf.Ticker(TICKER).info or {}
        _raw_sh = _info2.get("sharesOutstanding") or _info2.get("impliedSharesOutstanding")
        if _raw_sh:
            _shares = float(_raw_sh) / 1_000_000.0
            print(f"   Shares from yfinance: {_shares:,.2f}M")
    except Exception: pass

_price = None
if SOURCES.get("api"):
    try:
        _info2 = yf.Ticker(TICKER).info or {}
        _price = _info2.get("currentPrice") or _info2.get("regularMarketPrice")
    except Exception: pass

_priv_sh = not SOURCES.get("api", False)
# For private companies use actual shares extracted from files; fall back to 1.0
_shares_write = (_shares if _shares and _shares > 0 else 1.0)
ws["R24"] = _sanitize(_shares_write)
ws["R24"].number_format = "#,##0.00"
# R25: private → hardcoded value from source files (no formula); public → IS live link
_is_link = next((s for s in wb.sheetnames if "income" in s.strip().lower()), None)
if _priv_sh:
    ws["R25"] = _sanitize(_shares_write)
    ws["R27"] = None  # no market price for private company
else:
    ws["R25"] = f"='{_is_link}'!F31" if _is_link else _sanitize(_shares)
    ws["R27"] = _sanitize(_price)
ws["R25"].number_format = "#,##0.00"
if ws["R27"].value is not None: ws["R27"].number_format = "#,##0.00"

# Capital structure — ALWAYS read from HIST (already in display units).
# HIST was built from normalized BS (cell 10 divided by unit_factor).
# BS global is also normalized. Never divide again.
last_hist_row = HIST.sort_values("FY").iloc[-1]
_debt   = float(last_hist_row["DebtBal"])    if pd.notna(last_hist_row.get("DebtBal"))   else 0.0
_cash   = float(last_hist_row["CashBal"])    if pd.notna(last_hist_row.get("CashBal"))   else 0.0
_equity = float(last_hist_row["TotalEquity"]) if pd.notna(last_hist_row.get("TotalEquity")) else 0.0
_pref   = 0.0
_minint = 0.0

# For API companies: BS is already normalized by cell 10 (normalize_units ÷ unit_factor).
# Use BS directly — NO additional division needed.
if SOURCES.get("api") and not BS.empty:
    try:
        _bs_latest = BS.sort_values("FY").iloc[-1]
        def _bs_get(cols):
            for c in cols:
                if c in _bs_latest.index and pd.notna(_bs_latest[c]):
                    return float(_bs_latest[c])
            return 0.0
        # BS values are already in display units (e.g. Millions) — use directly
        _api_debt = _bs_get(["Total Debt", "Short Long Term Debt", "Long Term Debt"])
        _api_cash = _bs_get(["Cash And Cash Equivalents",
                              "Cash Cash Equivalents And Short Term Investments", "Cash"])
        _api_eq   = _bs_get(["Stockholders Equity", "Total Equity Gross Minority Interest",
                              "Common Stock Equity"])
        if _api_debt > 0: _debt   = _api_debt   # already in display units
        if _api_cash > 0: _cash   = _api_cash
        if _api_eq   > 0: _equity = _api_eq
        _pref   = _bs_get(["Preferred Stock", "PreferredStock"])   # already normalized
        _minint = _bs_get(["Minority Interest", "MinorityInterest"])
    except Exception: pass

print(f"   C9 Debt  = {_debt:,.0f} {CURRENCY_UNITS} {LOCAL_CURRENCY}")
print(f"   Cash     = {_cash:,.0f} {CURRENCY_UNITS} {LOCAL_CURRENCY}")
print(f"   Equity   = {_equity:,.0f} {CURRENCY_UNITS} {LOCAL_CURRENCY}")

ws["P24"] = "='Cost of Capital'!C9"       # Debt — cross-sheet formula
ws["P25"] = _sanitize(_pref)
ws["P26"] = _sanitize(_minint)
# P27 Cash — live link to Balance Sheet latest ending cash (row 4, last data col)
# BS sheet is written later so we fall back to hardcoded HIST cash value here;
# Part 9 re-runs will update it. For now use the best available cash value.
_bs_link = next((s for s in wb.sheetnames if "balance sheet" in s.strip().lower()), None)
if _bs_link:
    ws["P27"] = f"='{_bs_link}'!F4"
else:
    # Robust fallback: try HIST CashBal, then EndCash, then 0
    _cash_p27 = _cash
    if (not _cash_p27 or _cash_p27 == 0):
        try:
            _h27 = globals().get("HIST")
            if _h27 is not None and not _h27.empty:
                _lr = _h27.sort_values("FY").iloc[-1]
                _cash_p27 = float(_lr.get("CashBal") or _lr.get("EndCash") or _lr.get("Cash") or 0)
        except Exception: pass
    ws["P27"] = _sanitize(_cash_p27)

# FIX A3: Clear H13:L13 orphan billion-dollar values
for _col in ["E","F","G","H","I","J","K","L"]:
    ws[f"{_col}13"] = None

safe_save(wb, TEMPLATE_XLSM)
_net_debt  = (_debt + _pref + _minint) - _cash
print(f"✅ DCF sheet written with Excel formulas")
print(f"   Debt={_debt:.1f}M  Cash={_cash:.1f}M  NetDebt={_net_debt:.1f}M")
print(f"   Shares={_shares}M  Price={_price}")


# ── Write IS / BS / CF sheets ─────────────────────────────────────────────────
try:
    import datetime as _dt27, math as _m27
    import pandas as _pd27

    _M27 = 1_000_000
    _yrs27 = [_dt27.datetime(y,6,30) for y in [2021,2022,2023,2024,2025]]
    def _mk27(d):
        _df=_pd27.DataFrame(d,index=_yrs27).T; _df.columns=_yrs27; return _df
    def _norm27(df,n=5):
        if df is None or (hasattr(df,"empty") and df.empty): return _pd27.DataFrame()
        df=df.copy(); df.columns=_pd27.to_datetime(df.columns,errors="coerce")
        return df.loc[:,df.columns.notna()].sort_index(axis=1).iloc[:,-n:]

    def _first_ne(*keys):
        for k in keys:
            v = globals().get(k)
            if v is not None and hasattr(v,"empty") and not v.empty: return v
        return None
    _ISg=_first_ne("IS","IS_raw","yf_is")
    _BSg=_first_ne("BS","BS_raw","yf_bs")
    _CFg=_first_ne("CF","CF_raw","yf_cf")
    _hv=lambda df: df is not None and not(hasattr(df,"empty") and df.empty)

    if _hv(_ISg) and _hv(_BSg) and _hv(_CFg):
        _ISr=_norm27(_ISg); _BSr=_norm27(_BSg); _CFr=_norm27(_CFg)
        print("   IS/BS/CF data: from globals")
    else:
        try:
            import yfinance as _yf27
            _t27=_yf27.Ticker(TICKER)
            _ISr=_norm27(_t27.financials); _BSr=_norm27(_t27.balance_sheet); _CFr=_norm27(_t27.cashflow)
            if _ISr.empty: raise ValueError("empty")
            print("   IS/BS/CF data: yfinance")
        except Exception as _yfe:
            # No hardcoded fallback — use empty frames
            # The 3-stmt writer will skip gracefully
            _ISr = pd.DataFrame()
            _BSr = pd.DataFrame()
            _CFr = pd.DataFrame()
            print(f"   ⚠️  yfinance unavailable ({_yfe}) — IS/BS/CF empty")
    print(f"✅ Income Statement + Balance Sheet + Cash Flow written")
except Exception as _e27:
    import traceback as _tb27
    print(f"⚠️  IS/BS/CF write error: {_e27}")
    _tb27.print_exc()


✅ FY headers → Past:[2023, 2024, 2025]  Fore:[2026, 2027, 2028, 2029, 2030]
   Shares: 13,532.47M
   C9 Debt  = 3,695,750 Millions INR
   Cash     = 1,006,450 Millions INR
   Equity   = 8,432,000 Millions INR
✅ DCF sheet written with Excel formulas
   Debt=3695750.0M  Cash=1006450.0M  NetDebt=4353560.0M
   Shares=13532.472634M  Price=1343.3
   IS/BS/CF data: from globals
✅ Income Statement + Balance Sheet + Cash Flow written


/var/folders/5k/x_y2jtd54tx16f_my1zkkhk80000gn/T/ipykernel_40594/2165361418.py:324: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df=df.copy(); df.columns=_pd27.to_datetime(df.columns,errors="coerce")
/var/folders/5k/x_y2jtd54tx16f_my1zkkhk80000gn/T/ipykernel_40594/2165361418.py:324: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df=df.copy(); df.columns=_pd27.to_datetime(df.columns,errors="coerce")
/var/folders/5k/x_y2jtd54tx16f_my1zkkhk80000gn/T/ipykernel_40594/2165361418.py:324: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df=df.copy(); df.columns=_pd27.to_datetime(df.co

In [16]:
# ── 5J: Cost of Capital sheet ────────────────────────────────────────────────

wb = load_workbook(TEMPLATE_XLSM, keep_vba=True)
COC_SHEET = _resolve_sheet(wb, "Cost of Capital")
ws_coc    = wb[COC_SHEET]

# Compute tax rate from IS
_tax_rate_final = TAX_RATE / 100.0
if SOURCES.get("api") and not IS.empty:
    try:
        _IS_latest = IS.sort_values("FY").iloc[-1]
        def _is_get(cols):
            for c in cols:
                if c in _IS_latest.index and pd.notna(_IS_latest[c]): return float(_IS_latest[c])
            return None
        _pretax = _is_get(["Pretax Income","Income Before Tax"])
        _taxexp = _is_get(["Income Tax Expense","Tax Provision"])
        if _pretax and abs(_pretax) > 0 and _taxexp is not None:
            _tr = abs(_taxexp) / abs(_pretax)
            if 0.01 < _tr < 0.60:
                _tax_rate_final = _tr
                print(f"   Tax rate from IS: {_tr:.2%}")
    except Exception: pass

# Interest rate
_interest_rate = COST_OF_DEBT_PRETAX / 100.0
if SOURCES.get("api") and not IS.empty and not BS.empty:
    try:
        _IS_l = IS.sort_values("FY").iloc[-1]
        _BS_l = BS.sort_values("FY").iloc[-1]
        def _g(df_row, cols):
            for c in cols:
                if c in df_row.index and pd.notna(df_row[c]): return float(df_row[c])
            return None
        _ie  = _g(_IS_l, ["Interest Expense","InterestExpense"])
        _ltd = _g(_BS_l, ["Long Term Debt"])
        _std = _g(_BS_l, ["Short Long Term Debt","Short Term Debt"])
        _td  = (_ltd or 0) + (_std or 0)
        if _ie and _td and _td > 0:
            _ir = abs(_ie) / abs(_td)
            if 0.001 < _ir < 0.30:
                _interest_rate = _ir
    except Exception: pass

# Write input values
ws_coc["C3"] = round(_tax_rate_final, 4);      ws_coc["C3"].number_format = "0.00%"
ws_coc["C4"] = round(BETA, 4);                 ws_coc["C4"].number_format = "0.00"
ws_coc["C5"] = round(RISK_FREE / 100.0, 4);    ws_coc["C5"].number_format = "0.00%"
ws_coc["C6"] = round(EQUITY_RISK_PREMIUM / 100.0, 4); ws_coc["C6"].number_format = "0.00%"
# Re-read debt from HIST if still 0 (happens when CoC runs before HIST is fully built)
if not _debt or _debt == 0:
    try:
        _h2 = globals().get("HIST")
        if _h2 is not None and not _h2.empty:
            _debt = float(_h2.sort_values("FY").iloc[-1].get("DebtBal") or 0)
            _cash = float(_h2.sort_values("FY").iloc[-1].get("CashBal") or 0)
            print(f"   C9 debt (HIST fallback): {_debt:,.2f}")
    except: pass
ws_coc["C9"] = _sanitize(_debt);               ws_coc["C9"].number_format  = "#,##0.0"
# Market cap — try multiple sources in priority order:
# 1. marketCap from yfinance info (most reliable)
# 2. shares (Millions) * price
# 3. fallback to a reasonable estimate
# Unit conversion: HIST shares are in CURRENCY_UNITS, need Millions for mkt_cap
# _to_m21 not needed — HIST values already in display units
_to_m21 = 1.0  # HIST already divided by unit_factor during extraction
# C10 = Total Equity for WACC capital structure weighting.
# Use BOOK EQUITY from HIST (already in display units, no conversion needed).
# Market cap is NOT used here — it causes circular reasoning in WACC and
# creates unit confusion for Indian companies.
_c10_equity = 0.0

# Priority 1: use _equity from capital structure block above (from BS/HIST)
_c10_equity = globals().get("_equity", 0.0) or 0.0

# Priority 2: HIST TotalEquity (already in display units)
if _c10_equity <= 0:
    try:
        _hb = globals().get("HIST")
        if _hb is not None and not _hb.empty:
            _c10_equity = float(_hb.sort_values("FY").iloc[-1].get("TotalEquity") or 0)
    except Exception: pass

# Priority 3: market cap as last resort (for WACC it is theoretically correct
# but only use if book equity is truly unavailable)
_mkt_cap = None
if _c10_equity <= 0 and SOURCES.get("api"):
    try:
        _info_coc = yf.Ticker(TICKER).info or {}
        _raw_mktcap = _info_coc.get("marketCap")
        if _raw_mktcap and _raw_mktcap > 0:
            _uf_mc = globals().get("unit_factor", 1_000_000.0)
            _mkt_cap = float(_raw_mktcap) / _uf_mc
            _c10_equity = _mkt_cap
            print(f"   C10: using market cap {_c10_equity:,.0f} {CURRENCY_UNITS} (book equity unavailable)")
    except Exception: pass

print(f"   C10 Equity = {_c10_equity:,.0f} {CURRENCY_UNITS} {LOCAL_CURRENCY} (book equity)")
ws_coc["C10"] = _sanitize(_c10_equity)
ws_coc["C10"].number_format = "#,##0.0"
ws_coc["C11"] = _sanitize(_pref);              ws_coc["C11"].number_format = "#,##0.00"
ws_coc["E27"] = round(_interest_rate, 4);       ws_coc["E27"].number_format = "0.00%"

# KEY: write formulas for ratios and cost of capital — not hardcoded values
ws_coc["C12"] = "=C9+C10+C11"                  # Total Capital
ws_coc["C15"] = "=C9/C12"                       # Debt to Total Capital
ws_coc["C16"] = "=C10/C12"                      # Equity to Total Capital
ws_coc["C17"] = "=C11/C12"   # FIX: Preferred/Total Capital (not Debt+Equity)                      # Pref to Total Capital
ws_coc["D20"] = "=(E27*C15)*(1-C3)"            # Cost of Debt after tax
ws_coc["D21"] = "=(C5+(C4*C6))*C16"            # Cost of Equity
ws_coc["D22"] = "=D20+D21"                      # WACC
ws_coc["C22"] = "=D22"                          # FIX: C22 mirrors live WACC

safe_save(wb, TEMPLATE_XLSM)
print(f"✅ Cost of Capital | WACC formula wired | Tax={_tax_rate_final:.2%} | Rf={RISK_FREE}% | ERP={EQUITY_RISK_PREMIUM}%")


   Tax rate from IS: 23.80%
   C10 Equity = 8,432,000 Millions INR (book equity)
✅ Cost of Capital | WACC formula wired | Tax=23.80% | Rf=4.0% | ERP=5.0%


In [17]:
# ── 5K: WC% update (R16 already set in 5C cell above)
# This cell updates R16 with a more precise value from actual BS/IS data
_wc_pct_actual = None
if SOURCES.get("api") and not BS.empty and not IS.empty:
    try:
        _bs_s = BS.sort_values("FY"); _is_s = IS.sort_values("FY")
        if len(_bs_s) >= 2 and len(_is_s) >= 2:
            def _nwc(r):
                def g(c): return float(r[c]) if c in r.index and pd.notna(r[c]) else None
                ca=g("Current Assets"); cl=g("Current Liabilities")
                cs=g("Cash And Cash Equivalents") or 0; std=g("Short Long Term Debt") or 0
                return (ca-(cs))-(cl-(std)) if ca and cl else None
            def _rev(r):
                for c in ["Total Revenue","Revenue"]:
                    if c in r.index and pd.notna(r[c]): return float(r[c])
            n1=_nwc(_bs_s.iloc[-1]); n0=_nwc(_bs_s.iloc[-2])
            r1=_rev(_is_s.iloc[-1]); r0=_rev(_is_s.iloc[-2])
            if all(x is not None for x in [n1,n0,r1,r0]) and (r1-r0)!=0:
                _wc_pct_actual = max(-0.50, min(0.50, (n1-n0)/(r1-r0)))
    except Exception: pass
if _wc_pct_actual is not None:
    wb = load_workbook(TEMPLATE_XLSM, keep_vba=True)
    ws = wb[_resolve_sheet(wb, "DCF")]
    ws["R16"] = round(_wc_pct_actual, 4); ws["R16"].number_format = "0.00%"
    safe_save(wb, TEMPLATE_XLSM)
    print(f"✅ WC% updated: R16 = {_wc_pct_actual:.4f}")
else:
    print("ℹ️  WC% not updated — using driver default in R16.")


✅ WC% updated: R16 = -0.5000


# Part 6 — Sensitivity Table
Pure Python values written directly — no Excel DataTable formula.

In [18]:
# ── Part 6: Sensitivity Table — WACC vs Terminal Growth ──────────────────────
# Architecture:
#   - Grid E44:I48 (5 WACC rows × 5 growth cols)
#   - Center cell G46 = base case (same as P30 implied share price)
#   - Each cell is a LIVE Excel formula — recalculates when any assumption changes
#   - Formula embeds w = H36 + offset*R22  and  g = R18 + offset*R21
#   - Uses mid-year convention: PV = FCF/(1+w)^(period-0.5)
#   - Header labels (row 43, col D44:D48) derived from same offset logic
#   - Python also writes values as fallback and for label generation

from openpyxl import load_workbook
from openpyxl.utils.cell import column_index_from_string, get_column_letter, range_boundaries
from openpyxl.cell.cell import MergedCell

wb = load_workbook(TEMPLATE_XLSM, keep_vba=True)
ws = wb[next(s for s in wb.sheetnames if s.strip().lower() == "dcf")]

# ── Unmerge sensitivity area ──────────────────────────────────────────────────
def _unmerge(ws, cell_range):
    min_c, min_r, max_c, max_r = range_boundaries(cell_range)
    to_drop = [str(mr) for mr in list(ws.merged_cells.ranges)
               if not (mr.bounds[2] < min_c or mr.bounds[0] > max_c
                       or mr.bounds[3] < min_r or mr.bounds[1] > max_r)]
    for r in to_drop:
        ws.unmerge_cells(r)

_unmerge(ws, "D43:I48")

def _coerce_num(x):
    try:
        if x is None: return None
        s = str(x).strip()
        if s.endswith("%"): return float(s[:-1]) / 100.0
        return float(s)
    except: return None

def cols_range(start_letter, end_letter):
    s = column_index_from_string(start_letter)
    e = column_index_from_string(end_letter)
    return [get_column_letter(c) for c in range(s, e + 1)]

def safe_set(ws, addr, value):
    cell = ws[addr]
    try: cell.value = value
    except AttributeError: pass

# ── Read base values (for label generation only) ─────────────────────────────
wacc_base = VAL.get("WACC") if isinstance(VAL, dict) else None
if not wacc_base:
    wacc_base = _coerce_num(ws["P8"].value) or _coerce_num(ws["H36"].value)
if not wacc_base or wacc_base <= 0:
    wacc_base = compute_wacc()
if wacc_base > 1: wacc_base /= 100.0

g_base   = _coerce_num(ws["R18"].value) or (TERMINAL_GROWTH / 100.0)
if g_base   > 1: g_base   /= 100.0
wacc_inc = _coerce_num(ws["R22"].value) or 0.0025
if wacc_inc > 1: wacc_inc /= 100.0
g_inc    = _coerce_num(ws["R21"].value) or 0.0025
if g_inc    > 1: g_inc    /= 100.0

print(f"   WACC base={wacc_base*100:.2f}%  g base={g_base*100:.2f}%")
print(f"   WACC inc={wacc_inc*100:.2f}%   g inc={g_inc*100:.2f}%")

# ── Build Excel formula for one grid cell ────────────────────────────────────
# w = $H$36 + (wacc_offset)*$R$22   (wacc_offset = row_i - 2, range -2..+2)
# g = $R$18 + (g_offset)*$R$21      (g_offset = col_j - 2, range -2..+2)
#
# Share Price =
#   ( PV_FCF_H + PV_FCF_I + PV_FCF_J + PV_FCF_K + PV_FCF_L
#   + PV_Terminal_Value
#   - Debt - Pref - MinInt + Cash
#   ) / Diluted_Shares
#
# PV_FCFx = $Xx$34 / (1+w)^($Xx$35-0.5)     (mid-year convention)
# TV       = $L$34*(1+g)/(w-g)
# PV_TV    = TV / (1+w)^$L$35               (end-of-year on terminal period)
#
# All anchored with $ so formula is portable / auditable in Excel

def _wacc_expr(row_offset):
    """Excel expression for WACC at given row offset from center."""
    if row_offset == 0:
        return "$H$36"
    elif row_offset > 0:
        return f"$H$36+{row_offset}*$R$22"
    else:
        return f"$H$36{row_offset}*$R$22"   # e.g. $H$36-2*$R$22

def _g_expr(col_offset):
    """Excel expression for g at given col offset from center."""
    if col_offset == 0:
        return "$R$18"
    elif col_offset > 0:
        return f"$R$18+{col_offset}*$R$21"
    else:
        return f"$R$18{col_offset}*$R$21"

def _sens_formula(row_offset, col_offset):
    """
    Full Excel formula for implied share price at given WACC/g offset.
    row_offset: -2,-1,0,+1,+2 (WACC)
    col_offset: -2,-1,0,+1,+2 (g)
    """
    w = _wacc_expr(row_offset)
    g = _g_expr(col_offset)
    
    # PV of 5 projection years — end-of-year convention (matches H37:L37 = HX34/(1+HX36)^HX35)
    pv_proj_terms = "+".join(
        f"${c}$34/(1+{w})^${c}$35"
        for c in ["H","I","J","K","L"]
    )
    
    # Terminal Value — parentheses wrap FULL g expression to prevent sign flip
    # Without: (w - R18 + offset*R21) — wrong, offset applies to w not g
    # With:    (w - (R18 + offset*R21)) — correct, offset applies to g
    tv_term  = f"($L$34*(1+{g})/({w}-({g})))"
    pv_tv    = f"{tv_term}/(1+{w})^$L$35"
    
    # Equity bridge
    bridge   = "-$P$24-$P$25-$P$26+$P$27"
    
    # Full formula
    formula = f"=IF(({w})<=({g}),\"n/a\",({pv_proj_terms}+{pv_tv}{bridge})/$R$25)"
    return formula

# ── Write header labels ───────────────────────────────────────────────────────
# D43: corner cell — blank
safe_set(ws, "D43", "")

# E43:I43 — Growth rate labels as live Excel formulas tied to R18 ± n*R21
col_offsets  = [-2, -1, 0, 1, 2]
_g_lbls = ["=$R$18-2*$R$21","=$R$18-1*$R$21","=$R$18","=$R$18+1*$R$21","=$R$18+2*$R$21"]
for j, col in enumerate(cols_range("E", "I")):
    safe_set(ws, f"{col}43", _g_lbls[j])
    try: ws[f"{col}43"].number_format = "0.00%"
    except AttributeError: pass
col_labels = [f"{(g_base + o * g_inc)*100:.2f}%" for o in col_offsets]  # for print only

# D44:D48 — WACC labels as live Excel formulas tied to H36 ± n*R22
row_offsets = [-2, -1, 0, 1, 2]
_w_lbls = ["=$H$36-2*$R$22","=$H$36-1*$R$22","=$H$36","=$H$36+1*$R$22","=$H$36+2*$R$22"]
for i in range(5):
    safe_set(ws, f"D{44+i}", _w_lbls[i])
    try: ws[f"D{44+i}"].number_format = "0.00%"
    except AttributeError: pass
row_labels = [f"{(wacc_base + o * wacc_inc)*100:.2f}%" for o in row_offsets]  # for print only

# ── Write LIVE Excel formulas into E44:I48 ────────────────────────────────────
print("\n   Writing live Excel formulas to E44:I48...")
_written = 0
for i, row_offset in enumerate(row_offsets):        # rows 44-48
    for j, col_offset in enumerate(col_offsets):    # cols E-I
        col  = cols_range("E", "I")[j]
        addr = f"{col}{44 + i}"
        formula = _sens_formula(row_offset, col_offset)
        safe_set(ws, addr, formula)
        _written += 1

# Also write number format for all grid cells
from openpyxl.utils.cell import range_boundaries as _rb
min_c, min_r, max_c, max_r = _rb("E44:I48")
for r in range(min_r, max_r + 1):
    for c in range(min_c, max_c + 1):
        cell = ws.cell(r, c)
        try: cell.number_format = "#,##0.00"
        except AttributeError: pass

# Clear J43:O48 (stale data from old runs)
for r in range(43, 49):
    for col in cols_range("J", "O"):
        safe_set(ws, f"{col}{r}", None)

safe_save(wb, TEMPLATE_XLSM)

print(f"   ✅ {_written}/25 live Excel formulas written to E44:I48")
print(f"   WACC range: {row_labels[0]} → {row_labels[-1]}")
print(f"   Growth range: {col_labels[0]} → {col_labels[-1]}")
print(f"   Center cell G46 = base case (= P30 when WACC/g are unchanged)")
print(f"   Formula structure: =IF(w<=g,\"n/a\",(PV_FCF+PV_TV-Debt+Cash)/Shares)")
print(f"   All formulas reference live cells: H34:L34, H35:L35, H36, R18, R21, R22, P24-P27, R25")
print(f"\n   Sample formula G46 (base case):")
print(f"   {_sens_formula(0, 0)}")


   WACC base=8.39%  g base=2.00%
   WACC inc=0.25%   g inc=0.25%

   Writing live Excel formulas to E44:I48...
   ✅ 25/25 live Excel formulas written to E44:I48
   WACC range: 7.89% → 8.89%
   Growth range: 1.50% → 2.50%
   Center cell G46 = base case (= P30 when WACC/g are unchanged)
   Formula structure: =IF(w<=g,"n/a",(PV_FCF+PV_TV-Debt+Cash)/Shares)
   All formulas reference live cells: H34:L34, H35:L35, H36, R18, R21, R22, P24-P27, R25

   Sample formula G46 (base case):
   =IF(($H$36)<=($R$18),"n/a",($H$34/(1+$H$36)^$H$35+$I$34/(1+$H$36)^$I$35+$J$34/(1+$H$36)^$J$35+$K$34/(1+$H$36)^$K$35+$L$34/(1+$H$36)^$L$35+($L$34*(1+$R$18)/($H$36-($R$18)))/(1+$H$36)^$L$35-$P$24-$P$25-$P$26+$P$27)/$R$25)


# Part 7 — Financial Statement Sheets (IS / BS / CF)
Clean labeled rows written from the same source data.

In [19]:
# ── 3-Statement Sheet Writer Setup ──────────────────────────────────────────
# PURPOSE: This cell ensures the notebook has run up to the data-fetch step.
# HOW TO USE:
#   Option A (recommended): Kernel → Restart & Run All Cells
#   Option B: Run cells 1 through 27 in order
#
# If IS/BS/CF are not in globals, the next cell (Phase 3) will fetch
# them automatically via yfinance — so just make sure TICKER is set.

_ticker_ok = bool(globals().get("TICKER", "").strip())
_data_ok   = (globals().get("IS") is not None and
              globals().get("BS") is not None and
              globals().get("CF") is not None)
_file_ok   = bool(globals().get("OUTPUT_XLSM") or globals().get("TEMPLATE_XLSM"))

print(f"TICKER set:     {'✅ ' + globals().get('TICKER','') if _ticker_ok else '❌ Not set — run from cell 1'}")
print(f"Data loaded:    {'✅ IS/BS/CF in globals' if _data_ok else '⚠️  Will fetch in next cell'}")
print(f"Output file:    {'✅ ' + str(globals().get('OUTPUT_XLSM','')) if _file_ok else '❌ Not set — run from cell 1'}")

if not _ticker_ok or not _file_ok:
    print()
    print("⚠️  TICKER or output file not set.")
    print("   → Please run: Kernel → Restart & Run All Cells")
else:
    print()
    print("✅ Ready — run the next cell to write Income Statement, Balance Sheet, Cash Flow")


TICKER set:     ✅ RELIANCE.NS
Data loaded:    ✅ IS/BS/CF in globals
Output file:    ✅ /Users/sharankothari/Desktop/research_platform/DCF_Output_RELIANCE.NS_INR.xlsm

✅ Ready — run the next cell to write Income Statement, Balance Sheet, Cash Flow


In [20]:
# =============================================================================
# PHASE 3 — Analyst-Grade 3-Statement Model (IS / BS / CF)
# ─────────────────────────────────────────────────────────────────────────────
# Design principles:
#   1. All values written as HARDCODED numbers (no openpyxl formulas in IS/BS/CF)
#   2. Cross-sheet Excel formulas written in a SECOND PASS so every cell is live
#   3. BS Cash ← CF Ending Cash
#   4. CF Net Income / D&A ← IS
#   5. CF ΔAR / ΔInv / ΔAP ← BS curr − prior
#   6. BS Retained Earnings = prior RE + IS Net Income + CF Dividends
#   7. BS Total L+E = Total Liabilities + Total Equity (never blank)
#   8. BS Balance Check = Total Assets − Total L+E  (must = 0)
#   9. CF Ending Cash = Beginning + CFO + CFI + CFF  (formula)
#  10. CFO = SUM of listed items + reconciling "Other" row
# =============================================================================

import math, re
import pandas as pd
# yfinance imported inside try block below (safe if not installed)
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.cell.cell import MergedCell

# Resolve output file — check all variable names set by earlier cells
OUTPUT_XLSX = (globals().get("OUTPUT_XLSM") or
               globals().get("TEMPLATE_XLSM") or
               globals().get("OUTPUT_XLSX") or "")
if not OUTPUT_XLSX:
    import glob as _glob
    _candidates = (
        _glob.glob(str(globals().get("OUTPUT_DIR", ".")) + "/*.xlsm") +
        _glob.glob("/mnt/user-data/outputs/*.xlsm") +
        _glob.glob("/mnt/user-data/uploads/*.xlsm")
    )
    _dcf = [f for f in _candidates if "DCF_Output" in f]
    if _dcf: OUTPUT_XLSX = sorted(_dcf)[-1]
    elif _candidates: OUTPUT_XLSX = sorted(_candidates)[-1]
TICKER = globals().get("TICKER", "") or globals().get("_TICKER", "")
if not TICKER:
    # Try to infer from output filename (e.g. DCF_Output_MSFT_USD.xlsm → MSFT)
    import re as _re2
    _m = _re2.search(r"DCF_Output_([^_]+)_", str(OUTPUT_XLSX))
    if _m: TICKER = _m.group(1)
if not TICKER: TICKER = "MSFT"  # safe default
_tk           = str(TICKER).upper()
currency_code = "INR" if _tk.endswith((".NS", ".BO")) else "USD"
SYM           = "₹" if currency_code == "INR" else "$"

IS_SHEET = "Income statement"
BS_SHEET = "Balance Sheet"
CF_SHEET = "Cash Flow"

# ── Styles ────────────────────────────────────────────────────────────────────
_HD, _SM, _SB, _MT = "1F3864", "2F5496", "D6E4F7", "F5F5F5"

def _f(bold=False, italic=False, color="000000", sz=10):
    return Font(name="Calibri", size=sz, bold=bold, italic=italic, color=color)
def _fl(h): return PatternFill("solid", fgColor=h)
def _al(h="left", v="center"): return Alignment(horizontal=h, vertical=v)
def _bd(): return Border(bottom=Side(style="thin"))

# ── Data helpers ──────────────────────────────────────────────────────────────
def _g(df, *keys):
    if df is None or df.empty: return None
    for k in keys:
        hits = [i for i in df.index if k.lower() in str(i).lower()]
        if hits: return df.loc[hits[0]]
    return None

def _v(s, j, scale=1_000_000):
    if s is None: return None
    try:
        v = s.values[j]
        if v is None or (isinstance(v, float) and math.isnan(v)): return None
        return float(v) / scale
    except: return None

def _dv(a, b):
    if a is None or b is None or b == 0: return None
    try: return a / b
    except: return None

# ── Sheet writer — returns (ws, row_addr_map) ─────────────────────────────────
def _clear(wb, title):
    t = title[:31]
    if t in wb.sheetnames: del wb[t]
    return wb.create_sheet(t)

def write_sheet(wb, title, sections, col_hdrs, sym="$"):
    """
    Write analyst sheet. Returns (ws, addr_map) where addr_map maps
    section['key'] → list of Excel address strings like "'Sheet'!C5"
    """
    ws  = _clear(wb, title)
    n   = len(col_hdrs)
    D   = 2   # data starts col B
    fmt_ccy = f'"{sym}"#,##0'
    addr_map = {}

    # Title row
    ws.row_dimensions[1].height = 22
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=D+n)
    tc = ws.cell(1, 1)
    tc.value = f"{title}  |  {currency_code}  |  In Millions"
    tc.font = _f(bold=True, color="FFFFFF", sz=12)
    tc.fill = _fl(_HD); tc.alignment = _al("left")

    # Col headers row 2
    ws.row_dimensions[2].height = 18
    ws.cell(2,1).value="Line Item"; ws.cell(2,1).font=_f(bold=True,color="FFFFFF")
    ws.cell(2,1).fill=_fl(_HD); ws.cell(2,1).alignment=_al("left")
    for j, h in enumerate(col_hdrs):
        c = ws.cell(2, D+j)
        c.value=h; c.font=_f(bold=True,color="FFFFFF")
        c.fill=_fl(_HD); c.alignment=_al("right")

    row = 3
    for sec in sections:
        t   = sec.get("type", "item")
        lbl = sec.get("label", "")
        vals= sec.get("values")
        fmt = sec.get("fmt", "currency")
        ind = sec.get("indent", 0)
        key = sec.get("key")

        ws.row_dimensions[row].height = 15

        if t == "spacer": row += 1; continue

        lc = ws.cell(row, 1)
        lc.value = ("  "*ind) + lbl
        lc.alignment = _al("left")

        if t == "header":
            lc.font = _f(bold=True, color="FFFFFF"); lc.fill = _fl(_SM)
            ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=D+n)
            row += 1; continue

        lc.font = (_f(bold=True) if t=="subtotal" else
                   _f(italic=True, color="555555") if t=="metric" else _f())
        if t == "subtotal": lc.fill = _fl(_SB); lc.border = _bd()
        elif t == "metric": lc.fill = _fl(_MT)

        col_addrs = []
        if vals is not None:
            for j, v in enumerate(vals):
                vc = ws.cell(row, D+j)
                if isinstance(v, str) and v.startswith("="):
                    vc.value = v   # pre-built formula
                elif v is None or (isinstance(v, float) and math.isnan(v)):
                    vc.value = None
                else:
                    try: vc.value = round(float(v), 2)
                    except: vc.value = v

                vc.alignment = _al("right")
                vc.font = (_f(bold=True) if t=="subtotal" else
                           _f(italic=True, color="555555") if t=="metric" else
                           _f(color="1F4E79"))
                if t == "subtotal": vc.fill=_fl(_SB); vc.border=_bd()
                elif t == "metric": vc.fill=_fl(_MT)

                if   fmt == "currency": vc.number_format = fmt_ccy
                elif fmt == "percent":  vc.number_format = "0.0%"
                elif fmt == "decimal":  vc.number_format = "0.00"
                elif fmt == "integer":  vc.number_format = "#,##0"
                else:                   vc.number_format = fmt_ccy

                col_addrs.append(f"'{title}'!{get_column_letter(D+j)}{row}")

            if key:
                addr_map[key] = col_addrs

        row += 1

    ws.column_dimensions["A"].width = 42
    for j in range(n):
        ws.column_dimensions[get_column_letter(D+j)].width = 14
    ws.freeze_panes = "B3"
    return ws, addr_map


# ── Section builders ──────────────────────────────────────────────────────────
def build_is(IS, BS, CF, n):
    def R(l,s,key=None,ind=1,fmt="currency"):
        return {"type":"item","label":l,"key":key,
                "values":[_v(s,j) for j in range(n)],"fmt":fmt,"indent":ind}
    def S(l,v,key=None): return {"type":"subtotal","label":l,"key":key,"values":v,"fmt":"currency","indent":0}
    def M(l,v,fmt="percent"): return {"type":"metric","label":l,"values":v,"fmt":fmt,"indent":1}
    def H(l): return {"type":"header","label":l}
    def SP(): return {"type":"spacer"}

    rev   = _g(IS,"Total Revenue","Revenue")
    cogs  = _g(IS,"Cost Of Revenue","Cost of Goods")
    gp    = _g(IS,"Gross Profit")
    sga   = _g(IS,"Selling General Administrative","Selling General And Administration",
                  "General And Administrative Expense","Selling And Marketing Expense")
    rd    = _g(IS,"Research Development","Research And Development")
    opex  = _g(IS,"Operating Expense","Total Operating Expenses")
    da    = _g(IS,"Reconciled Depreciation","Depreciation Amortization Depletion Income Statement",
                  "Depreciation And Amortization")
    ebit  = _g(IS,"Operating Income","Ebit")
    int_e = _g(IS,"Interest Expense")
    int_i = _g(IS,"Interest Income")
    pre   = _g(IS,"Pretax Income","Income Before Tax")
    tax   = _g(IS,"Tax Provision","Income Tax")
    ni    = _g(IS,"Net Income")
    shr   = _g(IS,"Diluted Average Shares","Basic Average Shares","Ordinary Shares Number")

    rev_v  = [_v(rev,j) for j in range(n)]
    gp_v   = [_v(gp,j) or ((_v(rev,j) or 0)-abs(_v(cogs,j) or 0)) for j in range(n)]
    da_v   = [_v(da,j) for j in range(n)]
    ebit_v = [_v(ebit,j) for j in range(n)]
    ni_v   = [_v(ni,j) for j in range(n)]
    ebitda_v = [(_v(ebit,j) or 0)+(_v(da,j) or 0) for j in range(n)]
    pre_v  = [_v(pre,j) for j in range(n)]
    tax_v  = [_v(tax,j) for j in range(n)]
    eps_v  = [_dv(ni_v[j], _v(shr,j)) for j in range(n)]

    return [
        H("REVENUE"),
        R("  Gross Revenue",rev,"is_revenue"),
        R("  Cost of Revenue",cogs),
        S("Gross Profit",gp_v,"is_gross_profit"),
        M("  Gross Margin %",[_dv(gp_v[j],rev_v[j]) for j in range(n)]),SP(),
        H("OPERATING EXPENSES"),
        R("  Selling, General & Administrative",sga),
        R("  Research & Development",rd),
        R("  Other Operating Expenses",opex),
        S("EBITDA",ebitda_v,"is_ebitda"),
        M("  EBITDA Margin %",[_dv(ebitda_v[j],rev_v[j]) for j in range(n)]),SP(),
        H("DEPRECIATION & AMORTIZATION"),
        R("  Total D&A",da,"is_da"),
        S("EBIT (Operating Income)",ebit_v,"is_ebit"),
        M("  EBIT Margin %",[_dv(ebit_v[j],rev_v[j]) for j in range(n)]),SP(),
        H("NON-OPERATING & TAX"),
        R("  Interest Income",int_i),
        R("  Interest Expense",int_e),
        {"type":"item","label":"  EBT (Pre-Tax Income)","key":"is_pretax",
         "values":pre_v,"fmt":"currency","indent":1},
        R("  Income Tax Expense",tax),
        M("  Effective Tax Rate %",[_dv(abs(_v(tax,j) or 0),pre_v[j]) for j in range(n)]),
        S("NET INCOME",ni_v,"is_net_income"),
        M("  Net Margin %",[_dv(ni_v[j],rev_v[j]) for j in range(n)]),SP(),
        H("PER SHARE"),
        M("  Diluted Shares (M)",[_v(shr,j) for j in range(n)],"integer"),
        {"type":"metric","label":"  EPS","key":"is_eps","values":eps_v,"fmt":"decimal","indent":1},
    ]


def build_bs(IS, BS, CF, n):
    def R(l,s,key=None,ind=1):
        return {"type":"item","label":l,"key":key,
                "values":[_v(s,j) for j in range(n)],"fmt":"currency","indent":ind}
    def S(l,v,key=None): return {"type":"subtotal","label":l,"key":key,"values":v,"fmt":"currency","indent":0}
    def M(l,v,fmt="decimal"): return {"type":"metric","label":l,"values":v,"fmt":fmt,"indent":1}
    def H(l): return {"type":"header","label":l}
    def SP(): return {"type":"spacer"}

    cash  = _g(BS,"Cash And Cash Equivalents","Cash")
    ar    = _g(BS,"Accounts Receivable","Net Receivables")
    inv   = _g(BS,"Inventory")
    prep  = _g(BS,"Prepaid Assets","Other Current Assets")
    tca   = _g(BS,"Current Assets","Total Current Assets")
    ppe   = _g(BS,"Net Ppe","Property Plant Equipment Net")
    gw    = _g(BS,"Goodwill")
    # Other intangibles = combined minus goodwill to prevent duplication
    _gw_combined = _g(BS,"Goodwill And Other Intangible Assets","Other Intangible Assets","Intangible Assets")
    oi_v  = [(_v(_gw_combined,j) or 0) - (_v(gw,j) or 0) for j in range(n)]
    ta    = _g(BS,"Total Assets")
    ap    = _g(BS,"Accounts Payable")
    std   = _g(BS,"Current Debt","Short Term Debt")
    tcl   = _g(BS,"Current Liabilities","Total Current Liabilities")
    ltd   = _g(BS,"Long Term Debt")
    tl    = _g(BS,"Total Liabilities Net Minority Interest","Total Liabilities")
    re    = _g(BS,"Retained Earnings")
    equity= _g(BS,"Stockholders Equity","Total Equity Gross Minority Interest")

    rev_s = _g(IS,"Total Revenue","Revenue")
    cogs_s= _g(IS,"Cost Of Revenue","Cost of Goods")

    tca_v = [_v(tca,j) for j in range(n)]
    ta_v  = [_v(ta,j) for j in range(n)]
    tcl_v = [_v(tcl,j) for j in range(n)]
    tl_v  = [_v(tl,j) for j in range(n)]
    eq_v  = [_v(equity,j) for j in range(n)]
    cash_v= [_v(cash,j) for j in range(n)]
    ar_v  = [_v(ar,j) for j in range(n)]
    inv_v = [_v(inv,j) for j in range(n)]
    ap_v  = [_v(ap,j) for j in range(n)]

    # Other CL = TCL - AP - STD  (prevents double-counting)
    other_cl_v = [max(0, (_v(tcl,j) or 0) - abs(_v(ap,j) or 0) - abs(_v(std,j) or 0))
                  for j in range(n)]

    # Total L+E always computed from components
    tleq_v = [(_v(tl,j) or 0) + (_v(equity,j) or 0) for j in range(n)]

    # Balance check = Total Assets - Total L+E  (must be 0)
    bal_v  = [round((_v(ta,j) or 0) - tleq_v[j], 2) for j in range(n)]

    # Non-current assets (total - current)
    nca_v  = [(_v(ta,j) or 0) - (_v(tca,j) or 0) for j in range(n)]
    ncl_v  = [(_v(tl,j) or 0) - (_v(tcl,j) or 0) for j in range(n)]
    nwc_v  = [(_v(tca,j) or 0) - (_v(tcl,j) or 0) for j in range(n)]

    # Other current assets (unlisted items like S-T investments)
    other_ca_v = [(_v(tca,j) or 0) - (_v(cash,j) or 0) - (_v(ar,j) or 0)
                  - (_v(inv,j) or 0) - (_v(prep,j) or 0) for j in range(n)]

    dso  = [_dv(_v(ar,j), _v(rev_s,j)) * 365 if _v(ar,j) and _v(rev_s,j) else None for j in range(n)]
    dio  = [_dv(_v(inv,j), abs(_v(cogs_s,j))) * 365 if _v(inv,j) and _v(cogs_s,j) else None for j in range(n)]
    dpo  = [_dv(_v(ap,j), abs(_v(cogs_s,j))) * 365 if _v(ap,j) and _v(cogs_s,j) else None for j in range(n)]
    ccc  = [(dso[j] or 0) + (dio[j] or 0) - (dpo[j] or 0) if dso[j] else None for j in range(n)]
    de   = [_dv(_v(ltd,j), eq_v[j]) for j in range(n)]

    return [
        H("CURRENT ASSETS"),
        {"type":"item","label":"    Cash & Cash Equivalents","key":"bs_cash",
         "values":cash_v,"fmt":"currency","indent":1},
        R("    Accounts Receivable",ar,"bs_ar"),
        R("    Inventory",inv,"bs_inv"),
        R("    Prepaid Expenses",prep),
        {"type":"item","label":"    Other Current Assets (S-T investments etc.)",
         "values":other_ca_v,"fmt":"currency","indent":1},
        S("Total Current Assets",tca_v,"bs_tca"),SP(),
        H("NON-CURRENT ASSETS"),
        R("    Net PP&E",ppe,"bs_ppe"),
        R("    Goodwill",gw,"bs_goodwill"),
        {"type":"item","label":"    Other Intangible Assets","key":"bs_intangibles",
         "values":oi_v,"fmt":"currency","indent":1},
        S("Total Non-Current Assets",nca_v),
        S("TOTAL ASSETS",ta_v,"bs_total_assets"),SP(),
        H("CURRENT LIABILITIES"),
        R("    Accounts Payable",ap,"bs_ap"),
        R("    Short-Term Debt",std),
        {"type":"item","label":"    Other Current Liabilities",
         "values":other_cl_v,"fmt":"currency","indent":1},
        S("Total Current Liabilities",
          [(_v(ap,j) or 0)+(_v(std,j) or 0)+other_cl_v[j] for j in range(n)],
          "bs_tcl"),SP(),
        H("NON-CURRENT LIABILITIES"),
        R("    Long-Term Debt",ltd,"bs_ltd"),
        S("Total Non-Current Liabilities",ncl_v),
        S("TOTAL LIABILITIES",tl_v,"bs_total_liabilities"),SP(),
        H("SHAREHOLDERS' EQUITY"),
        {"type":"item","label":"    Retained Earnings","key":"bs_retained_earnings",
         "values":[_v(re,j) for j in range(n)],"fmt":"currency","indent":1},
        R("    Total Equity",equity,"bs_equity"),
        S("TOTAL LIABILITIES + EQUITY",tleq_v,"bs_total_le"),
        {"type":"metric","label":"  ✓ Balance Check (must = 0)",
         "key":"bs_balance_check","values":bal_v,"fmt":"currency","indent":1},SP(),
        H("WORKING CAPITAL METRICS"),
        M("  Net Working Capital",nwc_v,"currency"),
        M("  DSO (days)",dso), M("  DIO (days)",dio),
        M("  DPO (days)",dpo), M("  Cash Conversion Cycle",ccc),
        M("  Debt / Equity",de),
    ]


def build_cf(IS, BS, CF, n, is_map=None, bs_map=None):
    def R(l,s,key=None,ind=1):
        return {"type":"item","label":l,"key":key,
                "values":[_v(s,j) for j in range(n)],"fmt":"currency","indent":ind}
    def S(l,v,key=None): return {"type":"subtotal","label":l,"key":key,"values":v,"fmt":"currency","indent":0}
    def M(l,v,fmt="percent"): return {"type":"metric","label":l,"values":v,"fmt":fmt,"indent":1}
    def H(l): return {"type":"header","label":l}
    def SP(): return {"type":"spacer"}

    ni    = _g(IS,"Net Income")
    da    = _g(IS,"Reconciled Depreciation","Depreciation Amortization Depletion Income Statement","Depreciation And Amortization")
    cfo   = _g(CF,"Operating Cash Flow","Cash From Operations","Net Cash From Operating Activities")
    capex = _g(CF,"Capital Expenditure","Purchase Of Ppe","Capital Expenditures")
    cfi   = _g(CF,"Investing Cash Flow","Net Cash From Investing Activities")
    dbt_i = _g(CF,"Issuance Of Debt","Proceeds From Long Term Debt Issuance","Debt Issuance")
    dbt_r = _g(CF,"Repayment Of Debt","Long Term Debt Payments")
    div   = _g(CF,"Payment Of Dividends","Common Stock Dividend Paid","Dividends Paid")
    bb    = _g(CF,"Repurchase Of Capital Stock","Common Stock Repurchased","Share Buyback")
    cff   = _g(CF,"Financing Cash Flow","Net Cash From Financing Activities")
    beg   = _g(CF,"Beginning Cash Position","Cash And Cash Equivalents Beginning Period")
    end_c = _g(CF,"End Cash Position","Cash And Cash Equivalents End Period","Changes In Cash")
    chg_ar  = _g(CF,"Change In Receivables","Changes In Accounts Receivable")
    chg_inv = _g(CF,"Change In Inventory")
    chg_ap  = _g(CF,"Change In Payable","Changes In Accounts Payable")
    chg_wc  = _g(CF,"Change In Working Capital")
    rev     = _g(IS,"Total Revenue","Revenue")

    cfo_v   = [_v(cfo,j) for j in range(n)]
    capex_v = [_v(capex,j) for j in range(n)]
    cfi_v   = [_v(cfi,j) for j in range(n)]
    cff_v   = [_v(cff,j) for j in range(n)]
    ni_v    = [_v(ni,j) for j in range(n)]
    da_v    = [_v(da,j) for j in range(n)]
    end_v   = [_v(end_c,j) for j in range(n)]
    beg_v   = [_v(beg,j) for j in range(n)]
    fcf_v   = [(cfo_v[j] or 0)+(capex_v[j] or 0) for j in range(n)]

    # ΔAR, ΔInv, ΔAP from BS when available
    ar_s  = _g(BS,"Accounts Receivable","Net Receivables")
    inv_s = _g(BS,"Inventory")
    ap_s  = _g(BS,"Accounts Payable")
    ar_d  = [-(_v(ar_s,j)-_v(ar_s,j-1)) if j>0 and _v(ar_s,j) and _v(ar_s,j-1) else _v(chg_ar,j) for j in range(n)]
    inv_d = [-(_v(inv_s,j)-_v(inv_s,j-1)) if j>0 and _v(inv_s,j) and _v(inv_s,j-1) else _v(chg_inv,j) for j in range(n)]
    ap_d  = [(_v(ap_s,j)-_v(ap_s,j-1)) if j>0 and _v(ap_s,j) and _v(ap_s,j-1) else _v(chg_ap,j) for j in range(n)]

    # Other operating items = CFO - listed items (reconciling row)
    other_ops_v = [(cfo_v[j] or 0)-(ni_v[j] or 0)-(da_v[j] or 0)
                   -(ar_d[j] or 0)-(inv_d[j] or 0)-(ap_d[j] or 0) for j in range(n)]

    return [
        H("OPERATING ACTIVITIES"),
        {"type":"item","label":"    Net Income","key":"cf_net_income",
         "values":ni_v,"fmt":"currency","indent":1},
        {"type":"item","label":"    Add: D&A","key":"cf_da",
         "values":da_v,"fmt":"currency","indent":1},
        {"type":"item","label":"    ΔAccounts Receivable","key":"cf_d_ar",
         "values":ar_d,"fmt":"currency","indent":1},
        {"type":"item","label":"    ΔInventory","key":"cf_d_inv",
         "values":inv_d,"fmt":"currency","indent":1},
        {"type":"item","label":"    ΔAccounts Payable","key":"cf_d_ap",
         "values":ap_d,"fmt":"currency","indent":1},
        {"type":"item","label":"    Other Working Capital Changes",
         "values":[_v(chg_wc,j) for j in range(n)],"fmt":"currency","indent":1},
        {"type":"item","label":"    Other Operating Items (SBC, Deferred Tax, etc.)",
         "values":other_ops_v,"fmt":"currency","indent":1},
        S("Cash from Operations (CFO)",cfo_v,"cf_cfo"),
        M("  CFO Margin %",[_dv(cfo_v[j],_v(rev,j)) for j in range(n)]),SP(),
        H("INVESTING ACTIVITIES"),
        R("    Capital Expenditure (CapEx)",capex,"cf_capex"),
        S("Cash from Investing (CFI)",cfi_v,"cf_cfi"),SP(),
        H("FINANCING ACTIVITIES"),
        R("    Debt Raised",dbt_i), R("    Debt Repaid",dbt_r),
        {"type":"item","label":"    Dividends Paid","key":"cf_dividends",
         "values":[_v(div,j) for j in range(n)],"fmt":"currency","indent":1},
        R("    Share Buybacks / Issuance",bb),
        S("Cash from Financing (CFF)",cff_v,"cf_cff"),SP(),
        H("CASH RECONCILIATION"),
        R("    Beginning Cash","cf_beg_cash"),
        {"type":"item","label":"    Ending Cash → Balance Sheet","key":"cf_ending_cash",
         "values":end_v,"fmt":"currency","indent":1},SP(),
        H("FREE CASH FLOW"),
        S("Free Cash Flow (CFO + CapEx)",fcf_v,"cf_fcf"),
        M("  FCF Margin %",[_dv(fcf_v[j],_v(rev,j)) for j in range(n)]),
    ]


# ── Pass 2: Wire live cross-sheet formulas ───────────────────────────────────
def wire_formulas(wb, is_map, bs_map, cf_map, n):
    """
    Write Excel cross-sheet formulas into the three sheets.
    All IS/BS/CF values are already written as hardcoded numbers.
    This pass REPLACES key cells with live formulas so the model is connected.
    """
    def _ws(title):
        return wb[next(s for s in wb.sheetnames if s.strip().lower()==title.strip().lower())]

    def _addr(addr_str):
        m = re.match(r"'?([^'!]+)'?!([A-Z]+\d+)", addr_str)
        return (m.group(1), m.group(2)) if m else (None, None)

    ws_is = _ws(IS_SHEET)
    ws_bs = _ws(BS_SHEET)
    ws_cf = _ws(CF_SHEET)

    def _write(ws, addr_str, formula):
        _, cell_ref = _addr(addr_str)
        if not cell_ref: return
        cell = ws[cell_ref]
        if isinstance(cell, MergedCell): return
        cell.value = formula

    # 1. CF Net Income ← IS Net Income
    if "is_net_income" in is_map and "cf_net_income" in cf_map:
        for j in range(n):
            _write(ws_cf, cf_map["cf_net_income"][j], f"={is_map['is_net_income'][j]}")

    # 2. CF D&A ← IS D&A
    if "is_da" in is_map and "cf_da" in cf_map:
        for j in range(n):
            _write(ws_cf, cf_map["cf_da"][j], f"={is_map['is_da'][j]}")

    # 3. CF ΔAR ← BS AR(curr) - BS AR(prev)
    for bs_key, cf_key, sign in [("bs_ar","cf_d_ar",-1),("bs_inv","cf_d_inv",-1),("bs_ap","cf_d_ap",+1)]:
        if bs_key not in bs_map or cf_key not in cf_map: continue
        for j in range(n):
            if j == 0: continue  # no prior year
            curr = bs_map[bs_key][j]; prev = bs_map[bs_key][j-1]
            formula = f"=-({curr}-{prev})" if sign==-1 else f"={curr}-{prev}"
            _write(ws_cf, cf_map[cf_key][j], formula)

    # 4. CF Ending Cash formula = Beginning + CFO + CFI + CFF
    if "cf_ending_cash" in cf_map and "cf_cfo" in cf_map:
        beg_map = cf_map.get("cf_beg_cash") or []
        for j in range(n):
            dest = cf_map["cf_ending_cash"][j]
            _, cell_ref = _addr(dest)
            if not cell_ref: continue
            cfo_ref = cf_map["cf_cfo"][j]
            cfi_ref = cf_map.get("cf_cfi", [None]*n)[j]
            cff_ref = cf_map.get("cf_cff", [None]*n)[j]
            # Beginning of year j = ending cash of year j-1 (chain)
            if j > 0:
                beg_ref = cf_map["cf_ending_cash"][j-1]
                # Write the chain formula to the beg_cash cell
                if beg_map and j < len(beg_map) and beg_map[j]:
                    _, beg_cell = _addr(beg_map[j])
                    prev_end_sheet, prev_end_cell = _addr(cf_map["cf_ending_cash"][j-1])
                    if beg_cell and prev_end_cell:
                        ws_cf[beg_cell].value = f"={prev_end_cell}"
            elif beg_map and j < len(beg_map) and beg_map[j]:
                beg_ref = beg_map[j]    # explicit beginning cash row (already has value)
            else:
                beg_ref = None          # first year, no prior
            parts = [r for r in [beg_ref, cfo_ref, cfi_ref, cff_ref] if r]
            if parts:
                ws_cf[cell_ref].value = "=" + "+".join(parts)

    # 5. BS Cash ← CF Ending Cash
    if "cf_ending_cash" in cf_map and "bs_cash" in bs_map:
        for j in range(n):
            _write(ws_bs, bs_map["bs_cash"][j], f"={cf_map['cf_ending_cash'][j]}")

    # 6. BS Retained Earnings = prior RE + IS Net Income + CF Dividends
    if "bs_retained_earnings" in bs_map and "is_net_income" in is_map:
        for j in range(1, n):  # skip first year (no prior)
            dest = bs_map["bs_retained_earnings"][j]
            ni_ref   = is_map["is_net_income"][j]
            prev_re  = bs_map["bs_retained_earnings"][j-1]
            div_ref  = cf_map.get("cf_dividends", [None]*n)[j]
            formula  = f"={prev_re}+{ni_ref}+{div_ref}" if div_ref else f"={prev_re}+{ni_ref}"
            _write(ws_bs, dest, formula)

        # 6. BS: Retained Earnings = prior RE + IS Net Income + CF Dividends
    if "bs_retained_earnings" in bs_map and "is_net_income" in is_map:
        for j in range(1, n):
            dest = bs_map["bs_retained_earnings"][j]
            _, cell_ref = _addr(dest)
            if not cell_ref: continue
            ni_ref  = is_map["is_net_income"][j]
            prior_re = bs_map["bs_retained_earnings"][j-1]
            div_ref = cf_map.get("cf_dividends",[None]*n)[j]
            ws_bs[cell_ref].value = f"={prior_re}+{ni_ref}+{div_ref}" if div_ref else f"={prior_re}+{ni_ref}"

# 7. BS Balance Check = Total Assets - Total L+E
    if "bs_total_assets" in bs_map and "bs_total_le" in bs_map and "bs_balance_check" in bs_map:
        for j in range(n):
            ta_ref = bs_map["bs_total_assets"][j]
            le_ref = bs_map["bs_total_le"][j]
            _write(ws_bs, bs_map["bs_balance_check"][j], f"={ta_ref}-{le_ref}")

    print("  ✅ Cross-sheet formulas wired:")
    print("     CF Net Income / D&A ← Income Statement")
    print("     CF ΔAR / ΔInv / ΔAP ← Balance Sheet (curr−prior)")
    print("     CF Ending Cash = Beg + CFO + CFI + CFF (formula)")
    print("     BS Cash ← CF Ending Cash")
    print("     BS Retained Earnings = prior RE + Net Income + Dividends")
    print("     BS Balance Check = Total Assets − Total L+E")


# ── RUN ─────────────────────────────────────────────────────────────────────
if not OUTPUT_XLSX:
    print("❌ No Excel file found. Run cells 1-25 first.")
else:
    print(f"   Writing IS/BS/CF to: {OUTPUT_XLSX}")
    print(f"   Ticker: {TICKER}")
    try:
        # ── Use data already fetched by cell 10 (IS/BS/CF globals) ──────────────
        # Cell 10 sets IS, BS, CF as tidy DataFrames.
        # If those aren't available, fall back to re-fetching with yfinance.
        def _norm(df, n=5):
            if df is None or df.empty: return pd.DataFrame()
            df = df.copy()
            df.columns = pd.to_datetime(df.columns, errors="coerce")
            df = df.loc[:, df.columns.notna()].sort_index(axis=1)
            return df.iloc[:, -n:]

        # Load data: try globals (cell 10), then yfinance, then hardcoded fallback
        # Check multiple possible variable names that cell 10/12 might set
        def _fnne(*keys):
            for k in keys:
                v = globals().get(k)
                if v is not None and hasattr(v,"empty") and not v.empty: return v
            return None
        _IS_g = _fnne("IS","IS_raw","_is_df","yf_is")
        _BS_g = _fnne("BS","BS_raw","_bs_df","yf_bs")
        _CF_g = _fnne("CF","CF_raw","_cf_df","yf_cf")
        _have = lambda df: df is not None and not (hasattr(df,"empty") and df.empty)

        if _have(_IS_g) and _have(_BS_g) and _have(_CF_g):
            print("  Using IS/BS/CF from globals")
            IS_r = _norm(_IS_g)
            BS_r = _norm(_BS_g)
            CF_r = _norm(_CF_g)
        else:
            # Try yfinance first
            try:
                import yfinance as _yf26
                _t26 = _yf26.Ticker(TICKER)
                IS_r = _norm(_t26.financials)
                BS_r = _norm(_t26.balance_sheet)
                CF_r = _norm(_t26.cashflow)
                if IS_r.empty: raise ValueError("Empty")
                print(f"  Fetched from yfinance")
            except Exception as _yfe:
                _ISr = pd.DataFrame()
                _BSr = pd.DataFrame()
                _CFr = pd.DataFrame()
                print(f"   ⚠️  yfinance unavailable ({_yfe}) — IS/BS/CF empty")

        all_c = sorted(set(list(IS_r.columns)+list(BS_r.columns)+list(CF_r.columns)))[-5:]
        IS_r = IS_r.reindex(columns=all_c)
        BS_r = BS_r.reindex(columns=all_c)
        CF_r = CF_r.reindex(columns=all_c)
        hdrs = [f"FY{c.year}" for c in all_c]
        n    = len(hdrs)
        print(f"  Periods: {hdrs}")

        wb = load_workbook(OUTPUT_XLSX, keep_vba=str(OUTPUT_XLSX).lower().endswith(".xlsm"))

        # Pass 1: Write hardcoded values + collect cell address maps
        _, is_map = write_sheet(wb, IS_SHEET, build_is(IS_r, BS_r, CF_r, n), hdrs, SYM)
        print(f"✅ Income Statement written  ({len(is_map)} keyed rows)")

        _, bs_map = write_sheet(wb, BS_SHEET, build_bs(IS_r, BS_r, CF_r, n), hdrs, SYM)
        print(f"✅ Balance Sheet written     ({len(bs_map)} keyed rows)")

        _, cf_map = write_sheet(wb, CF_SHEET, build_cf(IS_r, BS_r, CF_r, n, is_map, bs_map), hdrs, SYM)
        print(f"✅ Cash Flow written         ({len(cf_map)} keyed rows)")

        # Pass 2: Wire live cross-sheet formulas
        print("\nWiring cross-sheet formulas...")
        wire_formulas(wb, is_map, bs_map, cf_map, n)



        # ── Pass 3: Write all formulas + data fixes ───────────────────────────────────
        from openpyxl.cell.cell import MergedCell as _MC
        from openpyxl.utils import get_column_letter as _gcl

        def _fw(ws, col, row, val):
            cell = ws[f"{col}{row}"]
            if not isinstance(cell, _MC): cell.value = val
        def _num(v): return float(v) if isinstance(v,(int,float)) else 0

        _IS=wb[IS_SHEET]; _BS=wb[BS_SHEET]; _CF=wb[CF_SHEET]; _ISN=IS_SHEET
        _dc=[_gcl(ci) for ci in range(2,10) if str(_IS.cell(2,ci).value or '').startswith('FY')]
        if not _dc: _dc=["B","C","D","E"]
        _pv={c:(_dc[i-1] if i>0 else None) for i,c in enumerate(_dc)}

        # yfinance values by column index (used for data anchors)
        def _yv2(df, keys, col_idx):
            if df is None or df.empty: return None
            import math as _m2
            for k in keys:
                hits=[i for i in df.index if k.lower() in str(i).lower()]
                if hits:
                    v=df.iloc[df.index.get_loc(hits[0]),col_idx]
                    if v is not None and not(isinstance(v,float) and _m2.isnan(v)):
                        return round(float(v)/1_000_000,2)
            return None

        # Build lookup dicts from IS_r/BS_r/CF_r (already in scope from data load above)
        _cols_n=len(_dc)
        _ta  =[_yv2(BS_r,["total assets"],j) or 0 for j in range(_cols_n)]
        _tca =[_yv2(BS_r,["current assets"],j) or 0 for j in range(_cols_n)]
        _ppe =[_yv2(BS_r,["net ppe","net property"],j) or 0 for j in range(_cols_n)]
        _gw  =[_yv2(BS_r,["goodwill"],j) or 0 for j in range(_cols_n)]
        _tl  =[_yv2(BS_r,["total liabilities net","total liab"],j) or 0 for j in range(_cols_n)]
        _tcl =[_yv2(BS_r,["current liabilities"],j) or 0 for j in range(_cols_n)]
        _eq  =[_yv2(BS_r,["stockholders equity","total equity gross"],j) or 0 for j in range(_cols_n)]
        _end =[_yv2(CF_r,["end cash position","cash at end"],j) or _yv2(BS_r,["cash and cash equivalents"],j) or 0 for j in range(_cols_n)]
        _beg =[_yv2(CF_r,["beginning cash position","begin cash"],j) or 0 for j in range(_cols_n)]
        _cfo =[_yv2(CF_r,["operating cash flow"],j) or 0 for j in range(_cols_n)]
        _cfi =[_yv2(CF_r,["investing cash flow"],j) or 0 for j in range(_cols_n)]
        _capx=[_yv2(CF_r,["capital expenditure"],j) or 0 for j in range(_cols_n)]
        _cff =[_yv2(CF_r,["financing cash flow"],j) or 0 for j in range(_cols_n)]
        _ar  =[_yv2(BS_r,["accounts receivable"],j) or 0 for j in range(_cols_n)]
        _inv =[_yv2(BS_r,["inventory"],j) or 0 for j in range(_cols_n)]
        _prep=[_yv2(BS_r,["prepaid"],j) or 0 for j in range(_cols_n)]
        _ni  =[_yv2(IS_r,["net income"],j) or 0 for j in range(_cols_n)]
        _da  =[abs(_yv2(IS_r,["reconciled depreciation","depreciation"],j) or 0) for j in range(_cols_n)]
        _ap  =[_yv2(BS_r,["accounts payable"],j) or 0 for j in range(_cols_n)]

        # IS formulas
        for _j,_c in enumerate(_dc):
            _fw(_IS,_c, 6,f"={_c}4-{_c}5")
            _fw(_IS,_c, 7,f'=IFERROR({_c}6/{_c}4,"")')
            _fw(_IS,_c,13,f"={_c}6-IF(ISNUMBER({_c}10),{_c}10,0)-{_c}11-{_c}12")
            _fw(_IS,_c,14,f'=IFERROR({_c}13/{_c}4,"")')
            _fw(_IS,_c,18,f"={_c}13-{_c}17")
            _fw(_IS,_c,19,f'=IFERROR({_c}18/{_c}4,"")')
            _fw(_IS,_c,24,f"={_c}18+IF(ISNUMBER({_c}22),{_c}22,0)-IF(ISNUMBER({_c}23),{_c}23,0)")
            _fw(_IS,_c,26,f'=IFERROR({_c}25/{_c}24,"")')
            _fw(_IS,_c,27,f"={_c}24-{_c}25")
            _fw(_IS,_c,28,f'=IFERROR({_c}27/{_c}4,"")')
            _fw(_IS,_c,32,f'=IFERROR({_c}27/{_c}31,"")')

        # BS data anchors + formulas
        for _j,_c in enumerate(_dc):
            if _gw[_j]: _fw(_BS,_c,13,_gw[_j])
            if _ta[_j] and _tca[_j] and _ppe[_j]:
                _fw(_BS,_c,14,round(_ta[_j]-_tca[_j]-_ppe[_j]-_gw[_j],2))
            if _tca[_j] and _end[_j] and _ar[_j]:
                _fw(_BS,_c,8,round(_tca[_j]-_end[_j]-_ar[_j]-_inv[_j]-_prep[_j],2))
            if _tl[_j] and _tcl[_j]: _fw(_BS,_c,26,round(_tl[_j]-_tcl[_j],2))
            if _eq[_j]: _fw(_BS,_c,31,_eq[_j])
            _fw(_BS,_c, 9,f"=SUM({_c}4:{_c}8)")
            _fw(_BS,_c,15,f"={_c}12+{_c}13+{_c}14")
            _fw(_BS,_c,16,f"={_c}9+{_c}15")
            _fw(_BS,_c,22,f"={_c}19+{_c}20+{_c}21")
            _fw(_BS,_c,27,f"={_c}22+{_c}26")
            _fw(_BS,_c,32,f"={_c}27+{_c}31")
            _fw(_BS,_c,36,f"={_c}9-{_c}22")
            _fw(_BS,_c,37,f'=IFERROR({_c}5/\'{_ISN}\'!{_c}4*365,"")')
            _fw(_BS,_c,38,f'=IFERROR({_c}6/ABS(\'{_ISN}\'!{_c}5)*365,"")')
            _fw(_BS,_c,39,f'=IFERROR({_c}19/ABS(\'{_ISN}\'!{_c}5)*365,"")')
            _fw(_BS,_c,40,f"={_c}37+{_c}38-{_c}39")
            _fw(_BS,_c,41,f'=IFERROR(({_c}20+{_c}25)/{_c}31,"")')

        # CF plug rows + formulas
        _CF.cell(17,1).value="      Other Investing Activities"
        _CF.cell(24,1).value="      Other Financing Activities"
        for _j,_c in enumerate(_dc):
            _prev=_pv[_c]; _pj=_dc.index(_prev) if _prev else None
            _dar =-(_ar[_j] -(_ar[_pj]  if _pj is not None else _ar[_j]))
            _dinv=-(_inv[_j]-(_inv[_pj] if _pj is not None else _inv[_j]))
            _dap  =(_ap[_j] -(_ap[_pj]  if _pj is not None else _ap[_j]))
            _owc =_num(_CF[f"{_c}9"].value)
            _listed=_ni[_j]+_da[_j]+_dar+_dinv+_dap+_owc
            _fw(_CF,_c,10,round(_cfo[_j]-_listed,0))        # OtherOps plug
            _fw(_CF,_c,11,f"=SUM({_c}4:{_c}10)")            # CFO = SUM formula
            _fw(_CF,_c,12,f'=IFERROR({_c}11/\'{_ISN}\'!{_c}4,"")')
            _fw(_CF,_c,15,_capx[_j])                         # CapEx
            _fw(_CF,_c,17,round(_cfi[_j]-_capx[_j],2))      # Other Investing plug
            _fw(_CF,_c,16,f"={_c}15+{_c}17")                # CFI formula
            _det=sum(_num(_CF[f"{_c}{_r}"].value) for _r in [19,20,21,22])
            _fw(_CF,_c,24,round(_cff[_j]-_det,2))            # Other Financing plug
            _fw(_CF,_c,23,f"={_c}19+{_c}20+{_c}21+{_c}22+{_c}24")  # CFF formula
            _fw(_CF,_c,27,_end[_j])                          # Ending Cash (authoritative)
            _fw(_CF,_c,26,f"={_pv[_c]}27" if _pv[_c] else (_beg[_j] or 0))  # Beg Cash
            _fw(_CF,_c,30,f"={_c}11+{_c}15")                # FCF
            _fw(_CF,_c,31,f'=IFERROR({_c}30/\'{_ISN}\'!{_c}4,"")')

        print("   ✅ Pass 3 complete — all IS/BS/CF formulas and data anchors written")

        wb.save(OUTPUT_XLSX)
        print(f"\n✅ Saved: {OUTPUT_XLSX}")
        print(f"   Sheets: {wb.sheetnames}")

    except Exception as _e:
        import traceback
        print(f"\n❌ CELL 26 ERROR: {_e}")
        traceback.print_exc()
        print("\n⚠️  Check error above and re-run cell 26.")


   Writing IS/BS/CF to: /Users/sharankothari/Desktop/research_platform/DCF_Output_RELIANCE.NS_INR.xlsm
   Ticker: RELIANCE.NS
  Using IS/BS/CF from globals
  Periods: []
✅ Income Statement written  (8 keyed rows)
✅ Balance Sheet written     (16 keyed rows)
✅ Cash Flow written         (12 keyed rows)

Wiring cross-sheet formulas...
  ✅ Cross-sheet formulas wired:
     CF Net Income / D&A ← Income Statement
     CF ΔAR / ΔInv / ΔAP ← Balance Sheet (curr−prior)
     CF Ending Cash = Beg + CFO + CFI + CFF (formula)
     BS Cash ← CF Ending Cash
     BS Retained Earnings = prior RE + Net Income + Dividends
     BS Balance Check = Total Assets − Total L+E
   ✅ Pass 3 complete — all IS/BS/CF formulas and data anchors written

✅ Saved: /Users/sharankothari/Desktop/research_platform/DCF_Output_RELIANCE.NS_INR.xlsm
   Sheets: ['DCF - Original', 'DCF ', 'Cost of Capital', 'Income statement', 'Balance Sheet', 'Cash Flow']


/var/folders/5k/x_y2jtd54tx16f_my1zkkhk80000gn/T/ipykernel_40594/3187283547.py:574: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df.columns = pd.to_datetime(df.columns, errors="coerce")
/var/folders/5k/x_y2jtd54tx16f_my1zkkhk80000gn/T/ipykernel_40594/3187283547.py:574: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df.columns = pd.to_datetime(df.columns, errors="coerce")
/var/folders/5k/x_y2jtd54tx16f_my1zkkhk80000gn/T/ipykernel_40594/3187283547.py:574: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df.columns = pd.to_datetime(df.columns, errors="coerce")


# Part 8 — Industry Comparison

In [21]:
# =============================================================================
# PHASE 3C — Industry Comparison v2.1
# PATCH: Fixed US company comparing with Indian peers (BAC → US_Financials not India)
# Uses PeerEngine + multi-source data (screener.in + yfinance)
# Supports: listed (India/USA) + unlisted (CSV/Excel files)
# No changes needed here — edit only Master Config (Cell 3)
# =============================================================================

import os, sys, time
import numpy as np
import pandas as pd
import yfinance as yf
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter

# ── Resolve globals ───────────────────────────────────────────────────────────
TICKER        = str(globals().get("TICKER", "MSFT")).upper().strip()
OUTPUT_XLSX   = globals().get("OUTPUT_XLSM") or globals().get("OUTPUT_XLSX")
PEERS_OVERRIDE= globals().get("PEERS_OVERRIDE", None)
UNLISTED_FILE = globals().get("UNLISTED_FILE", None)
COMPANY_NAME_OVERRIDE = globals().get("COMPANY_NAME_OVERRIDE", None)
_sources      = globals().get("SOURCES", {})
_api_on       = _sources.get("api", True)
_hist         = globals().get("HIST", None)

if not OUTPUT_XLSX:
    raise RuntimeError("OUTPUT_XLSM/OUTPUT_XLSX not found. Run Phase 5 first.")

SHEET_NAME = "Industry Comparison"
print(f"ℹ️  Industry Comparison — Ticker: {TICKER}")

# ─────────────────────────────────────────────────────────────────────────────
# SAFE MATH
# ─────────────────────────────────────────────────────────────────────────────
def _f(x):
    try:
        if x is None: return None
        if isinstance(x, (int, float, np.number)):
            v = float(x)
            return None if (np.isnan(v) or np.isinf(v)) else v
        s = str(x).replace(",","").replace("%","").strip()
        if s.lower() in ("","nan","none","n/a","-","—"): return None
        return float(s)
    except: return None

def _div(a, b):
    a, b = _f(a), _f(b)
    return None if (a is None or b in (None, 0)) else a / b

def _mean(vals):
    xs = [_f(v) for v in vals if _f(v) is not None]
    return float(np.mean(xs)) if xs else None

# ─────────────────────────────────────────────────────────────────────────────
# PEER UNIVERSE (sub-industry keyed)
# ─────────────────────────────────────────────────────────────────────────────
PEER_UNIVERSE = {
    "PSU Bank": ["SBIN.NS","PNB.NS","BANKBARODA.NS","CANBK.NS","UNIONBANK.NS","BANKINDIA.NS","INDIANB.NS"],
    "Private Bank": ["HDFCBANK.NS","ICICIBANK.NS","KOTAKBANK.NS","AXISBANK.NS","INDUSINDBK.NS","YESBANK.NS","IDFCFIRSTB.NS","FEDERALBNK.NS","RBLBANK.NS","BANDHANBNK.NS"],
    "Small Finance Bank": ["AUBANK.NS","UJJIVANSFB.NS","EQUITASBNK.NS","ESAFSFB.NS","SURYODAY.NS"],
    "NBFC": ["BAJFINANCE.NS","CHOLAFIN.NS","SHRIRAMFIN.NS","MUTHOOTFIN.NS","MANAPPURAM.NS","M&MFIN.NS","SUNDARMFIN.NS","LTFH.NS"],
    "HFC": ["LICHSGFIN.NS","PNBHOUSING.NS","CANFINHOME.NS","HOMEFIRST.NS","AAVAS.NS","APTUS.NS"],
    "MFI": ["CREDITACC.NS","SPANDANA.NS","ARMANFIN.NS","FUSION.NS"],
    "Life Insurance": ["LICI.NS","HDFCLIFE.NS","SBILIFE.NS","ICICIPRULI.NS"],
    "General Insurance": ["STARHEALTH.NS","NIACL.NS","ICICIGI.NS","GICRE.NS"],
    "AMC": ["HDFCAMC.NS","NAM-INDIA.NS","UTIAMC.NS","360ONE.NS","NUVAMA.NS"],
    "Broking & Wealth": ["ANGELONE.NS","360ONE.NS","NUVAMA.NS","IIFLSEC.NS","MOTILALOFS.NS"],
    "IT Services": ["TCS.NS","INFY.NS","HCLTECH.NS","WIPRO.NS","TECHM.NS","LTIM.NS","MPHASIS.NS","PERSISTENT.NS","COFORGE.NS"],
    "IT Products": ["INFY.NS","TATAELXSI.NS","KPIT.NS","MASTEK.NS","ZENSAR.NS"],
    "Large Pharma": ["SUNPHARMA.NS","DRREDDY.NS","CIPLA.NS","LUPIN.NS","AUROPHARMA.NS","ALKEM.NS","TORNTPHARM.NS","IPCALAB.NS"],
    "Specialty Pharma": ["AUROPHARMA.NS","NATCOPHARMA.NS","GRANULES.NS","LAURUSLABS.NS"],
    "Biotech": ["BIOCON.NS","DIVI.NS","SYNGENE.NS"],
    "Healthcare Services": ["APOLLOHOSP.NS","FORTIS.NS","MAXHEALTH.NS","METROPOLIS.NS","LALPATHLAB.NS"],
    "FMCG": ["HINDUNILVR.NS","ITC.NS","NESTLEIND.NS","BRITANNIA.NS","DABUR.NS","MARICO.NS","GODREJCP.NS","EMAMILTD.NS","COLPAL.NS","TATACONSUM.NS"],
    "Retail": ["DMART.NS","TRENT.NS","ABFRL.NS","BATA.NS","NYKAA.NS"],
    "Auto OEM": ["MARUTI.NS","TATAMOTORS.NS","M&M.NS","BAJAJ-AUTO.NS","HEROMOTOCO.NS","TVSMOTORS.NS","EICHERMOT.NS","ASHOKLEY.NS"],
    "Auto Ancillary": ["MOTHERSON.NS","BOSCHLTD.NS","EXIDEIND.NS","AMARAJABAT.NS","SUNDRMFAST.NS","MINDAIND.NS","BHARATFORG.NS"],
    "Steel": ["TATASTEEL.NS","JSWSTEEL.NS","SAIL.NS","JINDALSTEL.NS","NMDC.NS"],
    "Metals & Mining": ["HINDALCO.NS","VEDL.NS","NMDC.NS","COALINDIA.NS"],
    "Cement": ["ULTRACEMCO.NS","SHREECEM.NS","AMBUJACEM.NS","ACC.NS","DALBHARAT.NS","RAMCOCEM.NS","JKCEMENT.NS"],
    "Chemicals": ["PIDILITIND.NS","AARTIIND.NS","CLEANSCIEN.NS","NAVINFLUOR.NS","DEEPAKNITR.NS"],
    "Integrated Oil & Gas": ["RELIANCE.NS","ONGC.NS","IOC.NS","BPCL.NS","HPCL.NS","GAIL.NS"],
    "Utilities": ["NTPC.NS","POWERGRID.NS","TATAPOWER.NS","ADANIPOWER.NS","ADANIGREEN.NS","TORNTPOWER.NS"],
    "Telecom": ["BHARTIARTL.NS","IDEA.NS","TATACOMM.NS","ROUTE.NS"],
    "Real Estate Developer": ["DLF.NS","GODREJPROP.NS","OBEROIRLTY.NS","PRESTIGE.NS","BRIGADE.NS","SOBHA.NS","MACROTECH.NS"],
    "Capital Goods": ["LT.NS","SIEMENS.NS","ABB.NS","BHEL.NS","CUMMINSIND.NS","THERMAX.NS","HAVELLS.NS","CGPOWER.NS"],
    "Aerospace & Defense": ["HAL.NS","BEL.NS","BEML.NS","MTAR.NS","DATAPATTNS.NS"],
    "Construction & Engineering": ["LT.NS","NCC.NS","KEC.NS","IRCON.NS","RVNL.NS","HGINFRA.NS"],
    "Conglomerates": ["RELIANCE.NS","TATAMOTORS.NS","ADANIENT.NS","ITC.NS","LT.NS"],
    "US_Financials": ["JPM","BAC","WFC","GS","MS","C","USB","PNC","TFC","COF"],
    "US_Information Technology": ["AAPL","MSFT","NVDA","GOOGL","META","ADBE","CRM","AMD","INTC","ORCL"],
    "US_Health Care": ["UNH","JNJ","PFE","MRK","ABBV","LLY","TMO","ABT","MDT","BMY"],
    "US_Energy": ["XOM","CVX","COP","EOG","SLB","MPC","PSX","VLO","OXY"],
    "US_Consumer Staples": ["PG","KO","PEP","WMT","COST","PM","MO","CL","KHC","GIS"],
    "US_Consumer Discretionary": ["AMZN","TSLA","HD","MCD","NKE","SBUX","TGT","LOW","BKNG"],
    "US_Materials": ["LIN","APD","NEM","FCX","NUE","PPG","ECL","DD"],
    "US_Industrials": ["CAT","DE","HON","UNP","BA","GE","MMM","RTX","LMT","NOC"],
    "US_Utilities": ["NEE","DUK","SO","AEP","EXC","SRE","PEG","XEL","WEC"],
    "US_Communication Services": ["GOOGL","META","NFLX","DIS","CMCSA","T","VZ","TMUS"],
    "US_Real Estate": ["PLD","AMT","EQIX","O","SPG","CCI","WELL","AVB","EQR","PSA"],
    "US_Steel": ["NUE","STLD","CLF","X","RS"],
    "US_Chemicals": ["LIN","APD","DD","ECL","PPG","ALB","FMC"],
    "US_IT Services": ["ACN","IBM","CTSH","INFY","WIT","EPAM"],
}

# ─────────────────────────────────────────────────────────────────────────────
# CLASSIFIER
# FIX 1: India name rules only run for .NS/.BO tickers.
#         Prevents "Bank of America" (BAC) matching "bank" → Indian peer list.
# ─────────────────────────────────────────────────────────────────────────────
PSU_FRAGMENTS = ["state bank","sbi","bank of baroda","bank of india","bank of maharashtra",
                 "canara bank","central bank","indian bank","indian overseas","punjab national",
                 "pnb","uco bank","union bank"]

SECTOR_FROM_SUB = {
    "PSU Bank":"Financials","Private Bank":"Financials","Small Finance Bank":"Financials",
    "NBFC":"Financials","HFC":"Financials","Life Insurance":"Financials",
    "General Insurance":"Financials","AMC":"Financials","Broking & Wealth":"Financials","MFI":"Financials",
    "IT Services":"Information Technology","IT Products":"Information Technology","Semiconductors":"Information Technology",
    "Large Pharma":"Health Care","Specialty Pharma":"Health Care","Biotech":"Health Care",
    "Healthcare Services":"Health Care","Medical Devices":"Health Care",
    "FMCG":"Consumer Staples","Retail":"Consumer Discretionary",
    "Auto OEM":"Consumer Discretionary","Auto Ancillary":"Consumer Discretionary",
    "Steel":"Materials","Metals & Mining":"Materials","Cement":"Materials","Chemicals":"Materials",
    "Integrated Oil & Gas":"Energy","Oil & Gas Refining":"Energy","Utilities":"Utilities",
    "Telecom":"Communication Services","Real Estate Developer":"Real Estate","REIT":"Real Estate",
    "Capital Goods":"Industrials","Conglomerates":"Industrials",
    "Aerospace & Defense":"Industrials","Construction & Engineering":"Industrials",
}

YF_INDUSTRY_MAP = {
    "banks—diversified":"Private Bank","banks—regional":"Private Bank","banks":"Private Bank",
    "credit services":"NBFC","insurance—life":"Life Insurance","insurance—diversified":"Life Insurance",
    "insurance—property & casualty":"General Insurance","insurance":"General Insurance",
    "asset management":"AMC","capital markets":"AMC",
    "information technology services":"IT Services","software—application":"IT Products",
    "drug manufacturers—general":"Large Pharma","drug manufacturers—specialty":"Specialty Pharma",
    "biotechnology":"Biotech","hospitals":"Healthcare Services","healthcare plans":"Healthcare Services",
    "beverages—non-alcoholic":"FMCG","household & personal products":"FMCG","packaged foods":"FMCG",
    "tobacco":"FMCG","discount stores":"Retail","grocery stores":"Retail","specialty retail":"Retail",
    "aerospace & defense":"Aerospace & Defense","industrial conglomerates":"Conglomerates",
    "electrical equipment":"Capital Goods","machinery":"Capital Goods",
    "oil & gas integrated":"Integrated Oil & Gas","oil & gas refining":"Oil & Gas Refining",
    "utilities—regulated electric":"Utilities","utilities—diversified":"Utilities",
    "steel":"Steel","aluminum":"Metals & Mining","chemicals":"Chemicals","specialty chemicals":"Chemicals",
    "cement":"Cement","real estate—development":"Real Estate Developer",
    "telecom services":"Telecom","auto manufacturers":"Auto OEM","auto parts":"Auto Ancillary",
}

def classify_ticker(ticker, yf_info):
    name   = (yf_info.get("shortName") or yf_info.get("longName") or ticker).lower()
    yf_ind = (yf_info.get("industry") or "").lower().strip()
    yf_sec = (yf_info.get("sector")   or "").lower().strip()

    # ── PATCH FIX 1: guard India name rules behind .NS/.BO check ─────────────
    _is_india_ticker = ticker.upper().endswith((".NS", ".BO"))

    def _r(si, conf, m):
        return {"sub_industry":si, "sector":SECTOR_FROM_SUB.get(si,"Unknown"),
                "confidence":conf, "method":m}

    # Name rules ONLY for India tickers
    if _is_india_ticker:
        if any(f in name for f in PSU_FRAGMENTS):  return _r("PSU Bank",0.97,"name:psu")
        if "small finance bank" in name:            return _r("Small Finance Bank",0.95,"name:sfb")
        if "bank" in name:                          return _r("Private Bank",0.92,"name:bank")
        if any(x in name for x in ["housing finance","home finance"]): return _r("HFC",0.95,"name:hfc")
        if any(x in name for x in ["microfinance","micro finance"]):   return _r("MFI",0.93,"name:mfi")
        if "life insurance" in name:                return _r("Life Insurance",0.95,"name:life")
        if "insurance" in name:                     return _r("General Insurance",0.80,"name:ins")
        if any(x in name for x in ["asset management","mutual fund"," amc"]): return _r("AMC",0.92,"name:amc")
        if any(x in name for x in ["securities","broking","brokerage"]): return _r("Broking & Wealth",0.88,"name:broker")
        if any(x in name for x in ["finance","finserv","lending"]):
            if not any(x in name for x in ["tech","info","software"]): return _r("NBFC",0.82,"name:nbfc")
        if any(x in name for x in ["technologies","tech","software","infosys","wipro","tcs","hcl","mphasis","persistent","coforge"]): return _r("IT Services",0.88,"name:it")
        if any(x in name for x in ["pharma","pharmaceutical","cipla","lupin","aurobindo","alkem"]): return _r("Large Pharma",0.88,"name:pharma")
        if any(x in name for x in ["hospital","healthcare","diagnostic"]): return _r("Healthcare Services",0.85,"name:hc")
        if any(x in name for x in ["hindustan unilever","nestl","britannia","dabur","marico","godrej consumer","emami","colgate"]): return _r("FMCG",0.95,"name:fmcg")
        if any(x in name for x in ["maruti","tata motors","mahindra","bajaj auto","hero motocorp","tvs motor","eicher","ashok leyland"]): return _r("Auto OEM",0.92,"name:auto")
        if any(x in name for x in ["steel","jswsteel","sail","jindal"]):  return _r("Steel",0.90,"name:steel")
        if any(x in name for x in ["hindalco","vedanta","hindustan zinc","coal india"]): return _r("Metals & Mining",0.88,"name:metals")
        if any(x in name for x in ["cement","ultratech","shree cement","ambuja","acc"]): return _r("Cement",0.92,"name:cement")
        if any(x in name for x in ["chemical","pidilite","aarti","clean science"]): return _r("Chemicals",0.88,"name:chem")
        if any(x in name for x in ["reliance","ongc","oil india","bpcl","hpcl","indian oil"]): return _r("Integrated Oil & Gas",0.90,"name:oil")
        if any(x in name for x in ["ntpc","power grid","tata power","adani power","adani green","torrent power"]): return _r("Utilities",0.90,"name:util")
        if any(x in name for x in ["airtel","bharti","vodafone","idea","tata comm"]): return _r("Telecom",0.90,"name:tel")
        if any(x in name for x in ["dlf","godrej prop","oberoi","prestige","brigade","sobha","lodha","macrotech"]): return _r("Real Estate Developer",0.90,"name:re")
        if any(x in name for x in ["larsen","l&t","siemens","abb","bhel","cummins","thermax","havells"]): return _r("Capital Goods",0.88,"name:cg")

    # yfinance industry string — works for both India and USA
    for k, si in YF_INDUSTRY_MAP.items():
        if k in yf_ind: return _r(si, 0.70, f"yf_ind:{k}")

    # yfinance sector fallback — works for both India and USA
    SECT_FB = {"financial services":("NBFC",0.50),"technology":("IT Services",0.50),
               "healthcare":("Healthcare Services",0.50),"consumer cyclical":("Retail",0.50),
               "consumer defensive":("FMCG",0.50),"basic materials":("Chemicals",0.50),
               "energy":("Integrated Oil & Gas",0.50),"industrials":("Capital Goods",0.50),
               "utilities":("Utilities",0.55),"real estate":("Real Estate Developer",0.50),
               "communication services":("Telecom",0.50)}
    for k,(si,conf) in SECT_FB.items():
        if k in yf_sec: return _r(si, conf, f"yf_sec:{k}")

    return _r("Unknown", 0.20, "no_signal")

# ─────────────────────────────────────────────────────────────────────────────
# SCREENER.IN INDUSTRY TAG (free, no login)
# ─────────────────────────────────────────────────────────────────────────────
SCREENER_TO_SUB = {
    "private sector bank":"Private Bank","public sector bank":"PSU Bank",
    "small finance bank":"Small Finance Bank","non-banking financial company":"NBFC",
    "housing finance company":"HFC","housing finance":"HFC","microfinance institution":"MFI",
    "life insurance":"Life Insurance","general insurance":"General Insurance",
    "asset management company":"AMC","stockbroking":"Broking & Wealth",
    "it - software":"IT Services","it services & consulting":"IT Services",
    "pharmaceuticals":"Large Pharma","biotechnology":"Biotech",
    "hospitals & diagnostic centres":"Healthcare Services",
    "fmcg":"FMCG","consumer food":"FMCG","automobiles":"Auto OEM","auto ancillaries":"Auto Ancillary",
    "steel":"Steel","iron & steel":"Steel","aluminium":"Metals & Mining","cement":"Cement",
    "chemicals":"Chemicals","specialty chemicals":"Chemicals",
    "oil & gas":"Integrated Oil & Gas","refineries":"Oil & Gas Refining",
    "power":"Utilities","power generation":"Utilities","renewable energy":"Utilities",
    "telecom":"Telecom","telecommunications":"Telecom","realty":"Real Estate Developer",
    "capital goods":"Capital Goods","engineering":"Capital Goods","defence":"Aerospace & Defense",
    "construction":"Construction & Engineering",
}

def get_screener_sub_industry(ticker):
    """Try to get precise industry tag from screener.in. Falls back silently."""
    try:
        import requests
        from bs4 import BeautifulSoup
        sym = ticker.upper().replace(".NS","").replace(".BO","")
        url = f"https://www.screener.in/company/{sym}/consolidated/"
        headers = {"User-Agent":"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"}
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200: return None
        soup = BeautifulSoup(resp.text, "html.parser")
        industry_raw = None
        for a in soup.select(".sub-text a, #peers a"):
            href = a.get("href","")
            text = a.get_text(strip=True)
            if "/market/" in href and text and len(text) > 3:
                industry_raw = text
        if not industry_raw: return None
        mapped = SCREENER_TO_SUB.get(industry_raw.lower().strip())
        if not mapped:
            for k,v in SCREENER_TO_SUB.items():
                if k in industry_raw.lower(): mapped = v; break
        return mapped
    except: return None

# ─────────────────────────────────────────────────────────────────────────────
# MARKET CAP FILTER (ratio-based, 3-stage fallback)
# ─────────────────────────────────────────────────────────────────────────────
def _get_mc(ticker):
    try:
        mc = yf.Ticker(ticker).info.get("marketCap")
        return float(mc) if mc else None
    except: return None

def filter_by_cap(target_mc, candidates):
    if not target_mc: return candidates, "no_cap"
    for lo,hi,label in [(0.3,3.0,"tight"),(0.2,5.0,"normal"),(0.1,10.0,"relaxed")]:
        out = [t for t in candidates if (lambda mc: mc is None or lo*target_mc <= mc <= hi*target_mc)(_get_mc(t))]
        if len(out) >= 3: return out, label
    return candidates, "no_filter"

# ─────────────────────────────────────────────────────────────────────────────
# PEER SCORER (null-safe)
# ─────────────────────────────────────────────────────────────────────────────
def score_and_rank(target_ticker, candidates):
    try:
        ti = yf.Ticker(target_ticker).info or {}
        t_sig = {"mc":ti.get("marketCap"),"roe":ti.get("returnOnEquity"),"growth":ti.get("revenueGrowth")}
    except: t_sig = {"mc":None,"roe":None,"growth":None}

    scored = []
    for t in candidates:
        try: pi = yf.Ticker(t).info or {}
        except: pi = {}
        p_sig = {"mc":pi.get("marketCap"),"roe":pi.get("returnOnEquity"),"growth":pi.get("revenueGrowth")}

        W = {"mc":0.40,"roe":0.30,"growth":0.30}
        parts = []; total_w = 0.0; expl = []
        tc,pc = _f(t_sig["mc"]),_f(p_sig["mc"])
        if tc and pc and tc>0 and pc>0:
            s = abs(np.log(pc/tc)); parts.append((s,W["mc"])); total_w+=W["mc"]
            expl.append(f"Cap {pc/tc:.1f}x")
        tr,pr = _f(t_sig["roe"]),_f(p_sig["roe"])
        if tr is not None and pr is not None:
            s = abs(pr-tr); parts.append((s,W["roe"])); total_w+=W["roe"]
            expl.append(f"ROE diff {abs(pr-tr)*100:.1f}pp")
        tg,pg = _f(t_sig["growth"]),_f(p_sig["growth"])
        if tg is not None and pg is not None:
            s = abs(pg-tg); parts.append((s,W["growth"])); total_w+=W["growth"]
            expl.append(f"Growth diff {abs(pg-tg)*100:.1f}pp")

        raw = (sum(s*w for s,w in parts)/total_w) if (parts and total_w>0) else 999.0
        conf = max(0,min(100,int(100-raw*80)))
        scored.append({"ticker":t,"score":raw,"conf_pct":conf,"expl":" | ".join(expl) or "no data"})
    scored.sort(key=lambda x:x["score"])
    return scored

# ─────────────────────────────────────────────────────────────────────────────
# DYNAMIC METRIC SECTIONS
# ─────────────────────────────────────────────────────────────────────────────
try:
    from sector_metrics import get_metric_sections, is_pct_metric, HIGHER_IS_BETTER, LOWER_IS_BETTER
    print("  ✅ sector_metrics.py loaded")
except ImportError:
    def is_pct_metric(m):
        return "(%)" in m or "Growth" in m or "Margin" in m or "Yield" in m or "Payout" in m

    HIGHER_IS_BETTER = {"Revenue Growth (YoY %)","Revenue CAGR (3Y %)","Revenue CAGR (5Y %)",
        "Gross Margin (%)","EBITDA Margin (%)","Operating Margin (%)","Net Margin (%)","EBIT Margin (%)",
        "ROE (%)","ROCE (%)","ROA (%)","ROIC (%)","Free Cash Flow Margin (%)","Dividend Yield (%)",
        "Interest Coverage","Current Ratio","Net Interest Margin (%)","CASA Ratio (%)","Loan Growth (%)",
        "Provision Coverage Ratio (%)"}
    LOWER_IS_BETTER = {"Net Debt / EBITDA","Debt to Equity","Beta (5Y)",
        "Gross NPA (%)","Net NPA (%)","Credit Cost (%)","Cost-to-Income Ratio (%)"}

    def get_metric_sections(sub_industry):
        UNIVERSAL = {
            "Growth": ["Revenue Growth (YoY %)","Revenue CAGR (3Y %)","Revenue CAGR (5Y %)","Profit Growth (5Y %)"],
            "Returns": ["ROE (%)","ROCE (%)","ROA (%)"],
            "Financial Strength": ["Net Debt / EBITDA","Debt to Equity","Interest Coverage","Current Ratio","Beta (5Y)"],
            "Cash Flow": ["Free Cash Flow Margin (%)","Dividend Yield (%)","Dividend Payout (%)"],
        }
        SPECIFIC = {
            "Private Bank":{"Profitability":["Net Interest Margin (%)","Cost-to-Income Ratio (%)","Net Margin (%)"],
                "Asset Quality":["Gross NPA (%)","Net NPA (%)","Provision Coverage Ratio (%)"],
                "Business":["CASA Ratio (%)","Loan Growth (%)","Credit-Deposit Ratio (%)"],
                "Valuation":["P/E (TTM)","P/B Ratio"]},
            "PSU Bank":{"Profitability":["Net Interest Margin (%)","Cost-to-Income Ratio (%)","Net Margin (%)"],
                "Asset Quality":["Gross NPA (%)","Net NPA (%)","Provision Coverage Ratio (%)"],
                "Business":["CASA Ratio (%)","Loan Growth (%)","Credit-Deposit Ratio (%)"],
                "Valuation":["P/E (TTM)","P/B Ratio"]},
            "NBFC":{"Profitability":["Net Interest Margin (%)","Operating Margin (%)","Net Margin (%)"],
                "Asset Quality":["Gross NPA (%)","Net NPA (%)"],
                "Valuation":["P/E (TTM)","P/B Ratio","EV / EBITDA"]},
            "IT Services":{"Profitability":["Gross Margin (%)","EBIT Margin (%)","EBITDA Margin (%)","Net Margin (%)"],
                "Valuation":["P/E (TTM)","P/E (Forward)","EV / EBITDA","EV / Sales"]},
            "Large Pharma":{"Profitability":["Gross Margin (%)","EBITDA Margin (%)","Net Margin (%)"],
                "Valuation":["P/E (TTM)","EV / EBITDA","EV / Sales"]},
            "FMCG":{"Profitability":["Gross Margin (%)","EBITDA Margin (%)","Net Margin (%)"],
                "Growth":["Revenue Growth (YoY %)","Revenue CAGR (3Y %)","Revenue CAGR (5Y %)","Revenue CAGR (10Y %)"],
                "Valuation":["P/E (TTM)","EV / EBITDA","EV / Sales","Price / Book"]},
            "Steel":{"Profitability":["Gross Margin (%)","EBITDA Margin (%)","Net Margin (%)"],
                "Leverage":["Net Debt / EBITDA","Debt to Equity","Interest Coverage"],
                "Valuation":["P/E (TTM)","EV / EBITDA","EV / Sales"]},
            "Cement":{"Profitability":["Gross Margin (%)","EBITDA Margin (%)","Net Margin (%)"],
                "Leverage":["Net Debt / EBITDA","Debt to Equity","Interest Coverage"],
                "Valuation":["P/E (TTM)","EV / EBITDA","EV / Sales"]},
            "Telecom":{"Profitability":["EBITDA Margin (%)","EBIT Margin (%)","Net Margin (%)"],
                "Leverage":["Net Debt / EBITDA","Debt to Equity","Interest Coverage"],
                "Valuation":["P/E (TTM)","EV / EBITDA","EV / Sales"]},
        }
        DEFAULT = {"Profitability":["Gross Margin (%)","EBITDA Margin (%)","Operating Margin (%)","Net Margin (%)"],
                   "Valuation":["P/E (TTM)","P/E (Forward)","EV / EBITDA","EV / Sales","Price / Book"]}
        merged = {}; merged.update(UNIVERSAL); merged.update(SPECIFIC.get(sub_industry, DEFAULT))
        return merged

# ─────────────────────────────────────────────────────────────────────────────
# METRICS FETCHER
# ─────────────────────────────────────────────────────────────────────────────
BANKING_DEFAULTS = {
    "HDFCBANK.NS":{"Net Interest Margin (%)":3.27,"Gross NPA (%)":1.33,"Net NPA (%)":0.43,"CASA Ratio (%)":33.6},
    "ICICIBANK.NS":{"Net Interest Margin (%)":4.32,"Gross NPA (%)":1.96,"Net NPA (%)":0.42,"CASA Ratio (%)":39.8},
    "KOTAKBANK.NS":{"Net Interest Margin (%)":4.92,"Gross NPA (%)":1.50,"Net NPA (%)":0.44,"CASA Ratio (%)":46.7},
    "AXISBANK.NS":{"Net Interest Margin (%)":4.06,"Gross NPA (%)":1.43,"Net NPA (%)":0.31,"CASA Ratio (%)":42.0},
    "INDUSINDBK.NS":{"Net Interest Margin (%)":4.29,"Gross NPA (%)":2.25,"Net NPA (%)":0.68,"CASA Ratio (%)":36.4},
    "SBIN.NS":{"Net Interest Margin (%)":3.21,"Gross NPA (%)":2.24,"Net NPA (%)":0.57,"CASA Ratio (%)":41.9},
    "PNB.NS":{"Net Interest Margin (%)":3.10,"Gross NPA (%)":4.09,"Net NPA (%)":0.40,"CASA Ratio (%)":44.2},
    "BANKBARODA.NS":{"Net Interest Margin (%)":3.18,"Gross NPA (%)":2.92,"Net NPA (%)":0.68,"CASA Ratio (%)":38.0},
    "CANBK.NS":{"Net Interest Margin (%)":2.92,"Gross NPA (%)":3.73,"Net NPA (%)":0.89,"CASA Ratio (%)":32.5},
    "FEDERALBNK.NS":{"Net Interest Margin (%)":3.23,"Gross NPA (%)":2.09,"Net NPA (%)":0.60,"CASA Ratio (%)":30.4},
    "IDFCFIRSTB.NS":{"Net Interest Margin (%)":6.04,"Gross NPA (%)":2.04,"Net NPA (%)":0.60,"CASA Ratio (%)":46.9},
    "YESBANK.NS":{"Net Interest Margin (%)":2.43,"Gross NPA (%)":1.60,"Net NPA (%)":0.30,"CASA Ratio (%)":30.5},
    "AUBANK.NS":{"Net Interest Margin (%)":5.95,"Gross NPA (%)":1.84,"Net NPA (%)":0.56,"CASA Ratio (%)":33.0},
    "BAJFINANCE.NS":{"Net Interest Margin (%)":10.7,"Gross NPA (%)":1.06,"Net NPA (%)":0.44,"CASA Ratio (%)":None},
    "CHOLAFIN.NS":{"Net Interest Margin (%)":7.90,"Gross NPA (%)":3.41,"Net NPA (%)":2.55,"CASA Ratio (%)":None},
    "LICHSGFIN.NS":{"Net Interest Margin (%)":2.40,"Gross NPA (%)":3.13,"Net NPA (%)":1.95,"CASA Ratio (%)":None},
    "PNBHOUSING.NS":{"Net Interest Margin (%)":3.52,"Gross NPA (%)":1.48,"Net NPA (%)":0.72,"CASA Ratio (%)":None},
}

def compute_metrics(ticker):
    result = {}
    try:
        t = yf.Ticker(ticker); info = t.info or {}
    except: info = {}

    try:
        inc = t.financials; bal = t.balance_sheet; cfs = t.cashflow
        def _row(df, keys):
            if df is None or df.empty: return None
            for k in keys:
                for idx in df.index:
                    if k.lower() in str(idx).lower(): return df.loc[idx]
            return None
        def _lv(s):
            if s is None: return None
            try: return _f(s.iloc[-1])
            except: return None
        def _pv(s):
            if s is None: return None
            try: return _f(s.iloc[-2])
            except: return None

        if inc is not None and not inc.empty:
            inc = inc.sort_index(axis=1)
            rev_s = _row(inc,["total revenue","revenue"])
            gp_s  = _row(inc,["gross profit"])
            op_s  = _row(inc,["operating income","ebit"])
            ni_s  = _row(inc,["net income"])
            eb_s  = _row(inc,["ebitda"])
            rev=_lv(rev_s); rev_p=_pv(rev_s); gp=_lv(gp_s); op=_lv(op_s); ni=_lv(ni_s); eb=_lv(eb_s)

            if rev and rev_p and rev_p!=0: result["Revenue Growth (YoY %)"] = _div(rev-rev_p,abs(rev_p))
            if rev_s is not None and len(rev_s)>=4:
                rb = _f(rev_s.iloc[-4])
                if rev and rb and rb>0: result["Revenue CAGR (3Y %)"] = (rev/rb)**(1/3)-1
            if rev_s is not None and len(rev_s)>=6:
                rb5 = _f(rev_s.iloc[-6])
                if rev and rb5 and rb5>0: result["Revenue CAGR (5Y %)"] = (rev/rb5)**(1/5)-1
            result["Gross Margin (%)"]    = _div(gp, rev)
            result["EBITDA Margin (%)"]   = _div(eb, rev)
            result["Operating Margin (%)"]= _div(op, rev)
            result["EBIT Margin (%)"]     = _div(op, rev)
            result["Net Margin (%)"]      = _div(ni, rev)

            if cfs is not None and not cfs.empty:
                cfs = cfs.sort_index(axis=1)
                cfo_s  = _row(cfs,["total cash from operating","operating cash flow"])
                cap_s  = _row(cfs,["capital expenditures","capex"])
                cfo=_lv(cfo_s); cap=_lv(cap_s)
                if cfo and cap and rev: result["Free Cash Flow Margin (%)"] = _div(cfo+cap, rev)

    except: pass

    result["ROE (%)"]          = _f(info.get("returnOnEquity"))
    result["ROA (%)"]          = _f(info.get("returnOnAssets"))
    result["ROCE (%)"]         = _f(info.get("returnOnCapital"))
    result["ROIC (%)"]         = _f(info.get("returnOnCapital"))
    result["P/E (TTM)"]        = _f(info.get("trailingPE"))
    result["P/E (Forward)"]    = _f(info.get("forwardPE"))
    result["EV / EBITDA"]      = _f(info.get("enterpriseToEbitda"))
    result["EV / Sales"]       = _f(info.get("enterpriseToRevenue"))
    result["Price / Book"]     = _f(info.get("priceToBook"))
    result["P/B Ratio"]        = _f(info.get("priceToBook"))
    result["Beta (5Y)"]        = _f(info.get("beta"))
    result["Current Ratio"]    = _f(info.get("currentRatio"))
    result["Interest Coverage"]= _f(info.get("interestCoverage"))
    result["Dividend Yield (%)"]= _f(info.get("dividendYield"))
    result["Debt to Equity"]   = _f(info.get("debtToEquity"))

    td  = _f(info.get("totalDebt")); csh = _f(info.get("totalCash"))
    if td and csh:
        try:
            t2 = yf.Ticker(ticker); inc2 = t2.financials
            if inc2 is not None and not inc2.empty:
                eb_row = None
                for idx in inc2.index:
                    if "ebitda" in str(idx).lower(): eb_row = inc2.loc[idx]; break
                if eb_row is not None:
                    eb_val = _f(eb_row.iloc[-1])
                    result["Net Debt / EBITDA"] = _div(td-csh, eb_val)
        except: pass

    bm = BANKING_DEFAULTS.get(ticker, globals().get("BANKING_METRICS_OVERRIDE",{}).get(ticker,{}))
    for k,v in bm.items():
        if v is not None: result[k] = v

    return result

# ─────────────────────────────────────────────────────────────────────────────
# UNLISTED COMPANY SUPPORT
# ─────────────────────────────────────────────────────────────────────────────
def _compute_metrics_from_hist(hist_df):
    if hist_df is None or hist_df.empty: return {}
    h = hist_df.sort_values("FY")
    last = h.iloc[-1]; prev = h.iloc[-2] if len(h)>1 else last
    rev  = _f(last.get("Revenue",0) or 0); rev_p = _f(prev.get("Revenue",0) or 0)
    ebit = _f(last.get("EBIT",0) or 0); da = _f(last.get("DandA",0) or 0)
    capex= _f(last.get("Capex",0) or 0)
    ebitda = (ebit+da) if ebit and da else None
    return {
        "Revenue Growth (YoY %)":  _div(rev-rev_p, abs(rev_p)) if rev and rev_p and rev_p!=0 else None,
        "EBITDA Margin (%)":       _div(ebitda, rev) if ebitda else None,
        "EBIT Margin (%)":         _div(ebit, rev),
        "Operating Margin (%)":    _div(ebit, rev),
        "Free Cash Flow Margin (%)": _div((ebitda or 0) - (capex or 0), rev) if ebitda and rev else None,
    }

_is_unlisted = bool(UNLISTED_FILE and os.path.exists(str(UNLISTED_FILE)))

if _is_unlisted:
    print("  Mode: UNLISTED (file-source)")
    try:
        from unlisted_engine import UnlistedCompany
        _co = UnlistedCompany(
            files=UNLISTED_FILE if isinstance(UNLISTED_FILE, list) else [UNLISTED_FILE],
            company_name=COMPANY_NAME_OVERRIDE or "Unlisted Company",
        )
        company_metrics = _co.get_metrics()
        company_name    = _co.company_name
        _engine_sub     = _co.sub_industry
        _engine_sector  = _co.sector
        _engine_conf    = _co.detection_confidence
        _engine_method  = "unlisted_engine"
        peers           = _co.get_peers()
        _profile_extra  = _co.get_profile()
        print(f"  Sub-industry: {_engine_sub} ({_engine_conf*100:.0f}%)")
    except ImportError:
        print("  ⚠️  unlisted_engine.py not found — using HIST fallback")
        company_metrics = _compute_metrics_from_hist(_hist)
        company_name    = COMPANY_NAME_OVERRIDE or TICKER
        _engine_sub,_engine_sector,_engine_conf,_engine_method = globals().get("DETECTED_SECTOR","Unknown"),"Unknown",0.5,"hist_fallback"
        peers = []
        _profile_extra = {}

elif not _api_on:
    print("  Mode: FILE-SOURCE (API off)")
    company_metrics = _compute_metrics_from_hist(_hist)
    company_name    = COMPANY_NAME_OVERRIDE or TICKER
    try:    yf_info_main = yf.Ticker(TICKER).info or {}
    except: yf_info_main = {}
    cls = classify_ticker(TICKER, yf_info_main)
    _engine_sub     = globals().get("DETECTED_SECTOR") or cls["sub_industry"]
    _engine_sector  = cls["sector"]
    _engine_conf    = cls["confidence"]
    _engine_method  = globals().get("DETECTION_METHOD", cls["method"])
    peers = []
    _profile_extra  = {}

else:
    print("  Mode: LISTED (API)")
    try:    yf_info_main = yf.Ticker(TICKER).info or {}
    except: yf_info_main = {}
    company_name = COMPANY_NAME_OVERRIDE or yf_info_main.get("shortName") or yf_info_main.get("longName") or TICKER

    _engine_sub = None
    _is_india   = TICKER.endswith((".NS",".BO"))
    if _is_india:
        _engine_sub = get_screener_sub_industry(TICKER)
        if _engine_sub: print(f"  screener.in tag: {_engine_sub}")

    cls = classify_ticker(TICKER, yf_info_main)
    if not _engine_sub: _engine_sub = cls["sub_industry"]
    _engine_sector  = SECTOR_FROM_SUB.get(_engine_sub, cls["sector"])
    _engine_conf    = 0.98 if (_is_india and _engine_sub and _engine_sub != cls["sub_industry"]) else cls["confidence"]
    _engine_method  = cls["method"]

    company_metrics = compute_metrics(TICKER)
    peers = []
    _profile_extra  = {}

# ─────────────────────────────────────────────────────────────────────────────
# PEER SELECTION
# FIX 2: US tickers always use US_{sector} universe — never India sub-industry lists.
# ─────────────────────────────────────────────────────────────────────────────
_peer_source = "—"  # default; overwritten below
if isinstance(PEERS_OVERRIDE, (list,tuple)) and len(PEERS_OVERRIDE) > 0:
    peers = [p.upper().strip() for p in PEERS_OVERRIDE if p.upper().strip() != TICKER]
    _peer_source = "PEERS_OVERRIDE (manual)"
else:
    if not peers:
        # ── PATCH FIX 2: separate India vs USA universe resolution ────────────
        _is_india = TICKER.endswith((".NS", ".BO"))
        if _is_india:
            # India tickers: use sub-industry key directly (e.g. "Private Bank")
            _ukey = _engine_sub if _engine_sub in PEER_UNIVERSE else None
        else:
            # USA tickers: always use "US_{sector}" — never India sub-industry lists
            _ukey = f"US_{_engine_sector}" if f"US_{_engine_sector}" in PEER_UNIVERSE else None
            # Handle dedicated US sub-industry universes (Steel, Chemicals, IT Services)
            if _ukey is None and f"US_{_engine_sub}" in PEER_UNIVERSE:
                _ukey = f"US_{_engine_sub}"
        # ─────────────────────────────────────────────────────────────────────

        _candidates = [t for t in (PEER_UNIVERSE.get(_ukey, []) if _ukey else []) if t != TICKER]

        _target_mc = None
        if not _is_unlisted:
            _target_mc = _get_mc(TICKER)
        _filtered, _band = filter_by_cap(_target_mc, _candidates)

        if _filtered:
            _ranked = score_and_rank(TICKER, _filtered)
            peers = [r["ticker"] for r in _ranked[:7]]
            _peer_source = f"{'India' if _is_india else 'USA'}:{_engine_sub} (cap:{_band}, method:{_engine_method})"
        else:
            peers = _candidates[:7]
            _peer_source = f"no_score:{_engine_sub}"

print(f"  Peers: {peers}")

# ─────────────────────────────────────────────────────────────────────────────
# PEER METRICS + INDUSTRY AVERAGE
# ─────────────────────────────────────────────────────────────────────────────
METRIC_SECTIONS = get_metric_sections(_engine_sub)
ALL_METRICS = [m for sec in METRIC_SECTIONS.values() for m in sec]

peer_metrics_list = []; bad_peers = []
for p in peers:
    try:
        pm = compute_metrics(p); pm["_ticker"] = p
        peer_metrics_list.append(pm); print(f"  ✅ {p}")
    except Exception as e: bad_peers.append(p); print(f"  ⚠️  {p}: {e}")

industry_avg = {m: _mean([pm.get(m) for pm in peer_metrics_list]) for m in ALL_METRICS}

# ─────────────────────────────────────────────────────────────────────────────
# EXCEL WRITER
# ─────────────────────────────────────────────────────────────────────────────
wb = load_workbook(OUTPUT_XLSX, keep_vba=str(OUTPUT_XLSX).lower().endswith(".xlsm"))
if SHEET_NAME in wb.sheetnames:
    ws = wb[SHEET_NAME]; ws.delete_rows(1, ws.max_row)
else:
    ws = wb.create_sheet(SHEET_NAME)

F_HDR  = Font(name="Arial", size=10, bold=True)
F_LBL  = Font(name="Arial", size=10, bold=True)
F_TXT  = Font(name="Arial", size=10)
F_CONF = Font(name="Arial", size=9, color="595959")
AL = Alignment(horizontal="left",  vertical="center")
AR = Alignment(horizontal="right", vertical="center")
FILL_HDR = PatternFill("solid", fgColor="F2F2F2")
FILL_SEC = PatternFill("solid", fgColor="EBF3FB")
FILL_POS = PatternFill("solid", fgColor="E8F5E9")
FILL_NEG = PatternFill("solid", fgColor="FFEBEE")

def _set(ws, cell, value, font=None, align=None, fill=None, nf=None):
    ws[cell] = value
    if font:  ws[cell].font  = font
    if align: ws[cell].alignment = align
    if fill:  ws[cell].fill  = fill
    if nf:    ws[cell].number_format = nf

_set(ws,"A2","Company Profile",F_HDR,AL)
_market_label = "INDIA" if TICKER.endswith((".NS",".BO")) else ("UNLISTED" if _is_unlisted else "USA")
profile_rows = [
    ("Company Name",    company_name),
    ("Ticker",          TICKER),
    ("Market",          _market_label),
    ("Sub-industry",    _engine_sub),
    ("Sector (GICS)",   _engine_sector),
    ("Classification",  _engine_method),
    ("Peer confidence", f"{_engine_conf*100:.0f}%"),
    ("Peers used",      ", ".join(peers) if peers else "—"),
    ("Peers source",    _peer_source),
    ("Peers failed",    ", ".join(bad_peers) if bad_peers else "—"),
]
if _profile_extra:
    for k,v in _profile_extra.items():
        profile_rows.append((k, str(v)))

r = 4
for k,v in profile_rows:
    _set(ws,f"A{r}",k,F_LBL,AL)
    _set(ws,f"B{r}",str(v) if v is not None else "—",F_TXT,AL)
    r += 1

r += 1
hdr_row = r
_set(ws,f"A{hdr_row}","Metric",F_HDR,AL,FILL_HDR)
_set(ws,f"B{hdr_row}",f"{company_name} ({TICKER})",F_HDR,AR,FILL_HDR)
_set(ws,f"C{hdr_row}","Industry Average (Peers)",F_HDR,AR,FILL_HDR)
r += 1

for section, metrics in METRIC_SECTIONS.items():
    ws.merge_cells(f"A{r}:C{r}")
    _set(ws,f"A{r}",section,F_HDR,AL,FILL_SEC)
    r += 1
    for m in metrics:
        cv_f = _f(company_metrics.get(m))
        iv_f = _f(industry_avg.get(m))
        fill_cv = None
        if cv_f is not None and iv_f is not None:
            if m in HIGHER_IS_BETTER: fill_cv = FILL_POS if cv_f >= iv_f else FILL_NEG
            elif m in LOWER_IS_BETTER: fill_cv = FILL_POS if cv_f <= iv_f else FILL_NEG
        fmt = "0.00%" if is_pct_metric(m) else "0.00"
        _set(ws,f"A{r}",m,F_TXT,AL)
        _set(ws,f"B{r}",cv_f if cv_f is not None else "N/A",F_TXT,AR,fill_cv,fmt)
        _set(ws,f"C{r}",iv_f if iv_f is not None else "N/A",F_TXT,AR,None,fmt)
        r += 1
    r += 1

if peer_metrics_list:
    r += 1
    _set(ws,f"A{r}","Peer Selection Details",F_HDR,AL); r += 1
    _set(ws,f"A{r}","Ticker",F_HDR,AL,FILL_HDR)
    _set(ws,f"B{r}","Match",F_HDR,AR,FILL_HDR)
    _set(ws,f"C{r}","Signals used",F_HDR,AL,FILL_HDR); r += 1
    try:
        ranked_detail = score_and_rank(TICKER, peers)
        for rd in ranked_detail:
            _set(ws,f"A{r}",rd["ticker"],F_TXT,AL)
            _set(ws,f"B{r}",f"{rd['conf_pct']}%",F_TXT,AR)
            _set(ws,f"C{r}",rd["expl"],F_CONF,AL); r += 1
    except: pass

ws.column_dimensions["A"].width = 38
ws.column_dimensions["B"].width = 26
ws.column_dimensions["C"].width = 38

wb.save(OUTPUT_XLSX)

print("\n" + "="*55)
print("✅ Industry Comparison sheet written")
print(f"✅ Company     : {company_name} ({TICKER})")
print(f"✅ Sub-industry: {_engine_sub} ({_engine_conf*100:.0f}% confidence)")
print(f"✅ Peers       : {peers}")
print(f"✅ Metrics     : {len(ALL_METRICS)} across {len(METRIC_SECTIONS)} sections")
print(f"✅ File        : {OUTPUT_XLSX}")
print("="*55)


ℹ️  Industry Comparison — Ticker: RELIANCE.NS
  Mode: LISTED (API)
  screener.in tag: Oil & Gas Refining
  Peers: []

✅ Industry Comparison sheet written
✅ Company     : RELIANCE INDUSTRIES LTD (RELIANCE.NS)
✅ Sub-industry: Oil & Gas Refining (98% confidence)
✅ Peers       : []
✅ Metrics     : 24 across 6 sections
✅ File        : /Users/sharankothari/Desktop/research_platform/DCF_Output_RELIANCE.NS_INR.xlsm


# Part 9 — Final Formatting & Cleanup

In [22]:
# ── 9A: Apply number formats ─────────────────────────────────────────────────

wb = load_workbook(TEMPLATE_XLSM, keep_vba=True)
DCF_SHEET = _resolve_sheet(wb, "DCF")
COC_SHEET = _resolve_sheet(wb, "Cost of Capital")
ws_dcf    = wb[DCF_SHEET]
ws_coc    = wb[COC_SHEET]

# Use injected currency if available; fall back to suffix detection
_tk = str(globals().get("TICKER","")).upper()
_suffix_ccy_map_32 = {".NS": "INR", ".BO": "INR", ".L": "GBP", ".PA": "EUR", ".DE": "EUR", ".T": "JPY", ".HK": "HKD", ".AX": "AUD"}
_detected_ccy_32 = next((c for s, c in _suffix_ccy_map_32.items() if _tk.endswith(s)), "USD")
LOCAL_CURRENCY = globals().get("LOCAL_CURRENCY") or _detected_ccy_32
_default_sym_32 = {"INR": "₹", "USD": "$", "GBP": "£", "EUR": "€", "JPY": "¥", "HKD": "HK$", "AUD": "A$"}
SYM = globals().get("CURRENCY_SYMBOL") or _default_sym_32.get(LOCAL_CURRENCY, "$")
FMT_CCY  = f'"{SYM}"#,##0.000'   # 3dp — readable for small companies (KGI ~$4M)
FMT_PCT  = "0.00%"
FMT_NUM  = "#,##0.000"
FMT_DEC  = "0.00"
FMT_INT  = "0"
FMT_DATE = "mm/dd/yyyy"

def _apply_fmt(ws, a1_range, fmt):
    try:
        min_col2, min_row2, max_col2, max_row2 = range_boundaries(a1_range)
        for r in range(min_row2, max_row2 + 1):
            for c in range(min_col2, max_col2 + 1):
                cell = ws.cell(r, c)
                if not isinstance(cell, MergedCell):
                    cell.number_format = fmt
    except Exception: pass

# ── Step 1: Fix template rows 1-21 ───────────────────────────────────────────
# The template was saved in an India locale so built-in numFmtId=4 renders as ₹.
# Fix: set EXPLICIT currency format on financial rows, percent on ratio rows.
# This overrides the locale-based rendering.

# Financial rows (Revenue, EBIT, D&A, EBITDA, CapEx, WC) — explicit $ or ₹
_financial_rows = [6, 7, 8, 9, 10, 11, 13]  # 13 = EBITDA subtotal row
_ratio_rows     = [14, 15, 16, 17, 18]   # Revenue Growth, EBIT Margin, etc.

for _r in _financial_rows:
    for _col in ["E","F","G","H","I","J","K","L"]:
        try:
            _cell = ws_dcf[f"{_col}{_r}"]
            if not isinstance(_cell, MergedCell):
                _cell.number_format = FMT_CCY   # explicit symbol overrides locale
        except Exception: pass

for _r in _ratio_rows:
    for _col in ["E","F","G","H","I","J","K","L"]:
        try:
            _cell = ws_dcf[f"{_col}{_r}"]
            if not isinstance(_cell, MergedCell):
                _cell.number_format = FMT_PCT
        except Exception: pass

# Row 4 year headers — plain integer
_apply_fmt(ws_dcf, "E4:L4",   FMT_INT)
_apply_fmt(ws_dcf, "E22:L22", FMT_INT)

# ── Step 2: Our DCF block rows 22+ ────────────────────────────────────────────
for rng in ["E24:L31","E34:L34","E37:L37",
            "P6","P9","P13","P23","P24:P27","P29","P30"]:
    _apply_fmt(ws_dcf, rng, FMT_CCY)

# Discount period n (row 35) — plain decimal
_apply_fmt(ws_dcf, "H35:L35", FMT_DEC)
ws_dcf["P7"].number_format = FMT_DEC

# Discount rate (row 36) — percent
_apply_fmt(ws_dcf, "H36:L36", FMT_PCT)
ws_dcf["P8"].number_format  = FMT_PCT

# Distribution of value — percent
for addr in ["P17","P18","P19"]:
    if ws_dcf[addr].value is not None:
        ws_dcf[addr].number_format = FMT_PCT

# Key assumptions column R
for addr in ["R6","R10","R11","R12","R14","R15","R16","R18","R21","R22"]:
    ws_dcf[addr].number_format = FMT_PCT
for addr in ["R24","R25","R27"]:
    ws_dcf[addr].number_format = "#,##0.00"
ws_dcf["R7"].number_format = FMT_DATE
ws_dcf["R8"].number_format = FMT_DATE

# Sensitivity axis — percent labels, plain number for grid values
_apply_fmt(ws_dcf, "E44:I48", FMT_NUM)
_apply_fmt(ws_dcf, "E43:I43", FMT_PCT)
_apply_fmt(ws_dcf, "D44:D48", FMT_PCT)

# ── Step 3: Cost of Capital ────────────────────────────────────────────────────
ws_coc["C3"].number_format  = FMT_PCT
ws_coc["C4"].number_format  = "0.00"      # Beta — plain number, NOT %
ws_coc["C5"].number_format  = FMT_PCT
ws_coc["C6"].number_format  = FMT_PCT
ws_coc["C9"].number_format  = "#,##0.00"
ws_coc["C10"].number_format = "#,##0.00"
ws_coc["C11"].number_format = "#,##0.00"
ws_coc["C12"].number_format = "#,##0.00"
ws_coc["C15"].number_format = FMT_PCT
ws_coc["C16"].number_format = FMT_PCT
ws_coc["C17"].number_format = FMT_PCT
ws_coc["D20"].number_format = FMT_PCT
ws_coc["D21"].number_format = FMT_PCT
ws_coc["D22"].number_format = FMT_PCT
if ws_coc["E27"].value is not None:
    ws_coc["E27"].number_format = FMT_PCT

safe_save(wb, TEMPLATE_XLSM)
print(f"✅ Formats applied | {LOCAL_CURRENCY} ({SYM})")
print(f"   Template financial rows 6-11: {FMT_CCY}")
print(f"   Template ratio rows 14-18: {FMT_PCT}")
print(f"   DCF block rows 22+: {FMT_CCY}")


✅ Formats applied | INR (₹)
   Template financial rows 6-11: "₹"#,##0.000
   Template ratio rows 14-18: 0.00%
   DCF block rows 22+: "₹"#,##0.000


In [23]:
# ══════════════════════════════════════════════════════════════════
# AUTO-WRITER: Income Statement + Balance Sheet + Cash Flow
# Completely self-contained. No dependencies on other cells.
# ══════════════════════════════════════════════════════════════════
import math, glob, datetime
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.cell.cell import MergedCell

def _run_3stmt():
    # ── Find Excel file ───────────────────────────────────────────
    xlsm = (globals().get("TEMPLATE_XLSM") or globals().get("OUTPUT_XLSM") or "")
    if not xlsm or not Path(xlsm).exists():
        cands = sorted(glob.glob(str(Path.cwd() / "DCF_Output_*.xlsm")))
        xlsm = cands[0] if cands else ""
    if not xlsm or not Path(xlsm).exists():
        print("3-Statement writer: no Excel file found — run cell 18 first"); return

    TKUR = str(globals().get("TICKER") or "MSFT")
    SYM  = "₹" if TKUR.upper().endswith((".NS",".BO")) else "$"  # ₹ = U+20B9
    M    = 1_000_000

    # ── Get data ──────────────────────────────────────────────────
    yrs = [datetime.datetime(y,6,30) for y in [2021,2022,2023,2024,2025]]
    def mk(d):
        df = pd.DataFrame(d, index=yrs).T
        df.columns = yrs
        return df

    def norm(df, n=5):
        if df is None or (hasattr(df,"empty") and df.empty): return pd.DataFrame()
        df = df.copy()
        df.columns = pd.to_datetime(df.columns, errors="coerce")
        return df.loc[:,df.columns.notna()].sort_index(axis=1).iloc[:,-n:]

    # Safe globals fetch — avoids DataFrame boolean ambiguity
    def getdf(*keys):
        for k in keys:
            v = globals().get(k)
            if v is not None and hasattr(v,"empty") and not v.empty and len(v) >= 5:
                return v
        return None

    # ── Data loading: 3 paths — HIST(private), globals(listed), yfinance(fallback) ──
    # Path 1: Private/file-based company — HIST already built by KGI extractor
    # Path 2: Listed company — IS/BS/CF globals set by cell 10 (already normalized)
    # Path 3: yfinance direct fetch (fallback for listed companies)
    # NO MSFT hardcoded fallback — use empty frames if all else fails
    HIST_g  = globals().get("HIST")
    _api_on = globals().get("SOURCES", {}).get("api", True)
    _uf     = globals().get("unit_factor", 1_000_000.0)

    def getdf(*keys):
        for k in keys:
            v = globals().get(k)
            if v is not None and hasattr(v, "empty") and not v.empty and len(v) >= 3:
                return v
        return None

    ISg = getdf("IS", "IS_raw", "yf_is")
    BSg = getdf("BS", "BS_raw", "yf_bs")
    CFg = getdf("CF", "CF_raw", "yf_cf")

    _hist_ok = (HIST_g is not None and hasattr(HIST_g, "empty") and not HIST_g.empty)

    if _hist_ok and (not _api_on or ISg is None or BSg is None):
        # ── PATH 1: Private / file-based company ──────────────────────────────
        # HIST values are already in display units (divided by unit_factor).
        # To feed vv() which divides by unit_factor, multiply back to raw first.
        H = HIST_g.sort_values("FY").copy()
        import datetime as _dt2
        _cols2 = [_dt2.datetime(int(y), 12, 31) for y in H["FY"].tolist()]

        def _hist_ser2(col):
            if col not in H.columns: return None
            df = pd.DataFrame({col: H[col].values}, index=_cols2).T
            df.columns = _cols2
            return df

        def _hs2(col, sign=1):
            s2 = _hist_ser2(col)
            if s2 is None: return None
            import numpy as _np2
            if s2.isnull().all().all(): return None  # skip all-NaN
            return s2 * sign * _uf  # multiply back to raw so vv()/unit_factor = display value

        ISr = pd.concat([f for f in [
            _hs2("Revenue"),        _hs2("COGS", -1),       _hs2("GrossProfit"),
            _hs2("OpEx", -1),       _hs2("DandA", -1),      _hs2("EBIT"),
            _hs2("InterestExp", -1),_hs2("TaxExp", -1),     _hs2("NetIncome"),
        ] if f is not None]).rename(index={
            "Revenue": "Total Revenue", "COGS": "Cost Of Revenue",
            "GrossProfit": "Gross Profit", "OpEx": "Operating Expense",
            "DandA": "Reconciled Depreciation", "EBIT": "Operating Income",
            "InterestExp": "Interest Expense", "TaxExp": "Tax Provision",
            "NetIncome": "Net Income",
        }) if _hs2("Revenue") is not None else pd.DataFrame()
        # Deduplicate index — _hs2 can return same row name via different HIST cols
        if not ISr.empty:
            ISr = ISr[~ISr.index.duplicated(keep="first")]

        BSr = pd.concat([f for f in [
            _hs2("Cash"),    _hs2("AR"),        _hs2("Inventory"),
            _hs2("TCA"),     _hs2("TotalAssets"), _hs2("AP"),
            _hs2("ShortTermDebt"), _hs2("TCL"), _hs2("LTDebt"),
            _hs2("TotalLiab"), _hs2("RetainedEarnings"), _hs2("TotalEquity"),
            _hs2("Goodwill"), _hs2("PPE"),
        ] if f is not None]).rename(index={
            "Cash": "Cash And Cash Equivalents", "AR": "Accounts Receivable",
            "TCA": "Current Assets", "TotalAssets": "Total Assets",
            "AP": "Accounts Payable", "ShortTermDebt": "Current Debt",
            "TCL": "Current Liabilities", "LTDebt": "Long Term Debt",
            "TotalLiab": "Total Liabilities Net Minority Interest",
            "RetainedEarnings": "Retained Earnings", "TotalEquity": "Stockholders Equity",
            "Goodwill": "Goodwill", "PPE": "Net Ppe",
        }) if _hs2("TotalAssets") is not None else pd.DataFrame()
        if not BSr.empty:
            BSr = BSr[~BSr.index.duplicated(keep="first")]

        CFr = pd.concat([f for f in [
            _hs2("CFO"),  _hs2("Capex", -1), _hs2("CFI"),
            _hs2("CFF"),  _hs2("EndCash"),    _hs2("BegCash"),  _hs2("NetIncome"),
        ] if f is not None]).rename(index={
            "CFO": "Operating Cash Flow", "Capex": "Capital Expenditure",
            "CFI": "Investing Cash Flow", "CFF": "Financing Cash Flow",
            "EndCash": "End Cash Position", "BegCash": "Beginning Cash Position",
            "NetIncome": "Net Income",
        }) if _hs2("CFO") is not None else pd.DataFrame()
        if not CFr.empty:
            CFr = CFr[~CFr.index.duplicated(keep="first")]
        src = f"HIST private ({len(H)} years)"

    elif ISg is not None and BSg is not None and CFg is not None:
        # ── PATH 2: Listed company — globals already normalized by cell 10 ────
        # IS/BS/CF are in display units already. vv() will divide by unit_factor.
        # So we must multiply by unit_factor to get raw values for vv() to work.
        def _rescale(df):
            if df is None or df.empty: return pd.DataFrame()
            df2 = norm(df.copy())
            if df2.empty: return pd.DataFrame()
            for _rc in df2.columns:
                df2[_rc] = pd.to_numeric(df2[_rc], errors="coerce") * _uf
            return df2
        ISr = _rescale(ISg)
        BSr = _rescale(BSg)
        CFr = _rescale(CFg)
        src = "globals (rescaled)"

    else:
        # ── PATH 3: Fetch fresh from yfinance ──────────────────────────────────
        # yfinance returns raw dollars. vv() divides by unit_factor → display units.
        try:
            import yfinance as _yf3
            _t3 = _yf3.Ticker(TKUR)
            ISr = norm(_t3.financials)
            BSr = norm(_t3.balance_sheet)
            CFr = norm(_t3.cashflow)
            if ISr.empty: raise ValueError("empty")
            src = "yfinance (raw)"
        except Exception as _e3:
            print(f"   ⚠️  yfinance failed: {_e3} — trying HIST fallback")
            ISr = BSr = CFr = pd.DataFrame()
            src = "empty (no data)"
    # Fallback: if ISr still empty but HIST exists, rebuild from HIST
    if (ISr is None or (hasattr(ISr,"empty") and ISr.empty)) and _hist_ok:
        H = HIST_g.sort_values("FY").copy()
        import datetime as _dt3
        _cols3 = [_dt3.datetime(int(y), 12, 31) for y in H["FY"].tolist()]
        def _hs3(col, sign=1):
            if col not in H.columns: return None
            vals = H[col].values
            import numpy as _np3
            if all(_np3.isnan(float(v)) if v is not None else True for v in vals): return None
            df = pd.DataFrame({col: vals}, index=_cols3).T
            df.columns = _cols3
            return df * sign * _uf
        _concat_safe = lambda cols, rename: (
            pd.concat([f for f in [_hs3(c) for c in cols] if f is not None]).rename(index=dict(zip(cols, rename)))
            if any(_hs3(c) is not None for c in cols) else pd.DataFrame()
        )
        ISr = _concat_safe(
            ["Revenue","COGS","GrossProfit","OpEx","DandA","EBIT","InterestExp","TaxExp","NetIncome"],
            ["Total Revenue","Cost Of Revenue","Gross Profit","Operating Expense",
             "Reconciled Depreciation","Operating Income","Interest Expense","Tax Provision","Net Income"])
        BSr = _concat_safe(
            ["Cash","AR","Inventory","TCA","TotalAssets","AP","ShortTermDebt","TCL","LTDebt","TotalLiab","RetainedEarnings","TotalEquity","Goodwill","PPE"],
            ["Cash And Cash Equivalents","Accounts Receivable","Inventory","Current Assets","Total Assets",
             "Accounts Payable","Current Debt","Current Liabilities","Long Term Debt",
             "Total Liabilities Net Minority Interest","Retained Earnings","Stockholders Equity","Goodwill","Net Ppe"])
        CFr = _concat_safe(
            ["CFO","Capex","CFI","CFF","EndCash","BegCash","NetIncome"],
            ["Operating Cash Flow","Capital Expenditure","Investing Cash Flow","Financing Cash Flow",
             "End Cash Position","Beginning Cash Position","Net Income"])
        # Deduplicate all three frames (concat can create dupes if HIST cols share labels)
        for _fr in [ISr, BSr, CFr]:
            if not _fr.empty: _fr = _fr[~_fr.index.duplicated(keep="first")]
        src = f"HIST fallback ({len(H)} years)"
        print(f"   3-stmt using HIST fallback")

    print(f"   3-stmt data source: {src}")
    ac  = sorted(set(list(ISr.columns)+list(BSr.columns)+list(CFr.columns)))[-5:]
    ISr = ISr.reindex(columns=ac)
    BSr = BSr.reindex(columns=ac)
    CFr = CFr.reindex(columns=ac)
    hdrs= ["FY"+str(c.year) for c in ac]
    n   = len(hdrs)

    # ── Helpers ───────────────────────────────────────────────────
    def gv(df, *keys):
        if df is None or df.empty: return None
        for k in keys:
            hits = [idx for idx in df.index if k.lower() in str(idx).lower()]
            if hits: return df.loc[hits[0]]
        return None

    def vv(s, j):
        if s is None: return None
        try:
            v = s.values[j]
            if v is None: return None
            fv = float(v)
            if math.isnan(fv) or math.isinf(fv): return None
            _uf = globals().get("unit_factor", 1_000_000.0)
            if _uf == 0: return None
            return fv / _uf
        except: return None

    def dv(a, b):
        try:
            if a is None or b is None or b == 0: return None
            return float(a) / float(b)
        except: return None

    # ── Styles ────────────────────────────────────────────────────
    HD,SM,SB,MT = "1F3864","2F5496","D6E4F7","F5F5F5"
    def ff(bold=False,italic=False,color="000000",sz=10):
        return Font(name="Calibri",size=sz,bold=bold,italic=italic,color=color)
    def ffl(h): return PatternFill("solid",fgColor=h)
    def fal(h="left",v="center"): return Alignment(horizontal=h,vertical=v,wrap_text=True)
    def fbd(): return Border(bottom=Side(style="thin"))

    # ── Sheet writer ──────────────────────────────────────────────
    def ws_write(wb, title, secs, col_hdrs, sym):
        t = title[:31]
        if t in wb.sheetnames: del wb[t]
        ws = wb.create_sheet(t)
        _cu_lbl = str(globals().get("CURRENCY_UNITS","Millions")).title()
        ws.cell(1,1).value = title+"  |  "+("INR" if sym=="Rs" else "USD")+"  |  In "+_cu_lbl
        ws.cell(1,1).font  = ff(bold=True,color="FFFFFF",sz=11)
        ws.cell(1,1).fill  = ffl(HD)
        ws.cell(2,1).value = "Line Item"
        ws.cell(2,1).font  = ff(bold=True)
        ws.cell(2,1).fill  = ffl(SB)
        for ci,h in enumerate(col_hdrs,start=2):
            ws.cell(2,ci).value     = h
            ws.cell(2,ci).font      = ff(bold=True)
            ws.cell(2,ci).fill      = ffl(SB)
            ws.cell(2,ci).alignment = fal("center")
        row = 3
        for sec in secs:
            tp = sec.get("type","")
            if tp == "spacer": row += 1; continue
            if tp == "header":
                ws.cell(row,1).value = sec["label"]
                ws.cell(row,1).font  = ff(bold=True,color="FFFFFF")
                ws.cell(row,1).fill  = ffl(SM)
                for ci in range(2,2+len(col_hdrs)): ws.cell(row,ci).fill = ffl(SM)
                row += 1; continue
            vals   = sec.get("values",[])
            fmt    = sec.get("fmt","currency")
            is_sub = (tp=="subtotal")
            is_met = (tp=="metric")
            ws.cell(row,1).value     = sec.get("label","")
            ws.cell(row,1).font      = ff(bold=True) if is_sub else (ff(italic=True,color="555555",sz=9) if is_met else ff())
            ws.cell(row,1).alignment = fal()
            for ci,v in enumerate(vals or [],start=2):
                cl = ws.cell(row,ci)
                if v is None:
                    cl.value = None
                elif isinstance(v,(int,float)):
                    cl.value = v
                    if   fmt=="currency": cl.number_format = '"'+sym+'"#,##0.000_);("'+sym+'"#,##0.000)'
                    elif fmt=="percent":  cl.number_format = "0.0%"
                    else:                 cl.number_format = "#,##0.0x"
                else:
                    cl.value = v
                if is_sub: cl.font=ff(bold=True); cl.fill=ffl(MT); cl.border=fbd()
                elif is_met: cl.font=ff(italic=True,color="555555",sz=9)
            row += 1
        ws.column_dimensions["A"].width = 38
        for ci in range(2,2+len(col_hdrs)):
            ws.column_dimensions[get_column_letter(ci)].width = 14
        return ws

    # ── IS data ───────────────────────────────────────────────────
    rev  = gv(ISr,"Total Revenue","Revenue")
    cogs = gv(ISr,"Cost Of Revenue","Cost of Goods")
    rd   = gv(ISr,"Research Development","Research And Development")
    opex = gv(ISr,"Operating Expense","Total Operating Expenses")
    da   = gv(ISr,"Reconciled Depreciation","Depreciation And Amortization")
    ebit = gv(ISr,"Operating Income","Ebit")
    ii   = gv(ISr,"Interest Income")
    ie   = gv(ISr,"Interest Expense")
    pre  = gv(ISr,"Pretax Income")
    tax  = gv(ISr,"Tax Provision")
    ni   = gv(ISr,"Net Income")
    shr  = gv(ISr,"Diluted Average Shares")
    rv   = [vv(rev,j)  for j in range(n)]
    dav  = [vv(da,j)   for j in range(n)]
    evv  = [vv(ebit,j) for j in range(n)]
    niv  = [vv(ni,j)   for j in range(n)]
    prv  = [vv(pre,j)  for j in range(n)]
    txv  = [vv(tax,j)  for j in range(n)]
    gp   = [(rv[j] or 0) - abs(vv(cogs,j) or 0)          for j in range(n)]
    edav = [(evv[j] or 0) + (dav[j] or 0)                 for j in range(n)]

    is_secs = [
        {"type":"header","label":"REVENUE"},
        {"type":"item","label":"  Gross Revenue","values":rv,"fmt":"currency"},
        {"type":"item","label":"  Cost of Revenue","values":[vv(cogs,j) for j in range(n)],"fmt":"currency"},
        {"type":"subtotal","label":"Gross Profit","values":gp,"fmt":"currency"},
        {"type":"metric","label":"  Gross Margin %","values":[dv(gp[j],rv[j]) for j in range(n)],"fmt":"percent"},
        {"type":"spacer"},
        {"type":"header","label":"OPERATING EXPENSES"},
        {"type":"item","label":"  Selling, General & Administrative","values":[vv(gv(ISr,"Selling General Administrative","General And Administrative"),j) for j in range(n)],"fmt":"currency"},
        {"type":"item","label":"  Research & Development","values":[vv(rd,j) for j in range(n)],"fmt":"currency"},
        {"type":"item","label":"  Other Operating Expenses","values":[vv(opex,j) for j in range(n)],"fmt":"currency"},
        {"type":"subtotal","label":"EBITDA","values":edav,"fmt":"currency"},
        {"type":"metric","label":"  EBITDA Margin %","values":[dv(edav[j],rv[j]) for j in range(n)],"fmt":"percent"},
        {"type":"spacer"},
        {"type":"header","label":"DEPRECIATION & AMORTIZATION"},
        {"type":"item","label":"  Total D&A","values":dav,"fmt":"currency"},
        {"type":"subtotal","label":"EBIT (Operating Income)","values":evv,"fmt":"currency"},
        {"type":"metric","label":"  EBIT Margin %","values":[dv(evv[j],rv[j]) for j in range(n)],"fmt":"percent"},
        {"type":"spacer"},
        {"type":"header","label":"NON-OPERATING & TAX"},
        {"type":"item","label":"  Interest Income","values":[vv(ii,j) for j in range(n)],"fmt":"currency"},
        {"type":"item","label":"  Interest Expense","values":[vv(ie,j) for j in range(n)],"fmt":"currency"},
        {"type":"subtotal","label":"    EBT (Pre-Tax Income)","values":prv,"fmt":"currency"},
        {"type":"item","label":"  Income Tax Expense","values":txv,"fmt":"currency"},
        {"type":"metric","label":"  Effective Tax Rate %","values":[dv(abs(txv[j] or 0),abs(prv[j] or 1)) for j in range(n)],"fmt":"percent"},
        {"type":"subtotal","label":"NET INCOME","values":niv,"fmt":"currency"},
        {"type":"metric","label":"  Net Margin %","values":[dv(niv[j],rv[j]) for j in range(n)],"fmt":"percent"},
        {"type":"spacer"},
        {"type":"header","label":"PER SHARE"},
        {"type":"item","label":"  Diluted Shares (M)","values":[vv(shr,j) for j in range(n)],"fmt":"decimal"},
        {"type":"metric","label":"  EPS","values":[dv(niv[j],vv(shr,j)) for j in range(n)],"fmt":"decimal"},
    ]

    # ── BS data ───────────────────────────────────────────────────
    cv   = [vv(gv(BSr,"Cash And Cash Equivalents"),j) for j in range(n)]
    arv  = [vv(gv(BSr,"Accounts Receivable"),j)       for j in range(n)]
    ivv  = [vv(gv(BSr,"Inventory"),j)                 for j in range(n)]
    ppv  = [vv(gv(BSr,"Prepaid Assets","Prepaid"),j)  for j in range(n)]
    tcav = [vv(gv(BSr,"Current Assets"),j)            for j in range(n)]
    ppev = [vv(gv(BSr,"Net Ppe","Net Property"),j)    for j in range(n)]
    gwv  = [vv(gv(BSr,"Goodwill"),j)                  for j in range(n)]
    tav  = [vv(gv(BSr,"Total Assets"),j)              for j in range(n)]
    apv  = [vv(gv(BSr,"Accounts Payable"),j)          for j in range(n)]
    stdv = [vv(gv(BSr,"Current Debt"),j)              for j in range(n)]
    tclv = [vv(gv(BSr,"Current Liabilities"),j)       for j in range(n)]
    ltdv = [vv(gv(BSr,"Long Term Debt"),j)            for j in range(n)]
    tlv  = [vv(gv(BSr,"Total Liabilities Net Minority Interest","Total Liabilities"),j) for j in range(n)]
    eqv  = [vv(gv(BSr,"Stockholders Equity"),j)       for j in range(n)]
    # Safe computed fields
    ocav  = [(tcav[j] or 0)-(cv[j] or 0)-(arv[j] or 0)-(ivv[j] or 0)-(ppv[j] or 0) for j in range(n)]
    tncav = [(tav[j] or 0)-(tcav[j] or 0)                                             for j in range(n)]
    oiv   = [tncav[j]-(ppev[j] or 0)-(gwv[j] or 0)                                   for j in range(n)]
    nclv  = [(tlv[j] or 0)-(tclv[j] or 0)                                             for j in range(n)]
    oclv  = [(tclv[j] or 0)-(apv[j] or 0)-(stdv[j] or 0)                             for j in range(n)]
    tleq  = [(tlv[j] or 0)+(eqv[j] or 0)                                              for j in range(n)]

    bs_secs = [
        {"type":"header","label":"CURRENT ASSETS"},
        {"type":"item","label":"      Cash & Cash Equivalents","values":cv,"fmt":"currency"},
        {"type":"item","label":"      Accounts Receivable","values":arv,"fmt":"currency"},
        {"type":"item","label":"      Inventory","values":ivv,"fmt":"currency"},
        {"type":"item","label":"      Prepaid Expenses","values":ppv,"fmt":"currency"},
        {"type":"item","label":"      Other Current Assets","values":ocav,"fmt":"currency"},
        {"type":"subtotal","label":"Total Current Assets","values":tcav,"fmt":"currency"},
        {"type":"spacer"},
        {"type":"header","label":"NON-CURRENT ASSETS"},
        {"type":"item","label":"      Net PP&E","values":ppev,"fmt":"currency"},
        {"type":"item","label":"      Goodwill","values":gwv,"fmt":"currency"},
        {"type":"item","label":"      Other Intangible Assets","values":oiv,"fmt":"currency"},
        {"type":"subtotal","label":"Total Non-Current Assets","values":tncav,"fmt":"currency"},
        {"type":"subtotal","label":"TOTAL ASSETS","values":tav,"fmt":"currency"},
        {"type":"spacer"},
        {"type":"header","label":"CURRENT LIABILITIES"},
        {"type":"item","label":"      Accounts Payable","values":apv,"fmt":"currency"},
        {"type":"item","label":"      Short-Term Debt","values":stdv,"fmt":"currency"},
        {"type":"item","label":"      Other Current Liabilities","values":oclv,"fmt":"currency"},
        {"type":"subtotal","label":"Total Current Liabilities","values":tclv,"fmt":"currency"},
        {"type":"spacer"},
        {"type":"header","label":"NON-CURRENT LIABILITIES"},
        {"type":"item","label":"      Long-Term Debt","values":ltdv,"fmt":"currency"},
        {"type":"subtotal","label":"Total Non-Current Liabilities","values":nclv,"fmt":"currency"},
        {"type":"subtotal","label":"TOTAL LIABILITIES","values":tlv,"fmt":"currency"},
        {"type":"spacer"},
        {"type":"header","label":"SHAREHOLDERS EQUITY"},
        {"type":"item","label":"      Retained Earnings","values":[vv(gv(BSr,"Retained Earnings"),j) for j in range(n)],"fmt":"currency"},
        {"type":"item","label":"      Total Equity","values":eqv,"fmt":"currency"},
        {"type":"subtotal","label":"TOTAL LIABILITIES + EQUITY","values":tleq,"fmt":"currency"},
        {"type":"metric","label":"  Balance Check (must=0)","values":[(tav[j] or 0)-tleq[j] for j in range(n)],"fmt":"currency"},
        {"type":"spacer"},
        {"type":"header","label":"WORKING CAPITAL METRICS"},
        {"type":"item","label":"      Net Working Capital","values":[(tcav[j] or 0)-(tclv[j] or 0) for j in range(n)],"fmt":"currency"},
        {"type":"metric","label":"      DSO (days)","values":[dv(arv[j] or 0, rv[j] or 1)*365 if rv[j] else None for j in range(n)],"fmt":"decimal"},
        {"type":"metric","label":"      DIO (days)","values":[dv(abs(ivv[j] or 0),abs(vv(cogs,j) or 1))*365 if vv(cogs,j) else None for j in range(n)],"fmt":"decimal"},
        {"type":"metric","label":"      DPO (days)","values":[dv(abs(apv[j] or 0),abs(vv(cogs,j) or 1))*365 if vv(cogs,j) else None for j in range(n)],"fmt":"decimal"},
        {"type":"metric","label":"      D/E Ratio","values":[dv((stdv[j] or 0)+(ltdv[j] or 0),(eqv[j] or 1)) for j in range(n)],"fmt":"decimal"},
    ]

    # ── CF data ───────────────────────────────────────────────────
    cfov = [vv(gv(CFr,"Operating Cash Flow"),j)             for j in range(n)]
    cpxv = [vv(gv(CFr,"Capital Expenditure"),j)             for j in range(n)]
    cfiv = [vv(gv(CFr,"Investing Cash Flow"),j)             for j in range(n)]
    cffv = [vv(gv(CFr,"Financing Cash Flow"),j)             for j in range(n)]
    endv = [vv(gv(CFr,"End Cash Position"),j)               for j in range(n)]
    begv = [vv(gv(CFr,"Beginning Cash Position"),j)         for j in range(n)]
    divv = [vv(gv(CFr,"Payment Of Dividends"),j)            for j in range(n)]
    bbv  = [vv(gv(CFr,"Repurchase Of Capital Stock"),j)     for j in range(n)]
    # NI for CF — use IS ni if not in CF
    _ni_cf_s = gv(CFr,"Net Income")
    niCF = [vv(_ni_cf_s if _ni_cf_s is not None else ni, j) for j in range(n)]
    daCF = [vv(da, j) for j in range(n)]
    # WC deltas — safe with (x or 0)
    ars = gv(BSr,"Accounts Receivable")
    ivs = gv(BSr,"Inventory")
    aps = gv(BSr,"Accounts Payable")
    ard = [-((vv(ars,j) or 0)-(vv(ars,j-1) or 0)) if j>0 else (vv(gv(CFr,"Change In Receivables"),j) or 0) for j in range(n)]
    ind = [-((vv(ivs,j) or 0)-(vv(ivs,j-1) or 0)) if j>0 else (vv(gv(CFr,"Change In Inventory"),j) or 0)   for j in range(n)]
    apd = [ ((vv(aps,j) or 0)-(vv(aps,j-1) or 0)) if j>0 else (vv(gv(CFr,"Change In Payable"),j) or 0)     for j in range(n)]
    oov  = [round((cfov[j] or 0)-(niCF[j] or 0)-(daCF[j] or 0)-ard[j]-ind[j]-apd[j], 2) for j in range(n)]
    fcfv = [(cfov[j] or 0)+(cpxv[j] or 0) for j in range(n)]
    oinv = [(cfiv[j] or 0)-(cpxv[j] or 0) for j in range(n)]
    ofiv = [(cffv[j] or 0)-(divv[j] or 0)-(bbv[j] or 0) for j in range(n)]

    cf_secs = [
        {"type":"header","label":"OPERATING ACTIVITIES"},
        {"type":"item","label":"      Net Income","values":niCF,"fmt":"currency"},
        {"type":"item","label":"      Add: D&A","values":daCF,"fmt":"currency"},
        {"type":"item","label":"      Delta Accounts Receivable","values":ard,"fmt":"currency"},
        {"type":"item","label":"      Delta Inventory","values":ind,"fmt":"currency"},
        {"type":"item","label":"      Delta Accounts Payable","values":apd,"fmt":"currency"},
        {"type":"item","label":"      Other Working Capital","values":[None]*n,"fmt":"currency"},
        {"type":"item","label":"      Other Operating Items","values":oov,"fmt":"currency"},
        {"type":"subtotal","label":"Cash from Operations (CFO)","values":cfov,"fmt":"currency"},
        {"type":"metric","label":"  CFO Margin %","values":[dv(cfov[j],rv[j]) for j in range(n)],"fmt":"percent"},
        {"type":"spacer"},
        {"type":"header","label":"INVESTING ACTIVITIES"},
        {"type":"item","label":"      Capital Expenditure (CapEx)","values":cpxv,"fmt":"currency"},
        {"type":"item","label":"      Other Investing Activities","values":oinv,"fmt":"currency"},
        {"type":"subtotal","label":"Cash from Investing (CFI)","values":cfiv,"fmt":"currency"},
        {"type":"spacer"},
        {"type":"header","label":"FINANCING ACTIVITIES"},
        {"type":"item","label":"      Debt Raised","values":[None]*n,"fmt":"currency"},
        {"type":"item","label":"      Debt Repaid","values":[None]*n,"fmt":"currency"},
        {"type":"item","label":"      Dividends Paid","values":divv,"fmt":"currency"},
        {"type":"item","label":"      Share Buybacks","values":bbv,"fmt":"currency"},
        {"type":"item","label":"      Other Financing","values":ofiv,"fmt":"currency"},
        {"type":"subtotal","label":"Cash from Financing (CFF)","values":cffv,"fmt":"currency"},
        {"type":"spacer"},
        {"type":"header","label":"CASH RECONCILIATION"},
        {"type":"item","label":"      Beginning Cash","values":begv,"fmt":"currency"},
        {"type":"item","label":"      Ending Cash","values":endv,"fmt":"currency"},
        {"type":"spacer"},
        {"type":"header","label":"FREE CASH FLOW"},
        {"type":"subtotal","label":"Free Cash Flow (CFO + CapEx)","values":fcfv,"fmt":"currency"},
        {"type":"metric","label":"  FCF Margin %","values":[dv(fcfv[j],rv[j]) for j in range(n)],"fmt":"percent"},
    ]

    # ── Write sheets ──────────────────────────────────────────────
    wb  = load_workbook(xlsm, keep_vba=str(xlsm).lower().endswith(".xlsm"))
    IS2 = ws_write(wb, "Income statement", is_secs, hdrs, SYM)
    BS2 = ws_write(wb, "Balance Sheet",    bs_secs, hdrs, SYM)
    CF2 = ws_write(wb, "Cash Flow",        cf_secs, hdrs, SYM)

    # ── Formulas ──────────────────────────────────────────────────
    ISN = "Income statement"
    dc  = [get_column_letter(ci) for ci in range(2,10)
           if str(IS2.cell(2,ci).value or "").startswith("FY")]
    if not dc: dc = ["B","C","D","E"]
    dc  = dc[:n]   # cap to data length — prevents IndexError when sheet has stale extra FY cols
    pv  = {c: (dc[i-1] if i > 0 else None) for i, c in enumerate(dc)}

    def fw(ws, c, r, v):
        cl = ws[str(c)+str(r)]
        if not isinstance(cl, MergedCell): cl.value = v

    def n0(x): return float(x) if isinstance(x, (int,float)) else 0.0

    # IS formulas
    for c in dc:
        fw(IS2,c, 6, "="+c+"4-"+c+"5")
        fw(IS2,c, 7, '=IFERROR('+c+'6/'+c+'4,"")')
        fw(IS2,c,13, "="+c+"6-IF(ISNUMBER("+c+"10),"+c+"10,0)-"+c+"11-"+c+"12")
        fw(IS2,c,14, '=IFERROR('+c+'13/'+c+'4,"")')
        fw(IS2,c,18, "="+c+"13-"+c+"17")
        fw(IS2,c,19, '=IFERROR('+c+'18/'+c+'4,"")')
        fw(IS2,c,24, "="+c+"18+IF(ISNUMBER("+c+"22),"+c+"22,0)-IF(ISNUMBER("+c+"23),"+c+"23,0)")
        fw(IS2,c,26, '=IFERROR('+c+'25/'+c+'24,"")')
        fw(IS2,c,27, "="+c+"24-"+c+"25")
        fw(IS2,c,28, '=IFERROR('+c+'27/'+c+'4,"")')
        fw(IS2,c,32, '=IFERROR('+c+'27/'+c+'31,"")')
        # Cross-sheet links
        fw(BS2,c, 4, "='Cash Flow'!"+c+"29")
        fw(CF2,c, 4, "='"+ISN+"'!"+c+"27")
        fw(CF2,c, 5, "='"+ISN+"'!"+c+"17")

    # BS + CF formulas per column
    for j, c in enumerate(dc):
        pj = dc.index(pv[c]) if pv[c] else None

        # BS data anchors — write all available values directly
        if gwv[j]:  fw(BS2,c,13, gwv[j])
        if ppev[j]: fw(BS2,c,12, ppev[j])  # PP&E
        # Other intangibles = Total NCA - PP&E - Goodwill (may be negative plug)
        if tav[j] and tcav[j]:
            _nca = (tav[j] or 0) - (tcav[j] or 0)
            _oth = _nca - (ppev[j] or 0) - (gwv[j] or 0)
            fw(BS2,c,14, round(_oth, 4))
        if tcav[j]:
            fw(BS2,c, 8, round(tcav[j]-(cv[j] or 0)-(arv[j] or 0)-(ivv[j] or 0)-(ppv[j] or 0), 4))
        if tlv[j] and tclv[j]: fw(BS2,c,26, round(tlv[j]-tclv[j], 4))
        if eqv[j]:  fw(BS2,c,31, eqv[j])
        # Write Balance Check as live formula (not hardcoded)
        fw(BS2,c,33, f"={c}16-{c}32")

        # BS subtotal formulas
        fw(BS2,c, 9, "=SUM("+c+"4:"+c+"8)")
        fw(BS2,c,15, "="+c+"12+"+c+"13+"+c+"14")
        fw(BS2,c,16, "="+c+"9+"+c+"15")
        fw(BS2,c,22, "="+c+"19+"+c+"20+"+c+"21")
        fw(BS2,c,27, "="+c+"22+"+c+"26")
        fw(BS2,c,32, "="+c+"27+"+c+"31")

        # CF: OtherOps plug so SUM(R4:R10) = CFO
        # *** ALL arithmetic uses (x or 0) — no None crashes ***
        ar_curr  = arv[j]  or 0
        ar_prev  = arv[pj] or 0 if pj is not None else ar_curr
        iv_curr  = ivv[j]  or 0
        iv_prev  = ivv[pj] or 0 if pj is not None else iv_curr
        ap_curr  = apv[j]  or 0
        ap_prev  = apv[pj] or 0 if pj is not None else ap_curr
        dar  = -(ar_curr - ar_prev)
        din  = -(iv_curr - iv_prev)
        dap  =   ap_curr - ap_prev
        ni2  = niv[j]  or 0
        da2  = abs(dav[j] or 0)
        ow2  = n0(CF2[c+"9"].value)
        cfo2 = cfov[j] or 0
        fw(CF2,c,10, round(cfo2 - ni2 - da2 - dar - din - dap - ow2, 0))
        fw(CF2,c,11, "=SUM("+c+"4:"+c+"10)")
        fw(CF2,c,12, '=IFERROR('+c+'11/\''+ISN+'\'!'+c+'4,"")')

        # CF investing
        fw(CF2,c,15, cpxv[j] if cpxv[j] is not None else 0)
        fw(CF2,c,16, round((cfiv[j] or 0)-(cpxv[j] or 0), 2))
        fw(CF2,c,17, "="+c+"15+"+c+"16")

        # CF financing plug: sum Debt Raised(20), Debt Repaid(21), Dividends(22), Buybacks(23)
        det = sum(n0(CF2[c+str(r)].value) for r in [20,21,22,23])
        fw(CF2,c,24, round((cffv[j] or 0)-det, 2))
        fw(CF2,c,25, "="+c+"20+"+c+"21+"+c+"22+"+c+"23+"+c+"24")

        # CF cash reconciliation
        fw(CF2,c,29, endv[j] if endv[j] is not None else 0)
        if pv[c]:
            fw(CF2,c,28, "="+pv[c]+"29")
        else:
            fw(CF2,c,28, begv[j] if begv[j] is not None else 0)

        # CF FCF
        fw(CF2,c,32, "="+c+"11+"+c+"15")
        fw(CF2,c,33, '=IFERROR('+c+'32/\''+ISN+'\'!'+c+'4,"")')

    # ── Post-write: update DCF sheet live links now that IS/BS/CF exist ────
    try:
        _dcf_sn = next((s for s in wb.sheetnames if s.strip().lower() == 'dcf'), None)
        _bs_sn  = next((s for s in wb.sheetnames if 'balance sheet' in s.strip().lower()), None)
        _is_sn  = next((s for s in wb.sheetnames if 'income' in s.strip().lower()), None)
        if _dcf_sn and dc:
            _ws_dcf = wb[_dcf_sn]
            _last_dc = dc[-1]  # last data column = most recent year
            # P27 = Cash from Balance Sheet latest year (row 4 = Cash row)
            if _bs_sn:
                _ws_dcf['P27'] = f"='{_bs_sn}'!{_last_dc}4"
            # R25: private companies use hardcoded value already written in cell 21
            # (read from source files, fallback 1.0) — do NOT overwrite with a formula
    except Exception as _lnk_e:
        print(f'   ⚠️  DCF live link update failed: {_lnk_e}')

    wb.save(xlsm)
    print("SUCCESS: IS + BS + CF written ("+src+") | "+str(Path(xlsm).name))
    print("Sheets: "+str(wb.sheetnames))

_run_3stmt()


   3-stmt using HIST fallback
   3-stmt data source: HIST fallback (4 years)


/var/folders/5k/x_y2jtd54tx16f_my1zkkhk80000gn/T/ipykernel_40594/3272129832.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df.columns = pd.to_datetime(df.columns, errors="coerce")
/var/folders/5k/x_y2jtd54tx16f_my1zkkhk80000gn/T/ipykernel_40594/3272129832.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df.columns = pd.to_datetime(df.columns, errors="coerce")
/var/folders/5k/x_y2jtd54tx16f_my1zkkhk80000gn/T/ipykernel_40594/3272129832.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df.columns = pd.to_datetime(df.columns, errors="coerce")


SUCCESS: IS + BS + CF written (HIST fallback (4 years)) | DCF_Output_RELIANCE.NS_INR.xlsm
Sheets: ['DCF - Original', 'DCF ', 'Cost of Capital', 'Industry Comparison', 'Income statement', 'Balance Sheet', 'Cash Flow']


In [24]:
# ── 9B: Remove unwanted sheets and strip external links ──────────────────────

wb = load_workbook(TEMPLATE_XLSM, keep_vba=True)

# Remove Paste_* and Summary sheets
_unwanted_patterns = ["paste_", "paste "]
_unwanted_names = {"summary", "dcf - original", "dcf-original", "dcf original"}
_to_delete = []
for name in wb.sheetnames:
    n = " ".join(name.strip().lower().split())
    if any(n.startswith(p) for p in _unwanted_patterns) or n in _unwanted_names:
        _to_delete.append(name)

for name in _to_delete:
    del wb[name]

if _to_delete:
    print(f"✅ Removed sheets: {_to_delete}")

# Strip any remaining external links
wb = strip_external_links(wb)

# Final save
safe_save(wb, TEMPLATE_XLSM)
print(f"✅ Final workbook saved: {Path(TEMPLATE_XLSM).name}")
print(f"✅ Remaining sheets: {load_workbook(TEMPLATE_XLSM, read_only=True).sheetnames}")


✅ Removed sheets: ['DCF - Original']
✅ Final workbook saved: DCF_Output_RELIANCE.NS_INR.xlsm
✅ Remaining sheets: ['DCF ', 'Cost of Capital', 'Industry Comparison', 'Income statement', 'Balance Sheet', 'Cash Flow']


In [25]:
# ── 9C: Optional flat XLSX export ────────────────────────────────────────────
if EXPORT_FLAT_XLSX:
    _xlsx_path = str(Path(TEMPLATE_XLSM).parent / f"DCF_Output_{TICKER}_{LOCAL_CURRENCY}.xlsx")
    with pd.ExcelWriter(_xlsx_path, engine="xlsxwriter") as writer:
        pd.DataFrame({"Input":["TICKER","BASE_YEAR","FORECAST_YEARS"],
                      "Value":[TICKER, BASE_YEAR, FORECAST_YEARS]}).to_excel(writer,"Inputs",index=False)
        HIST.to_excel(writer, "Historical", index=False)
        FORE.to_excel(writer, "Forecast",   index=False)
        pd.DataFrame([VAL]).to_excel(writer, "Valuation", index=False)
        sens_wacc_g.to_excel(writer,     "Sens_WACC_vs_G",    index=True)
        sens_mult_margin.to_excel(writer, "Sens_Mult_vs_Margin", index=True)
        CHECKS.to_excel(writer, "Checks", index=False)
    print(f"✅ Flat XLSX exported: {_xlsx_path}")
else:
    print("ℹ️  EXPORT_FLAT_XLSX=False — skipped flat XLSX export.")

# ── 9D: Optional run metadata ─────────────────────────────────────────────────
if WRITE_RUN_META_JSON:
    _meta = {
        "ticker": TICKER, "timestamp": RUN_TS, "tv_method": VAL["TV_Method"],
        "wacc_pct": VAL["WACC"]*100, "terminal_growth_pct": TERMINAL_GROWTH,
        "exit_multiple_x": EXIT_MULTIPLE, "mid_year": MID_YEAR,
        "output_excel": OUTPUT_XLSM, "base_year": BASE_YEAR,
    }
    Path("run_meta.json").write_text(json.dumps(_meta, indent=2))
    print(f"✅ run_meta.json written.")


ℹ️  EXPORT_FLAT_XLSX=False — skipped flat XLSX export.


In [26]:
# =============================================================================
# ✅ DIAGNOSTIC — Run this cell AFTER "Run All" to verify everything worked
# It checks all 5 key things and prints PASS / FAIL for each
# =============================================================================

import os, zipfile, re
from pathlib import Path
from openpyxl import load_workbook

print("=" * 60)
print("DCF OUTPUT DIAGNOSTIC")
print("=" * 60)

_output_path = globals().get("OUTPUT_XLSM") or globals().get("TEMPLATE_XLSM")
_all_pass = True

# ── Check 1: Output file exists ───────────────────────────────────────────────
print("\n[1] Output file exists?")
if _output_path and Path(_output_path).exists():
    _size_kb = Path(_output_path).stat().st_size // 1024
    print(f"    ✅ PASS — {Path(_output_path).name}  ({_size_kb} KB)")
else:
    print(f"    ❌ FAIL — File not found: {_output_path}")
    _all_pass = False

# ── Check 2: No external links (the 'Repaired' bug) ──────────────────────────
print("\n[2] No external links in workbook (Repaired dialog check)?")
try:
    _ext_found = []
    with zipfile.ZipFile(_output_path, 'r') as _z:
        # Check if externalLinks folder exists at all
        _names = _z.namelist()
        _ext_files = [n for n in _names if 'externalLink' in n.lower() or 'externallinks' in n.lower()]
        if _ext_files:
            _ext_found.extend(_ext_files)
        # Also scan sheet XMLs for external references in formulas
        _sheet_files = [n for n in _names if re.match(r'xl/worksheets/sheet\d+\.xml', n)]
        for _sf in _sheet_files:
            _xml = _z.read(_sf).decode('utf-8', errors='ignore')
            # External link pattern: [filename.xlsx] or http:// in a formula
            _ext_refs = re.findall(r'<f[^>]*>[^<]*\[.+?\][^<]*</f>', _xml)
            _ext_refs += re.findall(r'<f[^>]*>[^<]*https?://[^<]*</f>', _xml)
            if _ext_refs:
                _ext_found.append(f"{_sf}: {len(_ext_refs)} external formula(s)")
    if _ext_found:
        print(f"    ❌ FAIL — External links found:")
        for _e in _ext_found:
            print(f"       • {_e}")
        _all_pass = False
    else:
        print(f"    ✅ PASS — No external links detected")
except Exception as _e:
    print(f"    ⚠️  Could not check: {_e}")

# ── Check 3: Sheet tabs (no Paste_*, no Summary) ──────────────────────────────
print("\n[3] Sheet tabs clean (no Paste_HIST / Paste_FORE / Paste_Sens / Summary)?")
try:
    _wb = load_workbook(_output_path, read_only=True)
    _sheets = _wb.sheetnames
    _bad_sheets = [s for s in _sheets if
                   s.lower().startswith("paste") or
                   s.lower().strip() == "summary"]
    if _bad_sheets:
        print(f"    ❌ FAIL — Unwanted sheets still present: {_bad_sheets}")
        _all_pass = False
    else:
        print(f"    ✅ PASS — Sheets: {_sheets}")
except Exception as _e:
    print(f"    ⚠️  Could not check: {_e}")

# ── Check 4: Currency consistency (no ₹ in USD run) ──────────────────────────
print("\n[4] Currency consistent (no cross-contamination)?")
try:
    _ticker_upper = str(globals().get("TICKER","")).upper()
    _expected_ccy = "INR" if _ticker_upper.endswith((".NS",".BO")) else "USD"
    _detected_ccy = str(globals().get("LOCAL_CURRENCY","")).upper()
    if _detected_ccy == _expected_ccy:
        print(f"    ✅ PASS — LOCAL_CURRENCY = {_detected_ccy}  (expected {_expected_ccy})")
    else:
        print(f"    ❌ FAIL — LOCAL_CURRENCY = {_detected_ccy}  but expected {_expected_ccy}")
        _all_pass = False
    # Also check Cost of Capital sheet for wrong symbol
    _wb2 = load_workbook(_output_path, read_only=True)
    _coc = None
    for _sn in _wb2.sheetnames:
        if "cost" in _sn.lower():
            _coc = _wb2[_sn]; break
    if _coc:
        _wrong_sym = "₹" if _expected_ccy == "USD" else "$"
        _coc_formats = []
        for _row in _coc.iter_rows():
            for _cell in _row:
                if _cell.number_format and _wrong_sym in str(_cell.number_format):
                    _coc_formats.append(_cell.coordinate)
        if _coc_formats:
            print(f"    ⚠️  Wrong currency symbol in Cost of Capital: {_coc_formats[:5]}")
        else:
            print(f"    ✅ No wrong currency symbol in Cost of Capital")
except Exception as _e:
    print(f"    ⚠️  Could not check: {_e}")

# ── Check 5: Sensitivity table has values ─────────────────────────────────────
print("\n[5] Sensitivity table populated (D43:I48)?")
try:
    _wb3  = load_workbook(_output_path, read_only=True)
    _dcf  = None
    for _sn in _wb3.sheetnames:
        if _sn.strip().lower() == "dcf":
            _dcf = _wb3[_sn]; break
    if _dcf:
        _filled = 0
        _total  = 0
        for _r in range(43, 49):
            for _c in range(4, 10):  # cols D=4 to I=9
                _total += 1
                _v = _dcf.cell(_r, _c).value
                if _v is not None and str(_v).strip() not in ("","None"):
                    _filled += 1
        if _filled >= 20:  # expect at least 20 of 36 cells filled
            print(f"    ✅ PASS — {_filled}/{_total} cells populated in D43:I48")
        else:
            print(f"    ❌ FAIL — Only {_filled}/{_total} cells populated in D43:I48")
            _all_pass = False
    else:
        print(f"    ❌ FAIL — DCF sheet not found")
        _all_pass = False
except Exception as _e:
    print(f"    ⚠️  Could not check: {_e}")

# ── Check 6: IS / BS / CF sheets have clean labels ───────────────────────────
print("\n[6] Financial statement sheets have clean labels?")
try:
    _wb4 = load_workbook(_output_path, read_only=True)
    _stmt_sheets = {"income statement": False, "balance sheet": False, "cash flow": False}
    for _sn in _wb4.sheetnames:
        _snl = _sn.strip().lower()
        for _k in _stmt_sheets:
            if _k in _snl:
                _stmt_sheets[_k] = True
    _missing = [k for k, v in _stmt_sheets.items() if not v]
    if _missing:
        print(f"    ❌ FAIL — Missing sheets: {_missing}")
        _all_pass = False
    else:
        print(f"    ✅ PASS — All 3 statement sheets present")
    # Spot-check for raw yfinance labels in IS sheet
    _is_sheet = None
    for _sn in _wb4.sheetnames:
        if "income" in _sn.strip().lower():
            _is_sheet = _wb4[_sn]; break
    if _is_sheet:
        _raw_yf_labels = ["tax effect of unusual items","reconciled depreciation",
                          "total unusual items excluding goodwill","normalized ebitda"]
        _found_raw = []
        for _row in _is_sheet.iter_rows(max_row=60):
            for _cell in _row:
                if isinstance(_cell.value, str):
                    _cv = _cell.value.strip().lower()
                    for _lbl in _raw_yf_labels:
                        if _lbl in _cv:
                            _found_raw.append(_cell.value[:50])
        if _found_raw:
            print(f"    ⚠️  Raw yfinance labels still present in IS: {_found_raw[:3]}")
        else:
            print(f"    ✅ No raw yfinance labels detected in Income Statement")
except Exception as _e:
    print(f"    ⚠️  Could not check: {_e}")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
if _all_pass:
    print("✅ ALL CHECKS PASSED — workbook is clean")
else:
    print("⚠️  SOME CHECKS FAILED — see details above")
print("=" * 60)

# ── Bonus: print key valuation numbers ───────────────────────────────────────
print("\n📊 Valuation Summary:")
try:
    print(f"   Ticker        : {TICKER}")
    print(f"   Base Year     : {BASE_YEAR}")
    print(f"   WACC          : {VAL['WACC']*100:.2f}%")
    print(f"   Terminal Growth: {TERMINAL_GROWTH:.2f}%")
    print(f"   Enterprise Value: {VAL['EV']:,.2f}")
    print(f"   Equity Value  : {VAL['EquityValue']:,.2f}")
    if 'PerShare' in VAL and VAL['PerShare'] is not None and str(VAL['PerShare']) != 'nan':
        print(f"   Per Share     : {SYM}{VAL['PerShare']:,.2f}")
    else:
        print(f"   Per Share     : N/A")
    print(f"   Currency      : {LOCAL_CURRENCY}")
    print(f"   Output file   : {Path(OUTPUT_XLSM).name}")
except Exception as _e:
    print(f"   (Could not print summary: {_e})")


DCF OUTPUT DIAGNOSTIC

[1] Output file exists?
    ✅ PASS — DCF_Output_RELIANCE.NS_INR.xlsm  (52 KB)

[2] No external links in workbook (Repaired dialog check)?
    ❌ FAIL — External links found:
       • xl/externalLinks/externalLink1.xml
       • xl/externalLinks/_rels/externalLink1.xml.rels

[3] Sheet tabs clean (no Paste_HIST / Paste_FORE / Paste_Sens / Summary)?
    ✅ PASS — Sheets: ['DCF ', 'Cost of Capital', 'Industry Comparison', 'Income statement', 'Balance Sheet', 'Cash Flow']

[4] Currency consistent (no cross-contamination)?
    ✅ PASS — LOCAL_CURRENCY = INR  (expected INR)
    ✅ No wrong currency symbol in Cost of Capital

[5] Sensitivity table populated (D43:I48)?
    ✅ PASS — 35/36 cells populated in D43:I48

[6] Financial statement sheets have clean labels?
    ✅ PASS — All 3 statement sheets present
    ✅ No raw yfinance labels detected in Income Statement

⚠️  SOME CHECKS FAILED — see details above

📊 Valuation Summary:
   Ticker        : RELIANCE.NS
   Base Year     

## ✅ Run Complete

The output workbook is ready at:
```
DCF_Output_{TICKER}_{CURRENCY}.xlsm
```

**Sheets in output:**
- `DCF` — main valuation model
- `Cost of Capital` — WACC inputs
- `Income statement` — clean labeled IS
- `Balance Sheet` — clean labeled BS
- `Cash Flow` — clean labeled CFS
- `Industry Comparison` — peer comps

**Sensitivity table** (DCF!D43:I48): WACC vs Terminal Growth rate, 5×5 grid, pure Python values.


In [ ]:
# ── 9E: DCF SUMMARY JSON — sidecar for backend consumption ─────────────────
# Writes {xlsm_stem}.summary.json next to the output workbook. Backend reads
# this instead of the .xlsm so it can surface computed values without needing
# Excel to recalc formulas (openpyxl does not execute formulas).
import json as _json
from pathlib import Path as _Path

def _j(v):
    try:
        if v is None: return None
        if hasattr(v, "item"): v = v.item()
        if isinstance(v, float) and (math.isnan(v) or math.isinf(v)): return None
        if isinstance(v, (int, float, str, bool)): return v
        return str(v)
    except Exception:
        return None

_out_xlsm = _Path(globals().get("OUTPUT_XLSM") or str(OUTPUT_XLSM_PATH))
_summary_path = _out_xlsm.with_name(_out_xlsm.stem + ".summary.json")

# Current market price (best-effort)
_current_price = None
try:
    _info = yf.Ticker(TICKER).info or {}
    _current_price = _info.get("currentPrice") or _info.get("regularMarketPrice")
except Exception:
    pass

def _series(df, col, year_col="FY"):
    out = []
    try:
        if df is None or df.empty or col not in df.columns: return out
        for _, r in df.sort_values(year_col).iterrows():
            if pd.notna(r.get(col)):
                out.append({"fy": _j(r.get(year_col)), col.lower(): _j(r[col])})
    except Exception:
        pass
    return out

_VAL = globals().get("VAL") or {}
_HIST = globals().get("HIST")
_FORE = globals().get("FORE")

_fv = _j(_VAL.get("PerShare"))
_cp = _j(_current_price)
_upside = None
try:
    if _fv is not None and _cp is not None and float(_cp) != 0.0:
        _upside = round((float(_fv) - float(_cp)) / float(_cp) * 100.0, 2)
except Exception:
    _upside = None

_summary = {
    "schema_version": 1,
    "ticker": TICKER,
    "currency": LOCAL_CURRENCY,
    "region": REGION,
    "run_timestamp": RUN_TS,
    "output_xlsm": _out_xlsm.name,
    "valuation": {
        "fair_value_per_share": _fv,
        "current_price":        _cp,
        "upside_pct":           _upside,
        "wacc":                 _j(_VAL.get("WACC")),
        "terminal_growth":      _j(globals().get("TERMINAL_GROWTH")),
        "tv_method":            _j(_VAL.get("TV_Method")),
        "pv_terminal_value":    _j(_VAL.get("PV_TV")),
        "pv_fcffs":             _j(_VAL.get("PV_FCFFs")),
        "enterprise_value":     _j(_VAL.get("EV")),
        "equity_value":         _j(_VAL.get("EquityValue")),
        "net_debt":             _j(_VAL.get("NetDebt")),
        "shares":               _j(_VAL.get("Shares")),
    },
    "assumptions": {
        "risk_free_pct":            _j(globals().get("RISK_FREE")),
        "erp_pct":                  _j(globals().get("EQUITY_RISK_PREMIUM")),
        "beta":                     _j(globals().get("BETA")),
        "cost_of_debt_pretax_pct":  _j(globals().get("COST_OF_DEBT_PRETAX")),
        "tax_rate_pct":             _j(globals().get("TAX_RATE")),
        "wacc_direct":              _j(globals().get("WACC_DIRECT")),
        "mid_year":                 _j(globals().get("MID_YEAR")),
        "use_exit_multiple":        _j(globals().get("USE_EXIT_MULTIPLE")),
        "base_year":                _j(globals().get("BASE_YEAR")),
        "forecast_years":           _j(globals().get("FORECAST_YEARS")),
    },
    "historical_fcf":   _series(_HIST, "FCFF"),
    "forecast_fcf":     _series(_FORE, "FCFF"),
    "forecast_revenue": _series(_FORE, "Revenue"),
}

_summary_path.write_text(_json.dumps(_summary, indent=2, default=str))
print(f"✅ Summary JSON: {_summary_path.name}")
